
# 12. Production Inference & Model Contract Validation

## IPL Player Performance Analytics V2

### 1. Notebook Objective

The objective of this notebook is to establish and validate a reusable production inference workflow for the 10 selected IPL player performance prediction models.

This notebook validates the complete inference lifecycle, including:

- Model artifact loading
- Final model-selection verification
- Raw input feature schema validation
- Numerical and categorical feature validation
- Model-specific preprocessing
- ML and DL inference execution
- DL target inverse transformation
- Prediction output validation
- Business constraint validation
- Model inference contract generation
- Inference validation reporting

### 2. Production Model Selection

The final model-selection configuration is maintained in:

`artifacts/model_comparison/final_target_model_selection.json`

The selected production models are:

| Prediction Target | Selected Model | Model Type |
|---|---|---|
| Next Match Runs Scored | batting_original | ML |
| Next Match Balls Faced | DL_MLP | DL |
| Next Match Fours | DL_MLP | DL |
| Next Match Sixes | DL_MLP | DL |
| Next Match Strike Rate | DL_MLP | DL |
| Next Match Dismissals | DL_MLP | DL |
| Next Match Wickets | bowling_wickets | ML |
| Next Match Runs Conceded | DL_MLP | DL |
| Next Match Legal Balls | DL_MLP | DL |
| Next Match Economy Rate | DL_MLP | DL |

**Total:** 2 Machine Learning models and 8 Deep Learning models.

### 3. Inference Architecture

The inference workflow follows this sequence:

1. Load the final model-selection configuration.
2. Load the selected model artifacts.
3. Identify each model's required raw input features.
4. Validate the input DataFrame against the model contract.
5. Apply the saved preprocessing pipeline.
6. Generate predictions using the selected model.
7. Apply inverse target scaling for Deep Learning models.
8. Validate prediction outputs and business constraints.
9. Generate inference validation reports.

### 4. Important Engineering Principles

- Reuse existing trained model artifacts.
- Do not retrain models in this notebook.
- Do not modify completed notebooks 01–11.
- Do not overwrite existing model artifacts.
- Do not manually recreate fitted preprocessing transformations.
- Use model-specific feature schemas.
- Preserve chronological evaluation methodology.
- Prevent target leakage during inference.
- Never silently replace missing features with arbitrary values.
- Maintain reproducibility and traceability.

### 5. Expected Deliverables

The notebook should produce:

- Validated production model registry.
- Model-specific inference contracts.
- Model loading validation results.
- Prediction validation results for all 10 targets.
- Output constraint validation results.
- Final inference readiness report.

### 6. Scope and Limitations

This notebook focuses exclusively on model inference and contract validation.

Django application development, REST API implementation, frontend development, database integration, authentication, and deployment will be handled in subsequent implementation stages.

Existing model evaluation results remain the source of truth for model performance. Successful inference validation does not imply that a model has high predictive accuracy or is production-proven.


In [1]:

# ============================================================
# CELL 2: ENVIRONMENT, REPRODUCIBILITY AND PROJECT CONFIGURATION
# ============================================================

import os
import sys
import json
import random
import platform
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
import joblib

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. Reproducibility configuration
# ------------------------------------------------------------

RANDOM_STATE = 42

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("Reproducibility configuration initialized.")

# ------------------------------------------------------------
# 2. Project root discovery
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd()

PROJECT_ROOT = next(
    (
        parent
        for parent in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (
            (parent / "artifacts" / "dl_models").is_dir()
            and (parent / "models").is_dir()
            and (parent / "paths.py").is_file()
        )
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project root could not be identified. "
        "Expected artifacts/dl_models, models and paths.py."
    )

os.chdir(PROJECT_ROOT)

print("\nProject configuration:")
print("Current directory:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)

# ------------------------------------------------------------
# 3. Important project directories
# ------------------------------------------------------------

PATHS = {
    "models": PROJECT_ROOT / "models",
    "artifacts": PROJECT_ROOT / "artifacts",
    "dl_models": PROJECT_ROOT / "artifacts" / "dl_models",
    "model_comparison": (
        PROJECT_ROOT / "artifacts" / "model_comparison"
    ),
    "metrics": PROJECT_ROOT / "metrics",
    "notebooks": PROJECT_ROOT / "notebooks",
}

print("\nDirectory validation:")

for name, path in PATHS.items():
    print(f"{name:20} | Exists: {path.exists()} | {path}")

# ------------------------------------------------------------
# 4. Software version audit
# ------------------------------------------------------------

SOFTWARE_VERSIONS = {
    "Python": sys.version.split()[0],
    "Platform": platform.platform(),
    "NumPy": np.__version__,
    "Pandas": pd.__version__,
    "Scikit-learn": sklearn.__version__,
    "TensorFlow": tf.__version__,
    "Joblib": joblib.__version__,
}

print("\nSoftware versions:")

for library, version in SOFTWARE_VERSIONS.items():
    print(f"{library:20}: {version}")

# ------------------------------------------------------------
# 5. TensorFlow device information
# ------------------------------------------------------------

print("\nTensorFlow devices:")

for device in tf.config.list_physical_devices():
    print(device)

# ------------------------------------------------------------
# 6. Artifact registry path
# ------------------------------------------------------------

MODEL_SELECTION_PATH = (
    PATHS["model_comparison"]
    / "final_target_model_selection.json"
)

DL_REGISTRY_PATH = (
    PATHS["dl_models"]
    / "model_registry.json"
)

print("\nRegistry validation:")

for name, path in {
    "Final model selection": MODEL_SELECTION_PATH,
    "DL model registry": DL_REGISTRY_PATH,
}.items():

    print(f"{name}: {path.exists()}")

    if not path.exists():
        raise FileNotFoundError(
            f"Required registry file not found: {path}"
        )

# ------------------------------------------------------------
# 7. Environment validation summary
# ------------------------------------------------------------

ENVIRONMENT_SUMMARY = {
    "project_root": str(PROJECT_ROOT),
    "python_version": SOFTWARE_VERSIONS["Python"],
    "software_versions": SOFTWARE_VERSIONS,
    "random_state": RANDOM_STATE,
    "model_selection_exists": MODEL_SELECTION_PATH.exists(),
    "dl_registry_exists": DL_REGISTRY_PATH.exists(),
}

print("\nEnvironment summary:")
print(json.dumps(ENVIRONMENT_SUMMARY, indent=4))

print("\nCELL 2 COMPLETED SUCCESSFULLY.")


Reproducibility configuration initialized.

Project configuration:
Current directory: d:\Git\Personal\ipl-player-performance-v2\notebooks
Project root: d:\Git\Personal\ipl-player-performance-v2

Directory validation:
models               | Exists: True | d:\Git\Personal\ipl-player-performance-v2\models
artifacts            | Exists: True | d:\Git\Personal\ipl-player-performance-v2\artifacts
dl_models            | Exists: True | d:\Git\Personal\ipl-player-performance-v2\artifacts\dl_models
model_comparison     | Exists: True | d:\Git\Personal\ipl-player-performance-v2\artifacts\model_comparison
metrics              | Exists: True | d:\Git\Personal\ipl-player-performance-v2\metrics
notebooks            | Exists: True | d:\Git\Personal\ipl-player-performance-v2\notebooks

Software versions:
Python              : 3.12.16
Platform            : Windows-11-10.0.26300-SP0
NumPy               : 2.5.4
Pandas              : 3.1.0
Scikit-learn        : 1.9.1
TensorFlow          : 2.21.0
Joblib    

In [2]:

# ============================================================
# CELL 3: LOAD AND VALIDATE FINAL MODEL-SELECTION REGISTRY
# ============================================================

import json
from pathlib import Path

# ------------------------------------------------------------
# 1. Load final model-selection configuration
# ------------------------------------------------------------

with open(MODEL_SELECTION_PATH, "r", encoding="utf-8") as file:
    MODEL_SELECTION = json.load(file)

print("Final model-selection registry loaded successfully.")

print("\nProject:", MODEL_SELECTION.get("project"))
print("Selection version:", MODEL_SELECTION.get("selection_version"))
print("Created at:", MODEL_SELECTION.get("created_at_utc"))

# ------------------------------------------------------------
# 2. Expected production model mapping
# ------------------------------------------------------------

EXPECTED_MODEL_MAPPING = {
    "target_next_match_runs_scored": "batting_original",
    "target_next_match_balls_faced": "DL_MLP",
    "target_next_match_fours": "DL_MLP",
    "target_next_match_sixes": "DL_MLP",
    "target_next_match_strike_rate": "DL_MLP",
    "target_next_match_dismissals": "DL_MLP",
    "target_next_match_wickets": "bowling_wickets",
    "target_next_match_runs_conceded": "DL_MLP",
    "target_next_match_legal_balls": "DL_MLP",
    "target_next_match_economy_rate": "DL_MLP",
}

# ------------------------------------------------------------
# 3. Extract actual model mapping
# ------------------------------------------------------------

ACTUAL_MODEL_MAPPING = {
    target: details.get("model")
    for target, details in MODEL_SELECTION.get("models", {}).items()
}

# ------------------------------------------------------------
# 4. Validate target coverage
# ------------------------------------------------------------

expected_targets = set(EXPECTED_MODEL_MAPPING.keys())
actual_targets = set(ACTUAL_MODEL_MAPPING.keys())

missing_targets = expected_targets - actual_targets
unexpected_targets = actual_targets - expected_targets

assert not missing_targets, (
    f"Missing prediction targets: {missing_targets}"
)

assert not unexpected_targets, (
    f"Unexpected prediction targets: {unexpected_targets}"
)

# ------------------------------------------------------------
# 5. Validate selected model assignments
# ------------------------------------------------------------

mapping_errors = []

for target, expected_model in EXPECTED_MODEL_MAPPING.items():

    actual_model = ACTUAL_MODEL_MAPPING[target]

    if actual_model != expected_model:
        mapping_errors.append({
            "target": target,
            "expected": expected_model,
            "actual": actual_model,
        })

assert not mapping_errors, (
    f"Model-selection mapping mismatch: {mapping_errors}"
)

# ------------------------------------------------------------
# 6. Calculate model distribution
# ------------------------------------------------------------

ML_TARGETS = [
    target
    for target, model in ACTUAL_MODEL_MAPPING.items()
    if model != "DL_MLP"
]

DL_TARGETS = [
    target
    for target, model in ACTUAL_MODEL_MAPPING.items()
    if model == "DL_MLP"
]

assert len(ML_TARGETS) == 2
assert len(DL_TARGETS) == 8
assert len(ACTUAL_MODEL_MAPPING) == 10

# ------------------------------------------------------------
# 7. Display validated production mapping
# ------------------------------------------------------------

MODEL_SELECTION_DF = pd.DataFrame([
    {
        "prediction_target": target,
        "selected_model": model,
        "model_type": "DL" if model == "DL_MLP" else "ML",
        "selection_reason": MODEL_SELECTION["models"][target].get(
            "reason", "Not provided"
        ),
    }
    for target, model in ACTUAL_MODEL_MAPPING.items()
])

print("\nPRODUCTION MODEL-SELECTION SUMMARY")
display(MODEL_SELECTION_DF)

print("\nMODEL DISTRIBUTION")
print("Total targets:", len(ACTUAL_MODEL_MAPPING))
print("Machine Learning:", len(ML_TARGETS))
print("Deep Learning:", len(DL_TARGETS))

# ------------------------------------------------------------
# 8. Validate selection policy
# ------------------------------------------------------------

SELECTION_POLICY = MODEL_SELECTION.get("selection_policy", {})

print("\nSELECTION POLICY")
print(json.dumps(SELECTION_POLICY, indent=4))

assert SELECTION_POLICY.get("primary_metric") == "MAE", (
    "Unexpected primary selection metric."
)

print("\nCELL 3 COMPLETED SUCCESSFULLY.")


Final model-selection registry loaded successfully.

Project: IPL Player Performance Analytics V2
Selection version: 1.0
Created at: 2026-11-02T16:39:09.294479+00:00

PRODUCTION MODEL-SELECTION SUMMARY


,prediction_target,selected_model,model_type,selection_reason
0,target_next_match_runs_scored,batting_original,ML,Best balanced RMSE and R2; batting_sqrt has ma...
1,target_next_match_balls_faced,DL_MLP,DL,Lower MAE and RMSE than training-mean baseline.
2,target_next_match_fours,DL_MLP,DL,Lower MAE and RMSE than training-mean baseline.
3,target_next_match_sixes,DL_MLP,DL,"Improves MAE, RMSE and R2 over training-mean b..."
4,target_next_match_strike_rate,DL_MLP,DL,Better than training-mean baseline; R2 remains...
5,target_next_match_dismissals,DL_MLP,DL,Improves MAE and RMSE over training-mean basel...
6,target_next_match_wickets,bowling_wickets,ML,"ML beats DL and baseline on MAE, RMSE and R2."
7,target_next_match_runs_conceded,DL_MLP,DL,Better MAE and RMSE than both baselines.
8,target_next_match_legal_balls,DL_MLP,DL,Substantial improvement over both baselines.
9,target_next_match_economy_rate,DL_MLP,DL,Marginally lower MAE than saved dummy and bett...



MODEL DISTRIBUTION
Total targets: 10
Machine Learning: 2
Deep Learning: 8

SELECTION POLICY
{
    "primary_metric": "MAE",
    "supporting_metrics": [
        "RMSE",
        "R2",
        "Bias"
    ],
    "test_period": "2024-05-02 to 2026-05-29",
    "baseline": "Training_Mean_Baseline"
}

CELL 3 COMPLETED SUCCESSFULLY.


In [3]:

# ============================================================
# CELL 4: DEEP LEARNING MODEL REGISTRY AUDIT
# ============================================================

# ------------------------------------------------------------
# 1. Load DL model registry
# ------------------------------------------------------------

with open(DL_REGISTRY_PATH, "r", encoding="utf-8") as file:
    DL_REGISTRY = json.load(file)

print("DL model registry loaded successfully.")

print("\nREGISTRY STRUCTURE")
print("Top-level keys:", list(DL_REGISTRY.keys()))

print("\nCOMPLETE DL REGISTRY")
print(json.dumps(DL_REGISTRY, indent=4))

# ------------------------------------------------------------
# 2. Identify selected DL targets
# ------------------------------------------------------------

SELECTED_DL_TARGETS = [
    target
    for target, model_name in ACTUAL_MODEL_MAPPING.items()
    if model_name == "DL_MLP"
]

print("\nSELECTED DL TARGETS")
for target in SELECTED_DL_TARGETS:
    print("-", target)

assert len(SELECTED_DL_TARGETS) == 8, (
    "Expected exactly 8 selected DL targets."
)

# ------------------------------------------------------------
# 3. Validate selected DL artifact directories
# ------------------------------------------------------------

DL_ARTIFACT_CHECKS = []

for target in SELECTED_DL_TARGETS:

    artifact_dir = (
        PATHS["dl_models"]
        / target.replace("target_next_match_", "")
    )

    # Actual project structure uses separate batting/bowling folders.
    if not artifact_dir.exists():

        for category in ["batting", "bowling"]:
            candidate = PATHS["dl_models"] / category / target

            if candidate.exists():
                artifact_dir = candidate
                break

    required_files = [
        "model.keras",
        "preprocessor.joblib",
        "target_scaler.joblib",
        "metadata.json",
        "training_history.json",
    ]

    file_status = {
        filename: (artifact_dir / filename).is_file()
        for filename in required_files
    }

    DL_ARTIFACT_CHECKS.append({
        "target": target,
        "artifact_directory": str(artifact_dir),
        "directory_exists": artifact_dir.is_dir(),
        **file_status,
        "all_required_files_present": all(file_status.values()),
    })

DL_ARTIFACT_CHECK_DF = pd.DataFrame(DL_ARTIFACT_CHECKS)

print("\nSELECTED DL ARTIFACT VALIDATION")
display(DL_ARTIFACT_CHECK_DF)

# ------------------------------------------------------------
# 4. Summarize missing artifacts
# ------------------------------------------------------------

missing_artifacts = DL_ARTIFACT_CHECK_DF[
    ~DL_ARTIFACT_CHECK_DF["all_required_files_present"]
]

print("\nDL ARTIFACT SUMMARY")
print("Selected DL targets:", len(SELECTED_DL_TARGETS))
print(
    "Complete artifact directories:",
    int(DL_ARTIFACT_CHECK_DF["all_required_files_present"].sum())
)
print("Incomplete artifact directories:", len(missing_artifacts))

if not missing_artifacts.empty:
    print("\nARTIFACTS REQUIRING INVESTIGATION")
    display(missing_artifacts)

print("\nCELL 4 INSPECTION COMPLETED.")


DL model registry loaded successfully.

REGISTRY STRUCTURE
Top-level keys: ['project', 'registry_version', 'created_at_utc', 'models']

COMPLETE DL REGISTRY
{
    "project": "IPL Player Performance Analytics V2",
    "registry_version": "1.0",
    "created_at_utc": "2026-11-02T16:31:29.179497+00:00",
    "models": {
        "target_next_match_balls_faced": {
            "category": "batting",
            "target": "target_next_match_balls_faced",
            "model_path": "artifacts/dl_models/batting/target_next_match_balls_faced/model.keras",
            "preprocessor_path": "artifacts/dl_models/batting/target_next_match_balls_faced/preprocessor.joblib",
            "scaler_path": "artifacts/dl_models/batting/target_next_match_balls_faced/target_scaler.joblib",
            "history_path": "artifacts/dl_models/batting/target_next_match_balls_faced/training_history.json",
            "metadata_path": "artifacts/dl_models/batting/target_next_match_balls_faced/metadata.json",
            

,target,artifact_directory,directory_exists,model.keras,preprocessor.joblib,target_scaler.joblib,metadata.json,training_history.json,all_required_files_present
0,target_next_match_balls_faced,d:\Git\Personal\ipl-player-performance-v2\arti...,True,True,True,True,True,True,True
1,target_next_match_fours,d:\Git\Personal\ipl-player-performance-v2\arti...,True,True,True,True,True,True,True
2,target_next_match_sixes,d:\Git\Personal\ipl-player-performance-v2\arti...,True,True,True,True,True,True,True
3,target_next_match_strike_rate,d:\Git\Personal\ipl-player-performance-v2\arti...,True,True,True,True,True,True,True
4,target_next_match_dismissals,d:\Git\Personal\ipl-player-performance-v2\arti...,True,True,True,True,True,True,True
5,target_next_match_runs_conceded,d:\Git\Personal\ipl-player-performance-v2\arti...,True,True,True,True,True,True,True
6,target_next_match_legal_balls,d:\Git\Personal\ipl-player-performance-v2\arti...,True,True,True,True,True,True,True
7,target_next_match_economy_rate,d:\Git\Personal\ipl-player-performance-v2\arti...,True,True,True,True,True,True,True



DL ARTIFACT SUMMARY
Selected DL targets: 8
Complete artifact directories: 8
Incomplete artifact directories: 0

CELL 4 INSPECTION COMPLETED.


In [4]:

# ============================================================
# CELL 5: DL METADATA AND PREPROCESSING SCHEMA VALIDATION
# ============================================================

DL_SCHEMA_RESULTS = []
DL_SCHEMA_DETAILS = {}

for target in SELECTED_DL_TARGETS:

    print("\n" + "=" * 85)
    print("TARGET:", target)

    # --------------------------------------------------------
    # 1. Retrieve registry configuration
    # --------------------------------------------------------

    config = DL_REGISTRY["models"][target]

    preprocessor_path = PROJECT_ROOT / config["preprocessor_path"]
    scaler_path = PROJECT_ROOT / config["scaler_path"]
    metadata_path = PROJECT_ROOT / config["metadata_path"]

    # --------------------------------------------------------
    # 2. Load saved preprocessing artifacts
    # --------------------------------------------------------

    preprocessor = joblib.load(preprocessor_path)
    target_scaler = joblib.load(scaler_path)

    with open(metadata_path, "r", encoding="utf-8") as file:
        metadata = json.load(file)

    # --------------------------------------------------------
    # 3. Extract raw and transformed schemas
    # --------------------------------------------------------

    raw_features = list(preprocessor.feature_names_in_)
    transformed_features = list(
        preprocessor.get_feature_names_out()
    )

    raw_feature_count = len(raw_features)
    transformed_feature_count = len(transformed_features)

    registry_feature_count = config["feature_count"]

    metadata_target = metadata.get("target")
    metadata_feature_count = metadata.get("feature_count")

    # --------------------------------------------------------
    # 4. Validate schema consistency
    # --------------------------------------------------------

    registry_count_match = (
        transformed_feature_count == registry_feature_count
    )

    metadata_target_match = (
        metadata_target == target
    )

    metadata_count_match = (
        metadata_feature_count == transformed_feature_count
    )

    scaler_feature_count = getattr(
        target_scaler,
        "n_features_in_",
        None
    )

    scaler_shape_valid = (
        scaler_feature_count is None
        or scaler_feature_count == 1
    )

    # --------------------------------------------------------
    # 5. Store detailed schema information
    # --------------------------------------------------------

    DL_SCHEMA_DETAILS[target] = {
        "raw_features": raw_features,
        "transformed_features": transformed_features,
        "preprocessor": preprocessor,
        "target_scaler": target_scaler,
        "metadata": metadata,
    }

    DL_SCHEMA_RESULTS.append({
        "target": target,
        "category": config["category"],
        "raw_feature_count": raw_feature_count,
        "transformed_feature_count": transformed_feature_count,
        "registry_feature_count": registry_feature_count,
        "metadata_feature_count": metadata_feature_count,
        "registry_count_match": registry_count_match,
        "metadata_target_match": metadata_target_match,
        "metadata_count_match": metadata_count_match,
        "scaler_shape_valid": scaler_shape_valid,
        "all_checks_passed": all([
            registry_count_match,
            metadata_target_match,
            metadata_count_match,
            scaler_shape_valid,
        ]),
    })

    # --------------------------------------------------------
    # 6. Display target-specific findings
    # --------------------------------------------------------

    print("Category:", config["category"])
    print("Raw input features:", raw_feature_count)
    print("Transformed features:", transformed_feature_count)
    print("Registry feature count:", registry_feature_count)
    print("Metadata target:", metadata_target)
    print("Metadata feature count:", metadata_feature_count)
    print("Target scaler type:", type(target_scaler).__name__)
    print("Target scaler dimensions:", scaler_feature_count)

    print("\nSchema validation:")
    print("Registry count match:", registry_count_match)
    print("Metadata target match:", metadata_target_match)
    print("Metadata count match:", metadata_count_match)
    print("Scaler shape valid:", scaler_shape_valid)

# ------------------------------------------------------------
# 7. Generate consolidated validation report
# ------------------------------------------------------------

DL_SCHEMA_VALIDATION_DF = pd.DataFrame(DL_SCHEMA_RESULTS)

print("\n" + "=" * 85)
print("CONSOLIDATED DL SCHEMA VALIDATION")

display(DL_SCHEMA_VALIDATION_DF)

# ------------------------------------------------------------
# 8. Final validation summary
# ------------------------------------------------------------

passed_targets = int(
    DL_SCHEMA_VALIDATION_DF["all_checks_passed"].sum()
)

failed_targets = len(DL_SCHEMA_VALIDATION_DF) - passed_targets

print("\nVALIDATION SUMMARY")
print("Selected DL targets:", len(SELECTED_DL_TARGETS))
print("Passed:", passed_targets)
print("Failed:", failed_targets)

if failed_targets > 0:
    print("\nTARGETS REQUIRING INVESTIGATION:")
    display(
        DL_SCHEMA_VALIDATION_DF[
            ~DL_SCHEMA_VALIDATION_DF["all_checks_passed"]
        ]
    )
else:
    print("\nALL SELECTED DL SCHEMAS PASSED.")

print("\nCELL 5 COMPLETED.")



TARGET: target_next_match_balls_faced
Category: batting
Raw input features: 42
Transformed features: 59
Registry feature count: 59
Metadata target: target_next_match_balls_faced
Metadata feature count: 59
Target scaler type: StandardScaler
Target scaler dimensions: 1

Schema validation:
Registry count match: True
Metadata target match: True
Metadata count match: True
Scaler shape valid: True

TARGET: target_next_match_fours
Category: batting
Raw input features: 42
Transformed features: 59
Registry feature count: 59
Metadata target: target_next_match_fours
Metadata feature count: 59
Target scaler type: StandardScaler
Target scaler dimensions: 1

Schema validation:
Registry count match: True
Metadata target match: True
Metadata count match: True
Scaler shape valid: True

TARGET: target_next_match_sixes
Category: batting
Raw input features: 42
Transformed features: 59
Registry feature count: 59
Metadata target: target_next_match_sixes
Metadata feature count: 59
Target scaler type: Standa

,target,category,raw_feature_count,transformed_feature_count,registry_feature_count,metadata_feature_count,registry_count_match,metadata_target_match,metadata_count_match,scaler_shape_valid,all_checks_passed
0,target_next_match_balls_faced,batting,42,59,59,59,True,True,True,True,True
1,target_next_match_fours,batting,42,59,59,59,True,True,True,True,True
2,target_next_match_sixes,batting,42,59,59,59,True,True,True,True,True
3,target_next_match_strike_rate,batting,42,59,59,59,True,True,True,True,True
4,target_next_match_dismissals,batting,42,59,59,59,True,True,True,True,True
5,target_next_match_runs_conceded,bowling,30,30,30,30,True,True,True,True,True
6,target_next_match_legal_balls,bowling,30,30,30,30,True,True,True,True,True
7,target_next_match_economy_rate,bowling,30,30,30,30,True,True,True,True,True



VALIDATION SUMMARY
Selected DL targets: 8
Passed: 8
Failed: 0

ALL SELECTED DL SCHEMAS PASSED.

CELL 5 COMPLETED.


In [5]:

# ============================================================
# CELL 6: ML MODEL ARTIFACT AND FEATURE SCHEMA VALIDATION
# ============================================================

# ------------------------------------------------------------
# 1. Define selected ML artifact paths
# ------------------------------------------------------------

ML_ARTIFACT_CONFIG = {
    "target_next_match_runs_scored": {
        "model_name": "batting_original",
        "path": PATHS["models"] / "batting" / "original_gb_model.joblib",
        "expected_estimator": "GradientBoostingRegressor",
        "expected_raw_features": 43,
    },
    "target_next_match_wickets": {
        "model_name": "bowling_wickets",
        "path": PATHS["models"] / "bowling" / "wickets_model.joblib",
        "expected_estimator": "HistGradientBoostingRegressor",
        "expected_raw_features": 35,
    },
}

# ------------------------------------------------------------
# 2. Validate model-selection consistency
# ------------------------------------------------------------

ML_SCHEMA_RESULTS = []
ML_SCHEMA_DETAILS = {}

for target, config in ML_ARTIFACT_CONFIG.items():

    print("\n" + "=" * 85)
    print("TARGET:", target)

    # --------------------------------------------------------
    # 3. Verify selected model
    # --------------------------------------------------------

    selected_model = ACTUAL_MODEL_MAPPING[target]

    assert selected_model == config["model_name"], (
        f"Model selection mismatch for {target}: "
        f"{selected_model}"
    )

    model_path = config["path"]

    assert model_path.is_file(), (
        f"ML model artifact not found: {model_path}"
    )

    # --------------------------------------------------------
    # 4. Load fitted pipeline
    # --------------------------------------------------------

    pipeline = joblib.load(model_path)

    assert hasattr(pipeline, "named_steps"), (
        "Expected a fitted sklearn Pipeline."
    )

    assert "preprocessor" in pipeline.named_steps
    assert "model" in pipeline.named_steps

    preprocessor = pipeline.named_steps["preprocessor"]
    estimator = pipeline.named_steps["model"]

    # --------------------------------------------------------
    # 5. Extract raw and transformed schemas
    # --------------------------------------------------------

    raw_features = list(pipeline.feature_names_in_)

    preprocessor_features = list(
        preprocessor.feature_names_in_
    )

    transformed_features = list(
        preprocessor.get_feature_names_out()
    )

    raw_feature_count = len(raw_features)
    transformed_feature_count = len(transformed_features)

    # --------------------------------------------------------
    # 6. Validate schema consistency
    # --------------------------------------------------------

    raw_schema_match = (
        raw_features == preprocessor_features
    )

    feature_count_match = (
        raw_feature_count == config["expected_raw_features"]
    )

    estimator_match = (
        type(estimator).__name__
        == config["expected_estimator"]
    )

    no_duplicate_raw_features = (
        len(raw_features) == len(set(raw_features))
    )

    no_duplicate_transformed_features = (
        len(transformed_features)
        == len(set(transformed_features))
    )

    # --------------------------------------------------------
    # 7. Extract preprocessing configuration
    # --------------------------------------------------------

    transformer_summary = []

    for transformer_name, transformer, columns in (
        preprocessor.transformers_
    ):

        column_list = (
            list(columns)
            if not isinstance(columns, str)
            else [columns]
        )

        transformer_summary.append({
            "transformer": transformer_name,
            "columns": column_list,
            "column_count": len(column_list),
        })

    # --------------------------------------------------------
    # 8. Validate bowling feature metadata
    # --------------------------------------------------------

    metadata_match = True

    if target == "target_next_match_wickets":

        metadata_path = (
            PATHS["models"]
            / "bowling"
            / "wickets_feature_metadata.json"
        )

        with open(metadata_path, "r", encoding="utf-8") as file:
            ml_metadata = json.load(file)

        metadata_features = ml_metadata.get("features", [])

        metadata_match = (
            metadata_features == raw_features
            and ml_metadata.get("target") == target
        )

    # --------------------------------------------------------
    # 9. Store model details
    # --------------------------------------------------------

    ML_SCHEMA_DETAILS[target] = {
        "pipeline": pipeline,
        "preprocessor": preprocessor,
        "estimator": estimator,
        "raw_features": raw_features,
        "transformed_features": transformed_features,
        "transformers": transformer_summary,
    }

    checks = [
        raw_schema_match,
        feature_count_match,
        estimator_match,
        no_duplicate_raw_features,
        no_duplicate_transformed_features,
        metadata_match,
    ]

    ML_SCHEMA_RESULTS.append({
        "target": target,
        "selected_model": selected_model,
        "estimator": type(estimator).__name__,
        "raw_feature_count": raw_feature_count,
        "transformed_feature_count": transformed_feature_count,
        "raw_schema_match": raw_schema_match,
        "feature_count_match": feature_count_match,
        "estimator_match": estimator_match,
        "no_duplicate_raw_features": no_duplicate_raw_features,
        "no_duplicate_transformed_features": (
            no_duplicate_transformed_features
        ),
        "metadata_match": metadata_match,
        "all_checks_passed": all(checks),
    })

    # --------------------------------------------------------
    # 10. Display findings
    # --------------------------------------------------------

    print("Selected model:", selected_model)
    print("Estimator:", type(estimator).__name__)
    print("Raw features:", raw_feature_count)
    print("Transformed features:", transformed_feature_count)

    print("\nPreprocessing transformers:")

    for item in transformer_summary:
        print(
            f"{item['transformer']}: "
            f"{item['column_count']} columns"
        )

    print("\nValidation checks:")
    print("Raw schema match:", raw_schema_match)
    print("Raw feature count match:", feature_count_match)
    print("Estimator match:", estimator_match)
    print("Unique raw features:", no_duplicate_raw_features)
    print(
        "Unique transformed features:",
        no_duplicate_transformed_features
    )
    print("Metadata match:", metadata_match)

# ------------------------------------------------------------
# 11. Consolidated ML validation report
# ------------------------------------------------------------

ML_SCHEMA_VALIDATION_DF = pd.DataFrame(ML_SCHEMA_RESULTS)

print("\n" + "=" * 85)
print("CONSOLIDATED ML SCHEMA VALIDATION")

display(ML_SCHEMA_VALIDATION_DF)

# ------------------------------------------------------------
# 12. Final validation summary
# ------------------------------------------------------------

passed = int(
    ML_SCHEMA_VALIDATION_DF["all_checks_passed"].sum()
)

failed = len(ML_SCHEMA_VALIDATION_DF) - passed

print("\nML VALIDATION SUMMARY")
print("Selected ML targets:", len(ML_ARTIFACT_CONFIG))
print("Passed:", passed)
print("Failed:", failed)

assert failed == 0, (
    "One or more selected ML models failed validation."
)

print("\nALL SELECTED ML SCHEMAS PASSED.")

print("\nCELL 6 COMPLETED.")



TARGET: target_next_match_runs_scored
Selected model: batting_original
Estimator: GradientBoostingRegressor
Raw features: 43
Transformed features: 74

Preprocessing transformers:
numerical: 41 columns
categorical: 2 columns

Validation checks:
Raw schema match: True
Raw feature count match: True
Estimator match: True
Unique raw features: True
Unique transformed features: True
Metadata match: True

TARGET: target_next_match_wickets
Selected model: bowling_wickets
Estimator: HistGradientBoostingRegressor
Raw features: 35
Transformed features: 49

Preprocessing transformers:
numerical: 32 columns
categorical: 3 columns

Validation checks:
Raw schema match: True
Raw feature count match: True
Estimator match: True
Unique raw features: True
Unique transformed features: True
Metadata match: True

CONSOLIDATED ML SCHEMA VALIDATION


,target,selected_model,estimator,raw_feature_count,transformed_feature_count,raw_schema_match,feature_count_match,estimator_match,no_duplicate_raw_features,no_duplicate_transformed_features,metadata_match,all_checks_passed
0,target_next_match_runs_scored,batting_original,GradientBoostingRegressor,43,74,True,True,True,True,True,True,True
1,target_next_match_wickets,bowling_wickets,HistGradientBoostingRegressor,35,49,True,True,True,True,True,True,True



ML VALIDATION SUMMARY
Selected ML targets: 2
Passed: 2
Failed: 0

ALL SELECTED ML SCHEMAS PASSED.

CELL 6 COMPLETED.


In [6]:

# ============================================================
# CELL 7: BUILD UNIFIED PRODUCTION MODEL CONTRACT REGISTRY
# ============================================================

# ------------------------------------------------------------
# 1. Initialize unified registry
# ------------------------------------------------------------

PRODUCTION_MODEL_REGISTRY = {}

# ------------------------------------------------------------
# 2. Register selected ML models
# ------------------------------------------------------------

for target, config in ML_ARTIFACT_CONFIG.items():

    details = ML_SCHEMA_DETAILS[target]

    PRODUCTION_MODEL_REGISTRY[target] = {
        "target": target,
        "model_name": config["model_name"],
        "model_type": "ML",
        "category": (
            "batting"
            if target == "target_next_match_runs_scored"
            else "bowling"
        ),
        "model_path": str(config["path"]),
        "raw_features": details["raw_features"],
        "raw_feature_count": len(details["raw_features"]),
        "transformed_feature_count": len(
            details["transformed_features"]
        ),
        "estimator_type": type(
            details["estimator"]
        ).__name__,
        "preprocessing_type": "sklearn_pipeline",
        "target_scaling_required": False,
        "validated": True,
    }

# ------------------------------------------------------------
# 3. Register selected DL models
# ------------------------------------------------------------

for target in SELECTED_DL_TARGETS:

    config = DL_REGISTRY["models"][target]
    details = DL_SCHEMA_DETAILS[target]

    PRODUCTION_MODEL_REGISTRY[target] = {
        "target": target,
        "model_name": "DL_MLP",
        "model_type": "DL",
        "category": config["category"],
        "model_path": str(
            PROJECT_ROOT / config["model_path"]
        ),
        "preprocessor_path": str(
            PROJECT_ROOT / config["preprocessor_path"]
        ),
        "scaler_path": str(
            PROJECT_ROOT / config["scaler_path"]
        ),
        "metadata_path": str(
            PROJECT_ROOT / config["metadata_path"]
        ),
        "history_path": str(
            PROJECT_ROOT / config["history_path"]
        ),
        "raw_features": details["raw_features"],
        "raw_feature_count": len(details["raw_features"]),
        "transformed_feature_count": len(
            details["transformed_features"]
        ),
        "estimator_type": "Keras Sequential MLP",
        "preprocessing_type": "saved_column_transformer",
        "target_scaling_required": True,
        "validated": True,
    }

# ------------------------------------------------------------
# 4. Validate registry target coverage
# ------------------------------------------------------------

registry_targets = set(PRODUCTION_MODEL_REGISTRY.keys())
selected_targets = set(ACTUAL_MODEL_MAPPING.keys())

assert registry_targets == selected_targets, (
    "Unified registry does not match selected production targets."
)

# ------------------------------------------------------------
# 5. Validate model assignments
# ------------------------------------------------------------

for target, contract in PRODUCTION_MODEL_REGISTRY.items():

    assert contract["model_name"] == ACTUAL_MODEL_MAPPING[target], (
        f"Model assignment mismatch: {target}"
    )

    assert contract["raw_feature_count"] == len(
        contract["raw_features"]
    )

    assert contract["validated"] is True

# ------------------------------------------------------------
# 6. Generate unified contract summary
# ------------------------------------------------------------

PRODUCTION_MODEL_CONTRACT_DF = pd.DataFrame([
    {
        "target": target,
        "model_name": contract["model_name"],
        "model_type": contract["model_type"],
        "category": contract["category"],
        "raw_feature_count": contract["raw_feature_count"],
        "transformed_feature_count": (
            contract["transformed_feature_count"]
        ),
        "preprocessing_type": contract["preprocessing_type"],
        "target_scaling_required": (
            contract["target_scaling_required"]
        ),
        "validated": contract["validated"],
    }
    for target, contract in PRODUCTION_MODEL_REGISTRY.items()
])

print("UNIFIED PRODUCTION MODEL CONTRACT REGISTRY")

display(PRODUCTION_MODEL_CONTRACT_DF)

# ------------------------------------------------------------
# 7. Registry summary
# ------------------------------------------------------------

print("\nREGISTRY SUMMARY")
print("Total targets:", len(PRODUCTION_MODEL_REGISTRY))

print(
    "ML targets:",
    sum(
        item["model_type"] == "ML"
        for item in PRODUCTION_MODEL_REGISTRY.values()
    )
)

print(
    "DL targets:",
    sum(
        item["model_type"] == "DL"
        for item in PRODUCTION_MODEL_REGISTRY.values()
    )
)

print(
    "Validated contracts:",
    sum(
        item["validated"]
        for item in PRODUCTION_MODEL_REGISTRY.values()
    )
)

assert len(PRODUCTION_MODEL_REGISTRY) == 10

print("\nALL 10 PRODUCTION MODEL CONTRACTS REGISTERED.")

print("\nCELL 7 COMPLETED.")


UNIFIED PRODUCTION MODEL CONTRACT REGISTRY


,target,model_name,model_type,category,raw_feature_count,transformed_feature_count,preprocessing_type,target_scaling_required,validated
0,target_next_match_runs_scored,batting_original,ML,batting,43,74,sklearn_pipeline,False,True
1,target_next_match_wickets,bowling_wickets,ML,bowling,35,49,sklearn_pipeline,False,True
2,target_next_match_balls_faced,DL_MLP,DL,batting,42,59,saved_column_transformer,True,True
3,target_next_match_fours,DL_MLP,DL,batting,42,59,saved_column_transformer,True,True
4,target_next_match_sixes,DL_MLP,DL,batting,42,59,saved_column_transformer,True,True
5,target_next_match_strike_rate,DL_MLP,DL,batting,42,59,saved_column_transformer,True,True
6,target_next_match_dismissals,DL_MLP,DL,batting,42,59,saved_column_transformer,True,True
7,target_next_match_runs_conceded,DL_MLP,DL,bowling,30,30,saved_column_transformer,True,True
8,target_next_match_legal_balls,DL_MLP,DL,bowling,30,30,saved_column_transformer,True,True
9,target_next_match_economy_rate,DL_MLP,DL,bowling,30,30,saved_column_transformer,True,True



REGISTRY SUMMARY
Total targets: 10
ML targets: 2
DL targets: 8
Validated contracts: 10

ALL 10 PRODUCTION MODEL CONTRACTS REGISTERED.

CELL 7 COMPLETED.


In [7]:

# ============================================================
# CELL 8: LOAD ALL 10 SELECTED PRODUCTION MODELS
# ============================================================

LOADED_MODELS = {}
MODEL_LOADING_RESULTS = []

# ------------------------------------------------------------
# 1. Load selected ML pipelines
# ------------------------------------------------------------

for target, config in ML_ARTIFACT_CONFIG.items():

    model = joblib.load(config["path"])

    LOADED_MODELS[target] = {
        "model_type": "ML",
        "model": model,
        "preprocessor": model.named_steps["preprocessor"],
    }

    MODEL_LOADING_RESULTS.append({
        "target": target,
        "model_type": "ML",
        "loaded": True,
        "input_shape": getattr(model, "n_features_in_", None),
        "output_shape": "Single prediction",
    })

# ------------------------------------------------------------
# 2. Load selected DL models
# ------------------------------------------------------------

for target in SELECTED_DL_TARGETS:

    config = DL_REGISTRY["models"][target]

    model_path = PROJECT_ROOT / config["model_path"]

    model = tf.keras.models.load_model(
        model_path,
        compile=False
    )

    preprocessor = DL_SCHEMA_DETAILS[target]["preprocessor"]
    scaler = DL_SCHEMA_DETAILS[target]["target_scaler"]

    LOADED_MODELS[target] = {
        "model_type": "DL",
        "model": model,
        "preprocessor": preprocessor,
        "target_scaler": scaler,
    }

    MODEL_LOADING_RESULTS.append({
        "target": target,
        "model_type": "DL",
        "loaded": True,
        "input_shape": model.input_shape,
        "output_shape": model.output_shape,
    })

# ------------------------------------------------------------
# 3. Consolidated loading report
# ------------------------------------------------------------

MODEL_LOADING_DF = pd.DataFrame(MODEL_LOADING_RESULTS)

display(MODEL_LOADING_DF)

print("\nMODEL LOADING SUMMARY")
print("Expected models:", 10)
print("Loaded models:", len(LOADED_MODELS))
print("ML models:", sum(x["model_type"] == "ML" for x in LOADED_MODELS.values()))
print("DL models:", sum(x["model_type"] == "DL" for x in LOADED_MODELS.values()))

assert len(LOADED_MODELS) == 10
assert MODEL_LOADING_DF["loaded"].all()

print("\nALL 10 SELECTED MODELS LOADED SUCCESSFULLY.")
print("\nCELL 8 COMPLETED.")


,target,model_type,loaded,input_shape,output_shape
0,target_next_match_runs_scored,ML,True,43,Single prediction
1,target_next_match_wickets,ML,True,35,Single prediction
2,target_next_match_balls_faced,DL,True,"(None, 59)","(None, 1)"
3,target_next_match_fours,DL,True,"(None, 59)","(None, 1)"
4,target_next_match_sixes,DL,True,"(None, 59)","(None, 1)"
5,target_next_match_strike_rate,DL,True,"(None, 59)","(None, 1)"
6,target_next_match_dismissals,DL,True,"(None, 59)","(None, 1)"
7,target_next_match_runs_conceded,DL,True,"(None, 30)","(None, 1)"
8,target_next_match_legal_balls,DL,True,"(None, 30)","(None, 1)"
9,target_next_match_economy_rate,DL,True,"(None, 30)","(None, 1)"



MODEL LOADING SUMMARY
Expected models: 10
Loaded models: 10
ML models: 2
DL models: 8

ALL 10 SELECTED MODELS LOADED SUCCESSFULLY.

CELL 8 COMPLETED.


In [8]:

# ============================================================
# CELL 9: DISCOVER EXISTING INFERENCE TEST DATASETS
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# 1. Search relevant project directories
# ------------------------------------------------------------

SEARCH_DIRECTORIES = [
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "artifacts",
    PROJECT_ROOT / "notebooks",
]

SUPPORTED_EXTENSIONS = {
    ".csv", ".parquet", ".pkl", ".pickle", ".joblib"
}

dataset_candidates = []

for search_dir in SEARCH_DIRECTORIES:

    if not search_dir.exists():
        continue

    for path in search_dir.rglob("*"):

        if not path.is_file():
            continue

        if ".git" in path.parts:
            continue

        if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue

        filename = path.name.lower()
        relative_path = str(path.relative_to(PROJECT_ROOT)).lower()

        relevant = any(
            keyword in filename or keyword in relative_path
            for keyword in [
                "test",
                "feature",
                "batting",
                "bowling",
                "dataset",
            ]
        )

        if relevant:
            dataset_candidates.append({
                "file_name": path.name,
                "relative_path": str(path.relative_to(PROJECT_ROOT)),
                "extension": path.suffix.lower(),
                "size_mb": round(path.stat().st_size / (1024 ** 2), 3),
            })

# ------------------------------------------------------------
# 2. Display discovered datasets
# ------------------------------------------------------------

DATASET_DISCOVERY_DF = pd.DataFrame(dataset_candidates)

if DATASET_DISCOVERY_DF.empty:
    print("No candidate datasets found.")
else:
    DATASET_DISCOVERY_DF = (
        DATASET_DISCOVERY_DF
        .drop_duplicates(subset=["relative_path"])
        .sort_values("relative_path")
        .reset_index(drop=True)
    )

    display(DATASET_DISCOVERY_DF)

# ------------------------------------------------------------
# 3. Identify chronological test-related candidates
# ------------------------------------------------------------

test_candidates = DATASET_DISCOVERY_DF[
    DATASET_DISCOVERY_DF["file_name"].str.contains(
        "test", case=False, na=False
    )
] if not DATASET_DISCOVERY_DF.empty else pd.DataFrame()

print("\nDATASET DISCOVERY SUMMARY")
print("Total candidate files:", len(DATASET_DISCOVERY_DF))
print("Test-related files:", len(test_candidates))

print("\nCELL 9 COMPLETED.")


,file_name,relative_path,extension,size_mb
0,preprocessor.joblib,artifacts\dl_models\batting\target_next_match_...,.joblib,0.007
1,target_scaler.joblib,artifacts\dl_models\batting\target_next_match_...,.joblib,0.001
2,preprocessor.joblib,artifacts\dl_models\batting\target_next_match_...,.joblib,0.007
3,target_scaler.joblib,artifacts\dl_models\batting\target_next_match_...,.joblib,0.001
4,preprocessor.joblib,artifacts\dl_models\batting\target_next_match_...,.joblib,0.007
5,target_scaler.joblib,artifacts\dl_models\batting\target_next_match_...,.joblib,0.001
6,preprocessor.joblib,artifacts\dl_models\batting\target_next_match_...,.joblib,0.007
7,target_scaler.joblib,artifacts\dl_models\batting\target_next_match_...,.joblib,0.001
8,preprocessor.joblib,artifacts\dl_models\batting\target_next_match_...,.joblib,0.007
9,target_scaler.joblib,artifacts\dl_models\batting\target_next_match_...,.joblib,0.001



DATASET DISCOVERY SUMMARY
Total candidate files: 40
Test-related files: 5

CELL 9 COMPLETED.


In [9]:

# ============================================================
# CELL 9A: IDENTIFY CHRONOLOGICAL TEST DATASETS
# ============================================================

if test_candidates.empty:
    print("No test-related files found.")
else:
    display(
        test_candidates[
            ["file_name", "relative_path", "extension", "size_mb"]
        ].reset_index(drop=True)
    )

    print("\nCOMPLETE TEST DATASET PATHS")

    for relative_path in test_candidates["relative_path"]:
        print(PROJECT_ROOT / relative_path)


,file_name,relative_path,extension,size_mb
0,mlp_test_summary.csv,artifacts\dl_validation\mlp_test_summary.csv,.csv,0.002
1,constrained_test_predictions.csv,artifacts\model_comparison\constrained_test_pr...,.csv,2.317
2,ml_dl_test_comparison_20261102_161403.csv,artifacts\model_comparison\ml_dl_test_comparis...,.csv,0.001
3,ml_dl_test_comparison_20261102_161447.csv,artifacts\model_comparison\ml_dl_test_comparis...,.csv,0.001
4,mlp_test_summary.csv,notebooks\artifacts\dl_validation\mlp_test_sum...,.csv,0.002



COMPLETE TEST DATASET PATHS
d:\Git\Personal\ipl-player-performance-v2\artifacts\dl_validation\mlp_test_summary.csv
d:\Git\Personal\ipl-player-performance-v2\artifacts\model_comparison\constrained_test_predictions.csv
d:\Git\Personal\ipl-player-performance-v2\artifacts\model_comparison\ml_dl_test_comparison_20261102_161403.csv
d:\Git\Personal\ipl-player-performance-v2\artifacts\model_comparison\ml_dl_test_comparison_20261102_161447.csv
d:\Git\Personal\ipl-player-performance-v2\notebooks\artifacts\dl_validation\mlp_test_summary.csv


In [10]:

# ============================================================
# CELL 10: INSPECT EXISTING TEST AND EVALUATION CSV FILES
# ============================================================

CSV_INSPECTION_RESULTS = {}
CSV_SUMMARY = []

unique_paths = sorted(
    set(
        PROJECT_ROOT / relative_path
        for relative_path in test_candidates["relative_path"]
    )
)

for file_path in unique_paths:

    print("\n" + "=" * 85)
    print("FILE:", file_path.relative_to(PROJECT_ROOT))

    if not file_path.exists():
        print("File not found.")
        continue

    df = pd.read_csv(file_path)

    CSV_INSPECTION_RESULTS[file_path.name] = df

    print("Shape:", df.shape)
    print("Columns:")
    print(df.columns.tolist())

    print("\nFirst 3 rows:")
    display(df.head(3))

    CSV_SUMMARY.append({
        "file_name": file_path.name,
        "relative_path": str(file_path.relative_to(PROJECT_ROOT)),
        "rows": len(df),
        "columns": len(df.columns),
        "column_names": ", ".join(df.columns.astype(str)),
    })

# ------------------------------------------------------------
# Consolidated report
# ------------------------------------------------------------

CSV_INSPECTION_DF = pd.DataFrame(CSV_SUMMARY)

print("\nCSV INSPECTION SUMMARY")
display(CSV_INSPECTION_DF)

print("\nCELL 10 COMPLETED.")



FILE: artifacts\dl_validation\mlp_test_summary.csv
Shape: (10, 12)
Columns:
['Target', 'Test_Samples', 'DL_MAE', 'Baseline_MAE', 'MAE_Improvement_Percent', 'DL_RMSE', 'Baseline_RMSE', 'RMSE_Improvement_Percent', 'DL_R2', 'Prediction_Min', 'Prediction_Max', 'Negative_Predictions']

First 3 rows:


,Target,Test_Samples,DL_MAE,Baseline_MAE,MAE_Improvement_Percent,DL_RMSE,Baseline_RMSE,RMSE_Improvement_Percent,DL_R2,Prediction_Min,Prediction_Max,Negative_Predictions
0,target_next_match_balls_faced,2375,9.364080,10.423990,10.167986,12.018597,12.874722,6.649655,0.127566,2.799105,30.958149,0
1,target_next_match_fours,2375,1.671235,1.827386,8.545051,2.295062,2.449228,6.294471,0.111262,0.179742,4.072121,0
2,target_next_match_sixes,2375,1.157966,1.210932,4.373938,1.789982,1.835988,2.505770,-0.022557,-0.014651,1.470601,2



FILE: artifacts\model_comparison\constrained_test_predictions.csv
Shape: (21468, 10)
Columns:
['category', 'target', 'selected_model', 'source_index', 'match_id', 'player', 'date', 'actual', 'raw_prediction', 'constrained_prediction']

First 3 rows:


,category,target,selected_model,source_index,match_id,player,date,actual,raw_prediction,constrained_prediction
0,batting,target_next_match_runs_scored,batting_original,53,1426292,A Badoni,2024-05-05,55.0,19.488948,19.0
1,batting,target_next_match_runs_scored,batting_original,54,1426295,A Badoni,2024-05-08,6.0,18.356989,18.0
2,batting,target_next_match_runs_scored,batting_original,55,1426302,A Badoni,2024-05-14,22.0,19.582317,20.0



FILE: artifacts\model_comparison\ml_dl_test_comparison_20261102_161403.csv
Shape: (11, 9)
Columns:
['category', 'target', 'model', 'rows', 'MAE', 'RMSE', 'R2', 'Bias', 'negative_predictions']

First 3 rows:


,category,target,model,rows,MAE,RMSE,R2,Bias,negative_predictions
0,Batting,target_next_match_runs_scored,batting_sqrt,2375,16.543208,23.685276,0.012017,-7.613649,0
1,Batting,target_next_match_runs_scored,batting_original,2375,16.643802,22.459996,0.111593,-2.564665,0
2,Batting,target_next_match_runs_scored,DL_MLP,2375,16.763107,22.593155,0.101028,-3.007407,0


... (3 outputs left out)

,Target,Test_Samples,DL_MAE,Baseline_MAE,MAE_Improvement_Percent,DL_RMSE,Baseline_RMSE,RMSE_Improvement_Percent,DL_R2,Prediction_Min,Prediction_Max,Negative_Predictions
0,target_next_match_balls_faced,2375,9.364080,10.423990,10.167986,12.018597,12.874722,6.649655,0.127566,2.799105,30.958149,0
1,target_next_match_fours,2375,1.671235,1.827386,8.545051,2.295062,2.449228,6.294471,0.111262,0.179742,4.072121,0
2,target_next_match_sixes,2375,1.157966,1.210932,4.373938,1.789982,1.835988,2.505770,-0.022557,-0.014651,1.470601,2



CSV INSPECTION SUMMARY


,file_name,relative_path,rows,columns,column_names
0,mlp_test_summary.csv,artifacts\dl_validation\mlp_test_summary.csv,10,12,"Target, Test_Samples, DL_MAE, Baseline_MAE, MA..."
1,constrained_test_predictions.csv,artifacts\model_comparison\constrained_test_pr...,21468,10,"category, target, selected_model, source_index..."
2,ml_dl_test_comparison_20261102_161403.csv,artifacts\model_comparison\ml_dl_test_comparis...,11,9,"category, target, model, rows, MAE, RMSE, R2, ..."
3,ml_dl_test_comparison_20261102_161447.csv,artifacts\model_comparison\ml_dl_test_comparis...,11,9,"category, target, model, rows, MAE, RMSE, R2, ..."
4,mlp_test_summary.csv,notebooks\artifacts\dl_validation\mlp_test_sum...,10,12,"Target, Test_Samples, DL_MAE, Baseline_MAE, MA..."



CELL 10 COMPLETED.


In [11]:

# ============================================================
# CELL 11: VALIDATE SAVED TEST PREDICTIONS
# ============================================================

PREDICTION_FILE = (
    PROJECT_ROOT
    / "artifacts"
    / "model_comparison"
    / "constrained_test_predictions.csv"
)

predictions_df = pd.read_csv(PREDICTION_FILE)

# ------------------------------------------------------------
# 1. Expected production model mapping
# ------------------------------------------------------------

EXPECTED_MODELS = {
    "target_next_match_runs_scored": "batting_original",
    "target_next_match_wickets": "bowling_wickets",
    "target_next_match_balls_faced": "DL_MLP",
    "target_next_match_fours": "DL_MLP",
    "target_next_match_sixes": "DL_MLP",
    "target_next_match_strike_rate": "DL_MLP",
    "target_next_match_dismissals": "DL_MLP",
    "target_next_match_runs_conceded": "DL_MLP",
    "target_next_match_legal_balls": "DL_MLP",
    "target_next_match_economy_rate": "DL_MLP",
}

REQUIRED_COLUMNS = [
    "category",
    "target",
    "selected_model",
    "source_index",
    "match_id",
    "player",
    "date",
    "actual",
    "raw_prediction",
    "constrained_prediction",
]

missing_columns = sorted(
    set(REQUIRED_COLUMNS) - set(predictions_df.columns)
)

assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)

# ------------------------------------------------------------
# 2. Numerical validation
# ------------------------------------------------------------

NUMERIC_COLUMNS = [
    "actual",
    "raw_prediction",
    "constrained_prediction",
]

for column in NUMERIC_COLUMNS:
    predictions_df[column] = pd.to_numeric(
        predictions_df[column],
        errors="coerce"
    )

numeric_valid = np.isfinite(
    predictions_df[NUMERIC_COLUMNS].to_numpy(dtype=float)
).all()

# ------------------------------------------------------------
# 3. Target and model coverage
# ------------------------------------------------------------

observed_targets = set(predictions_df["target"].dropna())

expected_targets = set(EXPECTED_MODELS)

target_coverage_valid = observed_targets == expected_targets

model_mapping_results = []

for target, expected_model in EXPECTED_MODELS.items():

    target_df = predictions_df[
        predictions_df["target"] == target
    ]

    observed_models = target_df["selected_model"].dropna().unique()

    model_mapping_results.append({
        "target": target,
        "expected_model": expected_model,
        "observed_models": list(observed_models),
        "rows": len(target_df),
        "model_match": (
            len(observed_models) == 1
            and observed_models[0] == expected_model
        ),
    })

MODEL_MAPPING_VALIDATION_DF = pd.DataFrame(
    model_mapping_results
)

# ------------------------------------------------------------
# 4. Per-target validation report
# ------------------------------------------------------------

TARGET_PREDICTION_SUMMARY = (
    predictions_df
    .groupby(["category", "target", "selected_model"], dropna=False)
    .agg(
        test_rows=("actual", "size"),
        actual_missing=("actual", lambda x: x.isna().sum()),
        raw_prediction_missing=(
            "raw_prediction", lambda x: x.isna().sum()
        ),
        constrained_prediction_missing=(
            "constrained_prediction", lambda x: x.isna().sum()
        ),
        actual_mean=("actual", "mean"),
        prediction_mean=("constrained_prediction", "mean"),
    )
    .reset_index()
)

display(MODEL_MAPPING_VALIDATION_DF)
display(TARGET_PREDICTION_SUMMARY)

# ------------------------------------------------------------
# 5. Final checks
# ------------------------------------------------------------

print("\nPREDICTION VALIDATION SUMMARY")
print("Total prediction rows:", len(predictions_df))
print("Observed targets:", len(observed_targets))
print("Expected targets:", len(expected_targets))
print("Target coverage valid:", target_coverage_valid)
print("Numerical predictions finite:", numeric_valid)
print(
    "Model mappings valid:",
    MODEL_MAPPING_VALIDATION_DF["model_match"].all()
)

assert target_coverage_valid
assert numeric_valid
assert MODEL_MAPPING_VALIDATION_DF["model_match"].all()

print("\nALL SAVED TEST PREDICTION CHECKS PASSED.")
print("CELL 11 COMPLETED.")


,target,expected_model,observed_models,rows,model_match
0,target_next_match_runs_scored,batting_original,[batting_original],2375,True
1,target_next_match_wickets,bowling_wickets,[bowling_wickets],1805,True
2,target_next_match_balls_faced,DL_MLP,[DL_MLP],2375,True
3,target_next_match_fours,DL_MLP,[DL_MLP],2375,True
4,target_next_match_sixes,DL_MLP,[DL_MLP],2375,True
5,target_next_match_strike_rate,DL_MLP,[DL_MLP],2373,True
6,target_next_match_dismissals,DL_MLP,[DL_MLP],2375,True
7,target_next_match_runs_conceded,DL_MLP,[DL_MLP],1805,True
8,target_next_match_legal_balls,DL_MLP,[DL_MLP],1805,True
9,target_next_match_economy_rate,DL_MLP,[DL_MLP],1805,True


,category,target,selected_model,test_rows,actual_missing,raw_prediction_missing,constrained_prediction_missing,actual_mean,prediction_mean
0,batting,target_next_match_balls_faced,DL_MLP,2375,0,0,0,15.128842,14.776421
1,batting,target_next_match_dismissals,DL_MLP,2375,0,0,0,0.786105,0.966737
2,batting,target_next_match_fours,DL_MLP,2375,0,0,0,2.084632,1.776421
3,batting,target_next_match_runs_scored,batting_original,2375,0,0,0,23.388632,20.824000
4,batting,target_next_match_sixes,DL_MLP,2375,0,0,0,1.240421,0.844211
5,batting,target_next_match_strike_rate,DL_MLP,2373,0,0,0,133.044311,117.169958
6,bowling,target_next_match_economy_rate,DL_MLP,1805,0,0,0,9.975718,8.640831
7,bowling,target_next_match_legal_balls,DL_MLP,1805,0,0,0,19.318560,20.659280
8,bowling,target_next_match_runs_conceded,DL_MLP,1805,0,0,0,30.822715,29.787258
9,bowling,target_next_match_wickets,bowling_wickets,1805,0,0,0,0.978947,0.981717



PREDICTION VALIDATION SUMMARY
Total prediction rows: 21468
Observed targets: 10
Expected targets: 10
Target coverage valid: True
Numerical predictions finite: True
Model mappings valid: True

ALL SAVED TEST PREDICTION CHECKS PASSED.
CELL 11 COMPLETED.


In [12]:

# ============================================================
# CELL 12: TRACE ORIGINAL TEST DATASET CREATION
# ============================================================

import json
import re

NOTEBOOK_11_PATH = (
    PROJECT_ROOT
    / "notebooks"
    / "11_dl_dataset_preparation_and_leakage_audit.ipynb"
)

assert NOTEBOOK_11_PATH.exists(), (
    f"Notebook not found: {NOTEBOOK_11_PATH}"
)

with open(NOTEBOOK_11_PATH, "r", encoding="utf-8") as file:
    notebook_11 = json.load(file)

SEARCH_TERMS = [
    "train_test_split",
    "test_df",
    "X_test",
    "y_test",
    "test_features",
    "test_target",
    "test_mask",
    "chronological",
    "split",
    "feature_datasets",
    "target_datasets",
    "batting_datasets",
    "bowling_datasets",
]

matched_cells = []

for cell_index, cell in enumerate(notebook_11["cells"]):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    matching_terms = [
        term for term in SEARCH_TERMS
        if term.lower() in source.lower()
    ]

    if matching_terms:
        matched_cells.append({
            "cell_index": cell_index,
            "matching_terms": matching_terms,
            "source": source,
        })

print("NOTEBOOK 11:", NOTEBOOK_11_PATH)
print("Total cells:", len(notebook_11["cells"]))
print("Matching code cells:", len(matched_cells))

for item in matched_cells:

    print("\n" + "=" * 90)
    print("CELL INDEX:", item["cell_index"])
    print("MATCHES:", item["matching_terms"])
    print("-" * 90)
    print(item["source"])

print("\nCELL 12 COMPLETED.")


NOTEBOOK 11: d:\Git\Personal\ipl-player-performance-v2\notebooks\11_dl_dataset_preparation_and_leakage_audit.ipynb
Total cells: 124
Matching code cells: 27

CELL INDEX: 4
MATCHES: ['split']
------------------------------------------------------------------------------------------
# ============================================================
# 3. CONSTANT FEATURE AND SEASON AUDIT
# ============================================================

def normalize_season(value):
    if pd.isna(value):
        return np.nan

    return int(str(value).split("/")[0])


for name, df in [
    ("Batting", batting_df),
    ("Bowling", bowling_df)
]:

    print("\n" + "=" * 65)
    print(name.upper())
    print("=" * 65)

    # Identify constant features
    constant_features = [
        col for col in df.columns
        if df[col].nunique(dropna=False) == 1
    ]

    print("Constant features:", constant_features)

    # Preview season normalization
    if "season" in df.columns:

        season_prev


## 13. Original Test Dataset Availability Audit

### Objective
Identify whether the original chronological test feature datasets from Notebook 11 are available in the current Notebook 12 kernel.

### Validation scope
- Inspect existing in-memory dataset variables.
- Identify batting and bowling test datasets.
- Inspect their structures and dimensions.
- Confirm whether raw test features are available for all 10 selected targets.
- Avoid creating new train/test splits or modifying existing datasets.

### Expected outcome
A clear inventory of available test datasets and the missing inputs required for production inference validation.


In [13]:

# ============================================================
# CELL 13: ORIGINAL TEST DATASET AVAILABILITY AUDIT
# ============================================================

import pandas as pd

candidate_variables = [
    "batting_dl_datasets",
    "bowling_dl_datasets",
    "batting_dl_processed",
    "bowling_dl_processed",
    "batting_dl_scaled",
    "bowling_dl_scaled",
    "batting_model_datasets",
    "bowling_model_datasets",
    "batting_datasets",
    "bowling_datasets",
]

print("=" * 90)
print("ORIGINAL TEST DATASET AVAILABILITY AUDIT")
print("=" * 90)

available_variables = {}

for variable_name in candidate_variables:

    if variable_name in globals():

        obj = globals()[variable_name]

        available_variables[variable_name] = obj

        print(f"\nVARIABLE: {variable_name}")
        print("Type:", type(obj).__name__)

        if isinstance(obj, dict):
            print("Top-level keys:", list(obj.keys())[:20])

        elif isinstance(obj, pd.DataFrame):
            print("Shape:", obj.shape)
            print("Columns:", obj.columns.tolist()[:15])

        elif isinstance(obj, (list, tuple)):
            print("Length:", len(obj))

    else:
        print(f"\nVARIABLE: {variable_name} | NOT AVAILABLE")


print("\n" + "=" * 90)
print("TEST DATASET STRUCTURE INSPECTION")
print("=" * 90)

for variable_name, obj in available_variables.items():

    if not isinstance(obj, dict):
        continue

    print(f"\nDATASET: {variable_name}")

    for key, value in list(obj.items())[:3]:

        print(f"\nKey: {key}")
        print("Type:", type(value).__name__)

        if isinstance(value, dict):

            print("Nested keys:", list(value.keys()))

            for nested_key, nested_value in value.items():

                if isinstance(nested_value, pd.DataFrame):
                    print(
                        f"  {nested_key}: DataFrame "
                        f"{nested_value.shape}"
                    )

        elif isinstance(value, pd.DataFrame):
            print("Shape:", value.shape)


print("\n" + "=" * 90)
print("AUDIT SUMMARY")
print("=" * 90)

print("Candidate variables found:", len(available_variables))
print("Available variables:", list(available_variables.keys()))

if not available_variables:
    print(
        "\nOriginal test datasets are not loaded in this kernel. "
        "We must locate their saved source before inference."
    )

print("\nCELL 13 COMPLETED.")


ORIGINAL TEST DATASET AVAILABILITY AUDIT

VARIABLE: batting_dl_datasets | NOT AVAILABLE

VARIABLE: bowling_dl_datasets | NOT AVAILABLE

VARIABLE: batting_dl_processed | NOT AVAILABLE

VARIABLE: bowling_dl_processed | NOT AVAILABLE

VARIABLE: batting_dl_scaled | NOT AVAILABLE

VARIABLE: bowling_dl_scaled | NOT AVAILABLE

VARIABLE: batting_model_datasets | NOT AVAILABLE

VARIABLE: bowling_model_datasets | NOT AVAILABLE

VARIABLE: batting_datasets | NOT AVAILABLE

VARIABLE: bowling_datasets | NOT AVAILABLE

TEST DATASET STRUCTURE INSPECTION

AUDIT SUMMARY
Candidate variables found: 0
Available variables: []

Original test datasets are not loaded in this kernel. We must locate their saved source before inference.

CELL 13 COMPLETED.



## 14. Saved Dataset Discovery

### Objective
Locate the saved datasets and artifacts generated by the previous notebooks.

### Inspection scope
- data/processed/
- artifacts/
- notebooks/artifacts/
- data/

Search for CSV, Parquet, Pickle, and Joblib files that may contain the original feature datasets or test splits.

Also inspect Notebook 11 for dataset export operations.

### Important
This is a read-only filesystem inspection. No files will be created, modified, or deleted.


In [14]:

# ============================================================
# CELL 14: SAVED DATASET DISCOVERY
# ============================================================

from pathlib import Path
import json

SEARCH_DIRECTORIES = [
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "artifacts",
    PROJECT_ROOT / "notebooks" / "artifacts",
]

SUPPORTED_EXTENSIONS = {
    ".csv",
    ".parquet",
    ".pkl",
    ".pickle",
    ".joblib",
    ".feather",
}

dataset_files = []

print("=" * 90)
print("SAVED DATASET DISCOVERY")
print("=" * 90)

for search_dir in SEARCH_DIRECTORIES:

    print(f"\nSEARCH DIRECTORY: {search_dir}")
    print("Exists:", search_dir.exists())

    if not search_dir.exists():
        continue

    for file_path in search_dir.rglob("*"):

        if not file_path.is_file():
            continue

        if file_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue

        size_mb = file_path.stat().st_size / (1024 ** 2)

        dataset_files.append({
            "file_name": file_path.name,
            "relative_path": str(file_path.relative_to(PROJECT_ROOT)),
            "extension": file_path.suffix.lower(),
            "size_mb": round(size_mb, 3),
        })

if dataset_files:

    dataset_inventory = pd.DataFrame(dataset_files)

    dataset_inventory = dataset_inventory.sort_values(
        ["extension", "relative_path"]
    ).reset_index(drop=True)

    display(dataset_inventory)

else:
    print("\nNo supported dataset files found.")


# ------------------------------------------------------------
# INSPECT NOTEBOOK 11 EXPORT OPERATIONS
# ------------------------------------------------------------

NOTEBOOK_11_PATH = (
    PROJECT_ROOT / "notebooks" /
    "11_dl_dataset_preparation_and_leakage_audit.ipynb"
)

print("\n" + "=" * 90)
print("NOTEBOOK 11 DATASET EXPORT INSPECTION")
print("=" * 90)

export_matches = []

if NOTEBOOK_11_PATH.exists():

    with open(NOTEBOOK_11_PATH, "r", encoding="utf-8") as file:
        notebook_11 = json.load(file)

    export_terms = [
        "to_csv(",
        "to_parquet(",
        "to_pickle(",
        "joblib.dump(",
        "pickle.dump(",
        "to_feather(",
    ]

    for index, cell in enumerate(notebook_11["cells"]):

        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))

        matching_lines = [
            line.strip()
            for line in source.splitlines()
            if any(term in line.lower() for term in export_terms)
        ]

        if matching_lines:

            export_matches.append({
                "cell_index": index,
                "export_operations": matching_lines,
            })

    for item in export_matches:
        print(f"\nCELL INDEX: {item['cell_index']}")

        for line in item["export_operations"]:
            print(" ", line)

else:
    print("Notebook 11 not found.")


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("DISCOVERY SUMMARY")
print("=" * 90)

print("Dataset/artifact files found:", len(dataset_files))
print("Notebook export cells found:", len(export_matches))

print("\nCELL 14 COMPLETED.")


SAVED DATASET DISCOVERY

SEARCH DIRECTORY: d:\Git\Personal\ipl-player-performance-v2\data
Exists: True

SEARCH DIRECTORY: d:\Git\Personal\ipl-player-performance-v2\artifacts
Exists: True

SEARCH DIRECTORY: d:\Git\Personal\ipl-player-performance-v2\notebooks\artifacts
Exists: True


,file_name,relative_path,extension,size_mb
0,mlp_generalization_audit.csv,artifacts\dl_validation\mlp_generalization_aud...,.csv,0.005
1,mlp_test_summary.csv,artifacts\dl_validation\mlp_test_summary.csv,.csv,0.002
2,mlp_validation_summary.csv,artifacts\dl_validation\mlp_validation_summary...,.csv,0.002
3,selected_model_artifact_audit.csv,artifacts\final_reports\selected_model_artifac...,.csv,0.005
4,constrained_test_predictions.csv,artifacts\model_comparison\constrained_test_pr...,.csv,2.317
5,corrected_comparison_alignment_audit.csv,artifacts\model_comparison\corrected_compariso...,.csv,0.001
6,corrected_ml_dl_baseline_comparison.csv,artifacts\model_comparison\corrected_ml_dl_bas...,.csv,0.004
7,final_target_model_selection.csv,artifacts\model_comparison\final_target_model_...,.csv,0.002
8,ml_dl_test_comparison_20261102_161403.csv,artifacts\model_comparison\ml_dl_test_comparis...,.csv,0.001
9,ml_dl_test_comparison_20261102_161447.csv,artifacts\model_comparison\ml_dl_test_comparis...,.csv,0.001



NOTEBOOK 11 DATASET EXPORT INSPECTION

CELL INDEX: 72
  seed_results_df.to_csv(
  seed_summary.to_csv(

CELL INDEX: 92
  validation_summary.to_csv(

CELL INDEX: 94
  test_summary.to_csv(

CELL INDEX: 97
  generalization_df.to_csv(

CELL INDEX: 113
  comparison_df.to_csv(comparison_path, index=False)
  selection_df.to_csv(selection_path, index=False)

CELL INDEX: 114
  joblib.dump(
  joblib.dump(

CELL INDEX: 118
  corrected_comparison_df.to_csv(
  corrected_alignment_df.to_csv(

CELL INDEX: 119
  selection_df.to_csv(

CELL INDEX: 120
  inference_audit_df.to_csv(

CELL INDEX: 121
  prediction_output_df.to_csv(
  constraint_audit_df.to_csv(

CELL INDEX: 122
  artifact_audit_df.to_csv(

DISCOVERY SUMMARY
Dataset/artifact files found: 56
Notebook export cells found: 11

CELL 14 COMPLETED.



## 15. Notebook 11 Dataset Construction and Export Audit

### Objective
Trace how the original batting and bowling feature datasets were constructed and determine whether raw test features were saved.

### Inspection scope
- Dataset construction cells.
- Chronological split logic.
- Feature preprocessing.
- Test prediction export.
- Model artifact export.
- Dataset file inventory.

### Expected outcome
Identify the exact source required to reconstruct the original test feature DataFrames without changing the trained models or their evaluation results.

This cell is read-only.


In [15]:

# ============================================================
# CELL 15: NOTEBOOK 11 DATASET CONSTRUCTION AUDIT
# ============================================================

import re

assert "notebook_11" in globals(), "Run Cell 14 first."

print("=" * 90)
print("NOTEBOOK 11: DATASET CONSTRUCTION AUDIT")
print("=" * 90)

KEYWORDS = [
    "batting_features",
    "bowling_features",
    "build_target_datasets",
    "preprocess_target_datasets",
    "scale_targets",
    "train_test_split",
    "chronological",
    "prediction_output_df",
    "inference_audit_df",
    "to_csv",
    "to_parquet",
    "joblib.dump",
]

for index, cell in enumerate(notebook_11["cells"]):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    matched = [
        keyword for keyword in KEYWORDS
        if keyword.lower() in source.lower()
    ]

    if not matched:
        continue

    print("\n" + "=" * 90)
    print(f"CELL INDEX: {index}")
    print("MATCHED TERMS:", matched)
    print("-" * 90)
    print(source)


# ------------------------------------------------------------
# FILTER EXISTING FILE INVENTORY
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("FEATURE DATASET FILE INVENTORY")
print("=" * 90)

if "dataset_inventory" in globals() and not dataset_inventory.empty:

    pattern = (
        r"batting|bowling|feature|dataset|processed|"
        r"training|validation|test|prediction"
    )

    relevant_files = dataset_inventory[
        dataset_inventory["relative_path"].str.contains(
            pattern,
            case=False,
            na=False,
        )
    ].reset_index(drop=True)

    display(relevant_files)

    print("Relevant files:", len(relevant_files))

else:
    print("Dataset inventory unavailable. Run Cell 14 again.")


print("\nCELL 15 COMPLETED.")


NOTEBOOK 11: DATASET CONSTRUCTION AUDIT

CELL INDEX: 14
MATCHED TERMS: ['chronological']
------------------------------------------------------------------------------------------
# ============================================================
# 11. CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT
# ============================================================

import pandas as pd

# Shared chronological boundaries
TRAIN_END = pd.Timestamp("2022-04-11")
VALIDATION_END = pd.Timestamp("2024-05-01")
TEST_START = pd.Timestamp("2024-05-02")


def chronological_split(df, dataset_name):

    data = df.copy()

    # Ensure datetime format
    data["date"] = pd.to_datetime(data["date"], errors="coerce")

    # Validate dates
    if data["date"].isna().any():
        raise ValueError(f"{dataset_name}: Missing or invalid dates found.")

    # Sort chronologically
    data = data.sort_values("date").reset_index(drop=True)

    # Split without shuffling
    train = data[data["date"] <= TRAIN_END].copy()

,file_name,relative_path,extension,size_mb
0,mlp_generalization_audit.csv,artifacts\dl_validation\mlp_generalization_aud...,.csv,0.005
1,mlp_test_summary.csv,artifacts\dl_validation\mlp_test_summary.csv,.csv,0.002
2,mlp_validation_summary.csv,artifacts\dl_validation\mlp_validation_summary...,.csv,0.002
3,constrained_test_predictions.csv,artifacts\model_comparison\constrained_test_pr...,.csv,2.317
4,ml_dl_test_comparison_20261102_161403.csv,artifacts\model_comparison\ml_dl_test_comparis...,.csv,0.001
5,ml_dl_test_comparison_20261102_161447.csv,artifacts\model_comparison\ml_dl_test_comparis...,.csv,0.001
6,deliveries.csv,data\processed\deliveries.csv,.csv,29.228
7,batting_all_model_comparison.csv,data\processed\features\batting_all_model_comp...,.csv,0.001
8,batting_baseline_comparison.csv,data\processed\features\batting_baseline_compa...,.csv,0.000
9,batting_baseline_results.csv,data\processed\features\batting_baseline_resul...,.csv,0.000


Relevant files: 45

CELL 15 COMPLETED.


In [17]:

# CELL 16 — FINAL SOURCE DATASET SCHEMA AUDIT
# READ ONLY — NO FILE MODIFICATIONS

from pathlib import Path
import pandas as pd

# --------------------------------------------------
# 1. Locate project root safely
# --------------------------------------------------

def find_project_root(start_path=None):
    start_path = Path(start_path or Path.cwd()).resolve()

    candidates = [start_path, *start_path.parents]

    for candidate in candidates:
        if (
            (candidate / "paths.py").is_file()
            and (candidate / "environment.yml").is_file()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Project root not found. Expected paths.py, "
        "environment.yml and notebooks directory."
    )


PROJECT_ROOT = find_project_root()

print("PROJECT ROOT:", PROJECT_ROOT)
print("CURRENT WORKING DIRECTORY:", Path.cwd())

# --------------------------------------------------
# 2. Dataset paths
# --------------------------------------------------

files = {
    "batting": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "features"
        / "batting_model_dataset.csv"
    ),
    "bowling": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "features"
        / "bowling_model_dataset.csv"
    ),
}

# --------------------------------------------------
# 3. Inspect datasets
# --------------------------------------------------

for category, path in files.items():

    print("\n" + "=" * 75)
    print(f"{category.upper()} DATASET")
    print("=" * 75)

    if not path.is_file():
        print("FILE NOT FOUND:", path)
        continue

    df = pd.read_csv(path, nrows=5)

    print("Path:", path)
    print("Columns:", len(df.columns))
    print("Sample rows:", len(df))
    print("Column names:")
    print(df.columns.tolist())

    display(df.head(3))

print("\nCELL 16 COMPLETED — READ ONLY")


PROJECT ROOT: D:\Git\Personal\ipl-player-performance-v2
CURRENT WORKING DIRECTORY: d:\Git\Personal\ipl-player-performance-v2

BATTING DATASET
Path: D:\Git\Personal\ipl-player-performance-v2\data\processed\features\batting_model_dataset.csv
Columns: 56
Sample rows: 5
Column names:
['match_id', 'player', 'is_super_over', 'batting_team', 'runs_scored', 'balls_faced', 'fours', 'sixes', 'dismissals', 'retired_hurt_count', 'strike_rate', 'batting_average', 'innings_batted', 'not_out_innings', 'season', 'date', 'match_type', 'target_next_match_runs_scored', 'target_next_match_balls_faced', 'target_next_match_fours', 'target_next_match_sixes', 'target_next_match_strike_rate', 'target_next_match_dismissals', 'previous_match_date', 'days_since_previous_match', 'previous_runs_scored', 'runs_scored_rolling_mean_3', 'runs_scored_rolling_mean_5', 'runs_scored_rolling_mean_10', 'runs_scored_expanding_mean', 'previous_balls_faced', 'balls_faced_rolling_mean_3', 'balls_faced_rolling_mean_5', 'balls_fac

,match_id,player,is_super_over,batting_team,runs_scored,balls_faced,fours,sixes,dismissals,retired_hurt_count,...,strike_rate_rolling_mean_3,strike_rate_rolling_mean_5,strike_rate_rolling_mean_10,strike_rate_expanding_mean,previous_dismissals,dismissals_rolling_mean_3,dismissals_rolling_mean_5,dismissals_rolling_mean_10,dismissals_expanding_mean,matches_played_before
0,548346,A Ashish Reddy,False,Deccan Chargers,10,10,0,1,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,548352,A Ashish Reddy,False,Deccan Chargers,3,3,0,0,1,0,...,100.0,100.0,100.0,100.0,1.0,1.0,1.0,1.0,1.0,1
2,548359,A Ashish Reddy,False,Deccan Chargers,8,8,1,0,1,0,...,100.0,100.0,100.0,100.0,1.0,1.0,1.0,1.0,1.0,2



BOWLING DATASET
Path: D:\Git\Personal\ipl-player-performance-v2\data\processed\features\bowling_model_dataset.csv
Columns: 43
Sample rows: 5
Column names:
['match_id', 'player', 'is_super_over', 'runs_conceded', 'legal_balls', 'wides_conceded', 'noballs_conceded', 'innings_bowled', 'wickets', 'overs_bowled', 'economy_rate', 'bowling_average', 'bowling_strike_rate', 'season', 'date', 'match_type', 'target_next_match_wickets', 'target_next_match_runs_conceded', 'target_next_match_legal_balls', 'target_next_match_economy_rate', 'previous_match_date', 'days_since_previous_match', 'previous_wickets', 'wickets_rolling_mean_3', 'wickets_rolling_mean_5', 'wickets_rolling_mean_10', 'wickets_expanding_mean', 'previous_runs_conceded', 'runs_conceded_rolling_mean_3', 'runs_conceded_rolling_mean_5', 'runs_conceded_rolling_mean_10', 'runs_conceded_expanding_mean', 'previous_legal_balls', 'legal_balls_rolling_mean_3', 'legal_balls_rolling_mean_5', 'legal_balls_rolling_mean_10', 'legal_balls_expandin

,match_id,player,is_super_over,runs_conceded,legal_balls,wides_conceded,noballs_conceded,innings_bowled,wickets,overs_bowled,...,legal_balls_rolling_mean_3,legal_balls_rolling_mean_5,legal_balls_rolling_mean_10,legal_balls_expanding_mean,previous_economy_rate,economy_rate_rolling_mean_3,economy_rate_rolling_mean_5,economy_rate_rolling_mean_10,economy_rate_expanding_mean,matches_played_before
0,548341,A Ashish Reddy,False,32,24,0,0,1,2,4.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,548346,A Ashish Reddy,False,11,13,1,0,1,1,2.166667,...,24.0,24.0,24.0,24.0,8.000000,8.000000,8.000000,8.000000,8.000000,1
2,548348,A Ashish Reddy,False,32,18,0,1,1,1,3.000000,...,18.5,18.5,18.5,18.5,5.076923,6.538462,6.538462,6.538462,6.538462,2



CELL 16 COMPLETED — READ ONLY


In [18]:

# CELL 17 — COMPLETE MODEL DATASET AUDIT
# READ ONLY — NO FILE MODIFICATIONS

import pandas as pd
import numpy as np

datasets = {
    "batting": PROJECT_ROOT / "data/processed/features/batting_model_dataset.csv",
    "bowling": PROJECT_ROOT / "data/processed/features/bowling_model_dataset.csv",
}

IDENTIFIER_COLUMNS = {
    "match_id",
    "player",
    "date",
}

for category, path in datasets.items():

    print("\n" + "=" * 90)
    print(f"{category.upper()} DATASET AUDIT")
    print("=" * 90)

    df = pd.read_csv(path)

    target_columns = [
        col for col in df.columns
        if col.startswith("target_next_match_")
    ]

    identifier_columns = [
        col for col in df.columns
        if col in IDENTIFIER_COLUMNS
    ]

    candidate_features = [
        col for col in df.columns
        if col not in target_columns
        and col not in identifier_columns
    ]

    print("File:", path)
    print("Dataset shape:", df.shape)
    print("Rows:", len(df))
    print("Total columns:", len(df.columns))

    print("\nIDENTIFIERS")
    print(identifier_columns)

    print("\nTARGET COLUMNS")
    print(target_columns)

    print("\nCANDIDATE INPUT FEATURES")
    print("Count:", len(candidate_features))
    print(candidate_features)

    print("\nDATA TYPES")
    print(df.dtypes.value_counts())

    print("\nMISSING VALUES")
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    display(missing.to_frame("missing_count"))

    print("\nDUPLICATE MATCH-PLAYER KEYS")
    if {"match_id", "player"}.issubset(df.columns):
        duplicate_count = df.duplicated(
            subset=["match_id", "player"]
        ).sum()
        print("Duplicate rows:", duplicate_count)
    else:
        print("Required key columns unavailable.")

    print("\nTARGET VALUE SUMMARY")
    if target_columns:
        display(df[target_columns].describe().T)

    print("\nDATE RANGE")
    if "date" in df.columns:
        dates = pd.to_datetime(df["date"], errors="coerce")
        print("Minimum:", dates.min())
        print("Maximum:", dates.max())
        print("Invalid dates:", dates.isna().sum())

print("\nCELL 17 COMPLETED — READ ONLY")



BATTING DATASET AUDIT
File: D:\Git\Personal\ipl-player-performance-v2\data\processed\features\batting_model_dataset.csv
Dataset shape: (18083, 56)
Rows: 18083
Total columns: 56

IDENTIFIERS
['match_id', 'player', 'date']

TARGET COLUMNS
['target_next_match_runs_scored', 'target_next_match_balls_faced', 'target_next_match_fours', 'target_next_match_sixes', 'target_next_match_strike_rate', 'target_next_match_dismissals']

CANDIDATE INPUT FEATURES
Count: 47
['is_super_over', 'batting_team', 'runs_scored', 'balls_faced', 'fours', 'sixes', 'dismissals', 'retired_hurt_count', 'strike_rate', 'batting_average', 'innings_batted', 'not_out_innings', 'season', 'match_type', 'previous_match_date', 'days_since_previous_match', 'previous_runs_scored', 'runs_scored_rolling_mean_3', 'runs_scored_rolling_mean_5', 'runs_scored_rolling_mean_10', 'runs_scored_expanding_mean', 'previous_balls_faced', 'balls_faced_rolling_mean_3', 'balls_faced_rolling_mean_5', 'balls_faced_rolling_mean_10', 'balls_faced_ex

,missing_count
batting_average,3962
previous_strike_rate,688
strike_rate_rolling_mean_5,636
strike_rate_expanding_mean,636
strike_rate_rolling_mean_10,636
strike_rate_rolling_mean_3,636
fours_rolling_mean_5,634
fours_rolling_mean_10,634
previous_match_date,634
previous_runs_scored,634



DUPLICATE MATCH-PLAYER KEYS
Duplicate rows: 0

TARGET VALUE SUMMARY


,count,mean,std,min,25%,50%,75%,max
target_next_match_runs_scored,18083.0,20.627330,21.818108,0.0,4.000000,13.000000,31.0,175.0
target_next_match_balls_faced,18083.0,15.413869,13.502011,0.0,5.000000,11.000000,22.0,73.0
target_next_match_fours,18083.0,1.861804,2.339975,0.0,0.000000,1.000000,3.0,19.0
target_next_match_sixes,18083.0,0.854615,1.438846,0.0,0.000000,0.000000,1.0,17.0
target_next_match_strike_rate,18028.0,115.864130,69.049496,0.0,71.428571,114.285714,155.0,600.0
target_next_match_dismissals,18083.0,0.781176,0.413461,0.0,1.000000,1.000000,1.0,1.0



DATE RANGE
Minimum: 2008-04-18 00:00:00
Maximum: 2026-05-29 00:00:00
Invalid dates: 0

BOWLING DATASET AUDIT
File: D:\Git\Personal\ipl-player-performance-v2\data\processed\features\bowling_model_dataset.csv
Dataset shape: (14123, 43)
Rows: 14123
Total columns: 43

IDENTIFIERS
['match_id', 'player', 'date']

TARGET COLUMNS
['target_next_match_wickets', 'target_next_match_runs_conceded', 'target_next_match_legal_balls', 'target_next_match_economy_rate']

CANDIDATE INPUT FEATURES
Count: 36
['is_super_over', 'runs_conceded', 'legal_balls', 'wides_conceded', 'noballs_conceded', 'innings_bowled', 'wickets', 'overs_bowled', 'economy_rate', 'bowling_average', 'bowling_strike_rate', 'season', 'match_type', 'previous_match_date', 'days_since_previous_match', 'previous_wickets', 'wickets_rolling_mean_3', 'wickets_rolling_mean_5', 'wickets_rolling_mean_10', 'wickets_expanding_mean', 'previous_runs_conceded', 'runs_conceded_rolling_mean_3', 'runs_conceded_rolling_mean_5', 'runs_conceded_rolling_me

,missing_count
bowling_average,5884
bowling_strike_rate,5884
previous_economy_rate,504
wickets_rolling_mean_5,503
days_since_previous_match,503
previous_match_date,503
wickets_rolling_mean_10,503
wickets_rolling_mean_3,503
wickets_expanding_mean,503
previous_wickets,503



DUPLICATE MATCH-PLAYER KEYS
Duplicate rows: 0

TARGET VALUE SUMMARY


,count,mean,std,min,25%,50%,75%,max
target_next_match_wickets,14123.0,0.920980,0.999072,0.0,0.000000,1.00,1.000000,6.0
target_next_match_runs_conceded,14123.0,26.876372,11.173536,0.0,19.000000,26.00,34.000000,76.0
target_next_match_legal_balls,14123.0,19.426821,6.074533,0.0,18.000000,24.00,24.000000,25.0
target_next_match_economy_rate,14122.0,8.622590,3.309615,0.0,6.333333,8.25,10.333333,54.0



DATE RANGE
Minimum: 2008-04-18 00:00:00
Maximum: 2026-05-29 00:00:00
Invalid dates: 0

CELL 17 COMPLETED — READ ONLY


In [19]:

# CELL 18 — SAVED MODEL INPUT SCHEMA AUDIT
# READ ONLY — DO NOT RETRAIN OR MODIFY ARTIFACTS

import json
import joblib
from pathlib import Path

ROOT = Path(PROJECT_ROOT)

DL_TARGETS = {
    "batting": [
        "target_next_match_balls_faced",
        "target_next_match_fours",
        "target_next_match_sixes",
        "target_next_match_strike_rate",
        "target_next_match_dismissals",
    ],
    "bowling": [
        "target_next_match_runs_conceded",
        "target_next_match_legal_balls",
        "target_next_match_economy_rate",
    ],
}

print("=" * 90)
print("DEEP LEARNING PREPROCESSOR INPUT SCHEMA")
print("=" * 90)

for category, targets in DL_TARGETS.items():

    for target in targets:

        artifact_dir = (
            ROOT / "artifacts" / "dl_models" /
            category / target
        )

        preprocessor_path = artifact_dir / "preprocessor.joblib"

        print(f"\n{category.upper()} — {target}")
        print("-" * 75)

        if not preprocessor_path.exists():
            print("PREPROCESSOR NOT FOUND:", preprocessor_path)
            continue

        preprocessor = joblib.load(preprocessor_path)

        raw_features = getattr(
            preprocessor, "feature_names_in_", None
        )

        transformed_features = preprocessor.get_feature_names_out()

        print("Raw input feature count:",
              len(raw_features) if raw_features is not None else "Unavailable")

        if raw_features is not None:
            print("Raw input features:")
            print(list(raw_features))

        print("Transformed feature count:", len(transformed_features))
        print("Transformed feature names:")
        print(list(transformed_features))

        scaler_path = artifact_dir / "target_scaler.joblib"

        if scaler_path.exists():
            scaler = joblib.load(scaler_path)
            print("Target scaler:", type(scaler).__name__)
            print("Scaler mean:", getattr(scaler, "mean_", None))
            print("Scaler scale:", getattr(scaler, "scale_", None))


print("\n" + "=" * 90)
print("SELECTED MACHINE LEARNING ARTIFACT DISCOVERY")
print("=" * 90)

model_root = ROOT / "models"

ml_candidates = [
    path for path in model_root.rglob("*.joblib")
    if "original_gb_model" in path.name.lower()
    or "wicket" in path.name.lower()
]

if not ml_candidates:
    print("No matching ML model artifacts found.")
else:
    for model_path in ml_candidates:

        print("\nMODEL FILE:", model_path.relative_to(ROOT))

        try:
            model = joblib.load(model_path)

            print("Estimator type:", type(model).__name__)
            print("n_features_in_:",
                  getattr(model, "n_features_in_", "Unavailable"))

            feature_names = getattr(
                model, "feature_names_in_", None
            )

            print("feature_names_in_:")
            print(
                list(feature_names)
                if feature_names is not None
                else "Unavailable"
            )

        except Exception as exc:
            print("Could not inspect:", repr(exc))


print("\n" + "=" * 90)
print("MODEL SELECTION FILES")
print("=" * 90)

selection_files = [
    ROOT / "artifacts/model_comparison/final_target_model_selection.json",
    ROOT / "artifacts/model_comparison/final_target_model_selection.csv",
]

for file_path in selection_files:

    if not file_path.exists():
        print("NOT FOUND:", file_path.relative_to(ROOT))
        continue

    print("\nFILE:", file_path.relative_to(ROOT))

    if file_path.suffix == ".json":
        with open(file_path, "r", encoding="utf-8") as f:
            print(json.dumps(json.load(f), indent=2, default=str))

    elif file_path.suffix == ".csv":
        display(pd.read_csv(file_path))


print("\nCELL 18 COMPLETED — READ ONLY")


DEEP LEARNING PREPROCESSOR INPUT SCHEMA

BATTING — target_next_match_balls_faced
---------------------------------------------------------------------------
Raw input feature count: 42
Raw input features:
['batting_team', 'runs_scored', 'balls_faced', 'fours', 'sixes', 'dismissals', 'retired_hurt_count', 'strike_rate', 'not_out_innings', 'season', 'days_since_previous_match', 'previous_runs_scored', 'runs_scored_rolling_mean_3', 'runs_scored_rolling_mean_5', 'runs_scored_rolling_mean_10', 'runs_scored_expanding_mean', 'previous_balls_faced', 'balls_faced_rolling_mean_3', 'balls_faced_rolling_mean_5', 'balls_faced_rolling_mean_10', 'balls_faced_expanding_mean', 'previous_fours', 'fours_rolling_mean_3', 'fours_rolling_mean_5', 'fours_rolling_mean_10', 'fours_expanding_mean', 'previous_sixes', 'sixes_rolling_mean_3', 'sixes_rolling_mean_5', 'sixes_rolling_mean_10', 'sixes_expanding_mean', 'previous_strike_rate', 'strike_rate_rolling_mean_3', 'strike_rate_rolling_mean_5', 'strike_rate_roll

,target,selected_model,model_type,test_rows,selected_MAE,selected_RMSE,selected_R2,selected_bias,baseline_MAE,baseline_RMSE,MAE_improvement_pct,selection_reason
0,target_next_match_runs_scored,batting_original,Machine Learning,2375,16.643802,22.459996,0.111593,-2.564665,18.094229,24.069572,8.015967,Best balanced RMSE and R2; batting_sqrt has ma...
1,target_next_match_balls_faced,DL_MLP,Deep Learning,2375,9.364080,12.018598,0.127566,-0.350296,10.423991,12.874722,10.167991,Lower MAE and RMSE than training-mean baseline.
2,target_next_match_fours,DL_MLP,Deep Learning,2375,1.671235,2.295062,0.111262,-0.309986,1.827386,2.449228,8.545037,Lower MAE and RMSE than training-mean baseline.
3,target_next_match_sixes,DL_MLP,Deep Learning,2375,1.157966,1.789982,-0.022558,-0.474558,1.210932,1.835988,4.373940,"Improves MAE, RMSE and R2 over training-mean b..."
4,target_next_match_strike_rate,DL_MLP,Deep Learning,2373,58.529766,76.054662,-0.023350,-15.874293,61.432419,78.134748,4.724955,Better than training-mean baseline; R2 remains...
5,target_next_match_dismissals,DL_MLP,Deep Learning,2375,0.272703,0.401838,0.039667,0.062993,0.340575,0.410122,19.928619,Improves MAE and RMSE over training-mean basel...
6,target_next_match_wickets,bowling_wickets,Machine Learning,1805,0.763657,0.993646,0.023595,-0.070748,0.782052,1.008695,2.352126,"ML beats DL and baseline on MAE, RMSE and R2."
7,target_next_match_runs_conceded,DL_MLP,Deep Learning,1805,9.227933,11.577274,0.103529,-1.037495,10.632706,13.223549,13.211808,Better MAE and RMSE than both baselines.
8,target_next_match_legal_balls,DL_MLP,Deep Learning,1805,3.872398,5.370624,0.193824,1.346289,5.053235,5.982715,23.367928,Substantial improvement over both baselines.
9,target_next_match_economy_rate,DL_MLP,Deep Learning,1805,2.821380,4.058312,-0.117294,-1.334910,2.933666,4.203205,3.827500,Marginally lower MAE than saved dummy and bett...



CELL 18 COMPLETED — READ ONLY


In [20]:

# CELL 19 — SELECTED MODEL / DATASET CONTRACT VALIDATION
# READ ONLY — NO TRAINING OR ARTIFACT MODIFICATION

from pathlib import Path
import joblib
import pandas as pd

ROOT = Path(PROJECT_ROOT)

DATASETS = {
    "batting": ROOT / "data/processed/features/batting_model_dataset.csv",
    "bowling": ROOT / "data/processed/features/bowling_model_dataset.csv",
}

SELECTED_MODELS = {
    "target_next_match_runs_scored": {
        "category": "batting",
        "model_name": "batting_original",
        "artifact": ROOT / "models/batting/original_gb_model.joblib",
        "type": "ML",
    },
    "target_next_match_balls_faced": {
        "category": "batting",
        "model_name": "DL_MLP",
        "artifact": ROOT / "artifacts/dl_models/batting/target_next_match_balls_faced/preprocessor.joblib",
        "type": "DL",
    },
    "target_next_match_fours": {
        "category": "batting",
        "model_name": "DL_MLP",
        "artifact": ROOT / "artifacts/dl_models/batting/target_next_match_fours/preprocessor.joblib",
        "type": "DL",
    },
    "target_next_match_sixes": {
        "category": "batting",
        "model_name": "DL_MLP",
        "artifact": ROOT / "artifacts/dl_models/batting/target_next_match_sixes/preprocessor.joblib",
        "type": "DL",
    },
    "target_next_match_strike_rate": {
        "category": "batting",
        "model_name": "DL_MLP",
        "artifact": ROOT / "artifacts/dl_models/batting/target_next_match_strike_rate/preprocessor.joblib",
        "type": "DL",
    },
    "target_next_match_dismissals": {
        "category": "batting",
        "model_name": "DL_MLP",
        "artifact": ROOT / "artifacts/dl_models/batting/target_next_match_dismissals/preprocessor.joblib",
        "type": "DL",
    },
    "target_next_match_wickets": {
        "category": "bowling",
        "model_name": "bowling_wickets",
        "artifact": ROOT / "models/bowling/wickets_model.joblib",
        "type": "ML",
    },
    "target_next_match_runs_conceded": {
        "category": "bowling",
        "model_name": "DL_MLP",
        "artifact": ROOT / "artifacts/dl_models/bowling/target_next_match_runs_conceded/preprocessor.joblib",
        "type": "DL",
    },
    "target_next_match_legal_balls": {
        "category": "bowling",
        "model_name": "DL_MLP",
        "artifact": ROOT / "artifacts/dl_models/bowling/target_next_match_legal_balls/preprocessor.joblib",
        "type": "DL",
    },
    "target_next_match_economy_rate": {
        "category": "bowling",
        "model_name": "DL_MLP",
        "artifact": ROOT / "artifacts/dl_models/bowling/target_next_match_economy_rate/preprocessor.joblib",
        "type": "DL",
    },
}

# Load source datasets once.
datasets = {}

for category, path in DATASETS.items():
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")

    datasets[category] = pd.read_csv(path)
    print(
        f"{category.upper()} dataset: "
        f"{datasets[category].shape}"
    )

results = []

for target, config in SELECTED_MODELS.items():

    category = config["category"]
    df = datasets[category]
    artifact_path = config["artifact"]

    if not artifact_path.exists():
        print(f"ARTIFACT MISSING: {artifact_path}")
        continue

    artifact = joblib.load(artifact_path)

    # DL artifact is the fitted preprocessor.
    # ML artifact is the fitted estimator pipeline.
    required_features = getattr(
        artifact, "feature_names_in_", None
    )

    if required_features is None:
        raise ValueError(
            f"Artifact does not expose feature_names_in_: {target}"
        )

    required_features = list(required_features)

    missing_columns = [
        col for col in required_features
        if col not in df.columns
    ]

    target_in_features = target in required_features

    missing_value_count = (
        int(df[required_features].isna().sum().sum())
        if not missing_columns else None
    )

    missing_value_columns = (
        df[required_features].isna().sum()
        .loc[lambda x: x > 0]
        .to_dict()
        if not missing_columns else {}
    )

    results.append({
        "target": target,
        "model": config["model_name"],
        "type": config["type"],
        "required_features": len(required_features),
        "missing_columns": missing_columns,
        "target_in_features": target_in_features,
        "missing_input_values": missing_value_count,
        "missing_value_columns": missing_value_columns,
        "contract_match": (
            len(missing_columns) == 0
            and not target_in_features
        ),
    })

result_df = pd.DataFrame(results)

print("\n" + "=" * 100)
print("MODEL / DATASET CONTRACT SUMMARY")
print("=" * 100)

display(
    result_df[
        [
            "target",
            "model",
            "type",
            "required_features",
            "missing_columns",
            "target_in_features",
            "missing_input_values",
            "contract_match",
        ]
    ]
)

print("\n" + "=" * 100)
print("MISSING INPUT VALUE DETAILS")
print("=" * 100)

for row in results:
    if row["missing_value_columns"]:
        print(f"\n{row['target']}:")
        print(row["missing_value_columns"])

print("\n" + "=" * 100)
print("FINAL CONTRACT AUDIT")
print("=" * 100)

print("Selected models checked:", len(results))
print(
    "Contracts matched:",
    sum(r["contract_match"] for r in results)
)
print(
    "Contracts failed:",
    sum(not r["contract_match"] for r in results)
)

print("\nCELL 19 COMPLETED — READ ONLY")


BATTING dataset: (18083, 56)
BOWLING dataset: (14123, 43)

MODEL / DATASET CONTRACT SUMMARY


,target,model,type,required_features,missing_columns,target_in_features,missing_input_values,contract_match
0,target_next_match_runs_scored,batting_original,ML,43,[],False,23735,True
1,target_next_match_balls_faced,DL_MLP,DL,42,[],False,19773,True
2,target_next_match_fours,DL_MLP,DL,42,[],False,19773,True
3,target_next_match_sixes,DL_MLP,DL,42,[],False,19773,True
4,target_next_match_strike_rate,DL_MLP,DL,42,[],False,19773,True
5,target_next_match_dismissals,DL_MLP,DL,42,[],False,19773,True
6,target_next_match_wickets,bowling_wickets,ML,35,[],False,22333,True
7,target_next_match_runs_conceded,DL_MLP,DL,30,[],False,10565,True
8,target_next_match_legal_balls,DL_MLP,DL,30,[],False,10565,True
9,target_next_match_economy_rate,DL_MLP,DL,30,[],False,10565,True



MISSING INPUT VALUE DETAILS

target_next_match_runs_scored:
{'strike_rate': 57, 'batting_average': 3962, 'days_since_previous_match': 634, 'previous_runs_scored': 634, 'runs_scored_rolling_mean_3': 634, 'runs_scored_rolling_mean_5': 634, 'runs_scored_rolling_mean_10': 634, 'runs_scored_expanding_mean': 634, 'previous_balls_faced': 634, 'balls_faced_rolling_mean_3': 634, 'balls_faced_rolling_mean_5': 634, 'balls_faced_rolling_mean_10': 634, 'balls_faced_expanding_mean': 634, 'previous_fours': 634, 'fours_rolling_mean_3': 634, 'fours_rolling_mean_5': 634, 'fours_rolling_mean_10': 634, 'fours_expanding_mean': 634, 'previous_sixes': 634, 'sixes_rolling_mean_3': 634, 'sixes_rolling_mean_5': 634, 'sixes_rolling_mean_10': 634, 'sixes_expanding_mean': 634, 'previous_strike_rate': 688, 'strike_rate_rolling_mean_3': 636, 'strike_rate_rolling_mean_5': 636, 'strike_rate_rolling_mean_10': 636, 'strike_rate_expanding_mean': 636, 'previous_dismissals': 634, 'dismissals_rolling_mean_3': 634, 'dismiss

In [21]:

# CELL 20 — SAVED PREPROCESSING / IMPUTATION AUDIT
# READ ONLY — DOES NOT MODIFY OR RETRAIN ARTIFACTS

from pathlib import Path
import joblib

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

ROOT = Path(PROJECT_ROOT)

ARTIFACTS = {
    "Batting ML — runs scored":
        ROOT / "models/batting/original_gb_model.joblib",

    "Bowling ML — wickets":
        ROOT / "models/bowling/wickets_model.joblib",

    "Batting DL — balls faced":
        ROOT / "artifacts/dl_models/batting/target_next_match_balls_faced/preprocessor.joblib",

    "Bowling DL — wickets":
        ROOT / "artifacts/dl_models/bowling/target_next_match_wickets/preprocessor.joblib",
}


def inspect_object(obj, indent=0, visited=None):
    """Recursively inspect fitted sklearn preprocessing components."""
    if visited is None:
        visited = set()

    prefix = " " * indent
    object_id = id(obj)

    if object_id in visited:
        return

    visited.add(object_id)

    print(f"{prefix}Object: {type(obj).__name__}")

    if isinstance(obj, SimpleImputer):
        print(f"{prefix}  strategy: {obj.strategy}")
        print(f"{prefix}  missing_values: {obj.missing_values}")

        if hasattr(obj, "statistics_"):
            print(f"{prefix}  fitted: YES")
            print(f"{prefix}  statistics count: {len(obj.statistics_)}")
            print(f"{prefix}  statistics: {obj.statistics_}")
        else:
            print(f"{prefix}  fitted: NO")

    elif isinstance(obj, Pipeline):
        print(f"{prefix}  steps: {list(obj.named_steps.keys())}")

        for step_name, step_obj in obj.steps:
            print(f"{prefix}  Step: {step_name}")
            inspect_object(step_obj, indent + 4, visited)

    elif isinstance(obj, ColumnTransformer):
        print(f"{prefix}  transformers:")

        for name, transformer, columns in obj.transformers:
            print(f"{prefix}  Transformer: {name}")
            print(f"{prefix}  Columns: {columns}")

            if transformer not in ("drop", "passthrough"):
                inspect_object(
                    transformer,
                    indent + 4,
                    visited
                )

        print(
            f"{prefix}  fitted: "
            f"{hasattr(obj, 'transformers_')}"
        )

    else:
        print(
            f"{prefix}  fitted feature count: "
            f"{getattr(obj, 'n_features_in_', 'Not exposed')}"
        )


for label, artifact_path in ARTIFACTS.items():

    print("\n" + "=" * 100)
    print(label)
    print("=" * 100)

    if not artifact_path.exists():
        print("ARTIFACT NOT FOUND:", artifact_path)
        continue

    artifact = joblib.load(artifact_path)

    inspect_object(artifact)

print("\nCELL 20 COMPLETED — READ ONLY")



Batting ML — runs scored
Object: Pipeline
  steps: ['preprocessor', 'model']
  Step: preprocessor
    Object: ColumnTransformer
      transformers:
      Transformer: numerical
      Columns: ['runs_scored', 'balls_faced', 'fours', 'sixes', 'dismissals', 'retired_hurt_count', 'strike_rate', 'batting_average', 'not_out_innings', 'days_since_previous_match', 'previous_runs_scored', 'runs_scored_rolling_mean_3', 'runs_scored_rolling_mean_5', 'runs_scored_rolling_mean_10', 'runs_scored_expanding_mean', 'previous_balls_faced', 'balls_faced_rolling_mean_3', 'balls_faced_rolling_mean_5', 'balls_faced_rolling_mean_10', 'balls_faced_expanding_mean', 'previous_fours', 'fours_rolling_mean_3', 'fours_rolling_mean_5', 'fours_rolling_mean_10', 'fours_expanding_mean', 'previous_sixes', 'sixes_rolling_mean_3', 'sixes_rolling_mean_5', 'sixes_rolling_mean_10', 'sixes_expanding_mean', 'previous_strike_rate', 'strike_rate_rolling_mean_3', 'strike_rate_rolling_mean_5', 'strike_rate_rolling_mean_10', 'stri

In [23]:

# CELL 21 — ACTUAL FITTED TRANSFORMER AUDIT
# READ ONLY — SUPPORTS NUMERIC AND CATEGORICAL IMPUTERS

from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

ROOT = Path(PROJECT_ROOT)

ARTIFACTS = {
    "Batting ML — runs scored":
        ROOT / "models/batting/original_gb_model.joblib",

    "Bowling ML — wickets":
        ROOT / "models/bowling/wickets_model.joblib",

    "Batting DL — balls faced":
        ROOT / "artifacts/dl_models/batting/target_next_match_balls_faced/preprocessor.joblib",

    "Bowling DL — wickets":
        ROOT / "artifacts/dl_models/bowling/target_next_match_wickets/preprocessor.joblib",
}


def inspect_fitted_component(obj, name, indent=0):

    prefix = " " * indent

    print(f"{prefix}{name}: {type(obj).__name__}")

    if isinstance(obj, Pipeline):

        print(f"{prefix}  steps: {list(obj.named_steps.keys())}")

        for step_name, step_obj in obj.steps:
            inspect_fitted_component(
                step_obj,
                step_name,
                indent + 4
            )

    elif isinstance(obj, ColumnTransformer):

        fitted_transformers = getattr(obj, "transformers_", None)

        if fitted_transformers is None:
            print(f"{prefix}  STATUS: NOT FITTED")
            return

        print(f"{prefix}  fitted: YES")
        print(
            f"{prefix}  n_features_in_: "
            f"{getattr(obj, 'n_features_in_', 'Unavailable')}"
        )

        for transformer_name, transformer, columns in fitted_transformers:

            print(f"\n{prefix}  Transformer: {transformer_name}")
            print(f"{prefix}  Input columns: {columns}")

            if isinstance(transformer, str):
                print(f"{prefix}  Transformer mode: {transformer}")
                continue

            inspect_fitted_component(
                transformer,
                transformer_name,
                indent + 4
            )

        if hasattr(obj, "get_feature_names_out"):
            try:
                names = obj.get_feature_names_out()
                print(f"{prefix}  Output feature count: {len(names)}")
            except Exception as exc:
                print(f"{prefix}  Feature-name error: {exc}")

    elif isinstance(obj, SimpleImputer):

        fitted = hasattr(obj, "statistics_")

        print(f"{prefix}  fitted: {fitted}")
        print(f"{prefix}  strategy: {obj.strategy}")

        if fitted:

            stats = np.asarray(obj.statistics_)

            print(f"{prefix}  statistics count: {len(stats)}")
            print(f"{prefix}  statistics dtype: {stats.dtype}")

            if np.issubdtype(stats.dtype, np.number):

                print(
                    f"{prefix}  statistics finite: "
                    f"{np.isfinite(stats).all()}"
                )

            else:

                valid = pd.notna(stats)

                print(
                    f"{prefix}  statistics non-missing: "
                    f"{valid.all()}"
                )

            print(f"{prefix}  statistics: {stats}")

    elif isinstance(obj, StandardScaler):

        fitted = hasattr(obj, "mean_") and hasattr(obj, "scale_")

        print(f"{prefix}  fitted: {fitted}")

        if fitted:

            print(f"{prefix}  n_features_in_: {obj.n_features_in_}")
            print(f"{prefix}  mean count: {len(obj.mean_)}")
            print(f"{prefix}  scale count: {len(obj.scale_)}")

            print(
                f"{prefix}  mean finite: "
                f"{np.isfinite(obj.mean_).all()}"
            )

            print(
                f"{prefix}  scale finite: "
                f"{np.isfinite(obj.scale_).all()}"
            )

    elif isinstance(obj, OneHotEncoder):

        fitted = hasattr(obj, "categories_")

        print(f"{prefix}  fitted: {fitted}")

        if fitted:

            print(
                f"{prefix}  encoded input features: "
                f"{len(obj.categories_)}"
            )

            print(
                f"{prefix}  categories per feature: "
                f"{[len(x) for x in obj.categories_]}"
            )


for label, path in ARTIFACTS.items():

    print("\n" + "=" * 100)
    print(label)
    print("=" * 100)

    if not path.exists():
        print("ARTIFACT NOT FOUND:", path)
        continue

    artifact = joblib.load(path)

    if isinstance(artifact, Pipeline):
        preprocessor = artifact.named_steps.get("preprocessor")
    else:
        preprocessor = artifact

    if preprocessor is None:
        print("PREPROCESSOR NOT FOUND")
        continue

    inspect_fitted_component(preprocessor, "preprocessor")

print("\nCELL 21 COMPLETED — READ ONLY")



Batting ML — runs scored
preprocessor: ColumnTransformer
  fitted: YES
  n_features_in_: 43

  Transformer: numerical
  Input columns: ['runs_scored', 'balls_faced', 'fours', 'sixes', 'dismissals', 'retired_hurt_count', 'strike_rate', 'batting_average', 'not_out_innings', 'days_since_previous_match', 'previous_runs_scored', 'runs_scored_rolling_mean_3', 'runs_scored_rolling_mean_5', 'runs_scored_rolling_mean_10', 'runs_scored_expanding_mean', 'previous_balls_faced', 'balls_faced_rolling_mean_3', 'balls_faced_rolling_mean_5', 'balls_faced_rolling_mean_10', 'balls_faced_expanding_mean', 'previous_fours', 'fours_rolling_mean_3', 'fours_rolling_mean_5', 'fours_rolling_mean_10', 'fours_expanding_mean', 'previous_sixes', 'sixes_rolling_mean_3', 'sixes_rolling_mean_5', 'sixes_rolling_mean_10', 'sixes_expanding_mean', 'previous_strike_rate', 'strike_rate_rolling_mean_3', 'strike_rate_rolling_mean_5', 'strike_rate_rolling_mean_10', 'strike_rate_expanding_mean', 'previous_dismissals', 'dismissa

In [36]:

# CELL 22 — PRODUCTION INFERENCE SMOKE TEST
# READ ONLY — NO RETRAINING OR ARTIFACT MODIFICATION

from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

ROOT = Path(PROJECT_ROOT)

# ------------------------------------------------------------
# 1. Selected production model mapping
# ------------------------------------------------------------

MODEL_CONFIG = {
    "target_next_match_runs_scored": {
        "category": "batting",
        "type": "ML",
        "model_path": ROOT / "models/batting/original_gb_model.joblib",
    },
    "target_next_match_balls_faced": {
        "category": "batting",
        "type": "DL",
    },
    "target_next_match_fours": {
        "category": "batting",
        "type": "DL",
    },
    "target_next_match_sixes": {
        "category": "batting",
        "type": "DL",
    },
    "target_next_match_strike_rate": {
        "category": "batting",
        "type": "DL",
    },
    "target_next_match_dismissals": {
        "category": "batting",
        "type": "DL",
    },
    "target_next_match_wickets": {
        "category": "bowling",
        "type": "ML",
        "model_path": ROOT / "models/bowling/wickets_model.joblib",
    },
    "target_next_match_runs_conceded": {
        "category": "bowling",
        "type": "DL",
    },
    "target_next_match_legal_balls": {
        "category": "bowling",
        "type": "DL",
    },
    "target_next_match_economy_rate": {
        "category": "bowling",
        "type": "DL",
    },
}

DATASET_PATHS = {
    "batting": ROOT / "data/processed/features/batting_model_dataset.csv",
    "bowling": ROOT / "data/processed/features/bowling_model_dataset.csv",
}

SAMPLE_SIZE = 10

# ------------------------------------------------------------
# 2. Load source datasets
# ------------------------------------------------------------

datasets = {
    category: pd.read_csv(path)
    for category, path in DATASET_PATHS.items()
}

print("DATASETS LOADED")

for category, df in datasets.items():
    print(f"{category}: {df.shape}")

# ------------------------------------------------------------
# 3. Execute selected models
# ------------------------------------------------------------

inference_results = []

for target, config in MODEL_CONFIG.items():

    category = config["category"]
    model_type = config["type"]
    df = datasets[category]

    print("\n" + "=" * 90)
    print(f"TARGET: {target}")
    print(f"SELECTED MODEL: {model_type}")
    print("=" * 90)

    try:

        # ----------------------------------------------------
        # ML INFERENCE
        # ----------------------------------------------------

        if model_type == "ML":

            model_path = config["model_path"]

            if not model_path.is_file():
                raise FileNotFoundError(model_path)

            model = joblib.load(model_path)

            required_features = list(model.feature_names_in_)

            missing = [
                col for col in required_features
                if col not in df.columns
            ]

            if missing:
                raise ValueError(
                    f"Missing raw features: {missing}"
                )

            X = df.loc[:, required_features].head(SAMPLE_SIZE)

            predictions = np.asarray(model.predict(X)).reshape(-1)

            transformed_count = model.named_steps[
                "preprocessor"
            ].get_feature_names_out().shape[0]

            expected_count = model.named_steps[
                "model"
            ].n_features_in_

            if transformed_count != expected_count:
                raise ValueError(
                    f"Transformed dimension mismatch: "
                    f"{transformed_count} != {expected_count}"
                )

        # ----------------------------------------------------
        # DL INFERENCE
        # ----------------------------------------------------

        else:

            artifact_dir = (
                ROOT / "artifacts" / "dl_models"
                / category / target
            )

            model_path = artifact_dir / "model.keras"
            preprocessor_path = artifact_dir / "preprocessor.joblib"
            scaler_path = artifact_dir / "target_scaler.joblib"

            for path in [
                model_path,
                preprocessor_path,
                scaler_path,
            ]:
                if not path.is_file():
                    raise FileNotFoundError(path)

            preprocessor = joblib.load(preprocessor_path)
            target_scaler = joblib.load(scaler_path)

            required_features = list(
                preprocessor.feature_names_in_
            )

            missing = [
                col for col in required_features
                if col not in df.columns
            ]

            if missing:
                raise ValueError(
                    f"Missing raw features: {missing}"
                )

            X = df.loc[:, required_features].head(SAMPLE_SIZE)

            # Transform using the saved fitted preprocessor.
            X_transformed = preprocessor.transform(X)

            X_transformed = np.asarray(
                X_transformed,
                dtype=np.float32
            )

            # Validate model input dimension.
            keras_model = tf.keras.models.load_model(
                model_path,
                compile=False
            )

            expected_count = int(
                keras_model.input_shape[-1]
            )

            actual_count = X_transformed.shape[1]

            if actual_count != expected_count:
                raise ValueError(
                    f"DL input dimension mismatch: "
                    f"{actual_count} != {expected_count}"
                )

            # Predict in standardized target space.
            scaled_predictions = keras_model.predict(
                X_transformed,
                verbose=0
            )

            scaled_predictions = np.asarray(
                scaled_predictions
            ).reshape(-1, 1)

            # Convert back to original cricket-statistic units.
            predictions = target_scaler.inverse_transform(
                scaled_predictions
            ).reshape(-1)

            transformed_count = actual_count

        # ----------------------------------------------------
        # 4. Prediction validation
        # ----------------------------------------------------

        finite = bool(np.isfinite(predictions).all())

        nonnegative = bool((predictions >= 0).all())

        expected_rows = min(SAMPLE_SIZE, len(df))

        shape_valid = len(predictions) == expected_rows

        if not finite:
            raise ValueError("Non-finite predictions detected.")

        if not shape_valid:
            raise ValueError(
                f"Prediction row count mismatch: "
                f"{len(predictions)} != {expected_rows}"
            )

        # Negative values are reported, not silently clipped.
        status = (
            "PASS"
            if finite and shape_valid and nonnegative
            else "WARNING"
        )

        inference_results.append({
            "target": target,
            "model_type": model_type,
            "sample_rows": len(X),
            "raw_features": len(required_features),
            "transformed_features": transformed_count,
            "prediction_count": len(predictions),
            "finite_predictions": finite,
            "nonnegative_predictions": nonnegative,
            "prediction_min": float(np.min(predictions)),
            "prediction_max": float(np.max(predictions)),
            "prediction_mean": float(np.mean(predictions)),
            "status": status,
        })

        print("Raw features:", len(required_features))
        print("Transformed features:", transformed_count)
        print("Prediction count:", len(predictions))
        print("Finite predictions:", finite)
        print("Non-negative predictions:", nonnegative)
        print("Prediction range:",
              float(np.min(predictions)),
              "to",
              float(np.max(predictions)))
        print("Status:", status)

    except Exception as exc:

        inference_results.append({
            "target": target,
            "model_type": model_type,
            "sample_rows": 0,
            "raw_features": None,
            "transformed_features": None,
            "prediction_count": 0,
            "finite_predictions": False,
            "nonnegative_predictions": False,
            "prediction_min": None,
            "prediction_max": None,
            "prediction_mean": None,
            "status": "FAIL",
            "error": repr(exc),
        })

        print("INFERENCE FAILED:", repr(exc))

# ------------------------------------------------------------
# 5. Consolidated report
# ------------------------------------------------------------

inference_report_df = pd.DataFrame(inference_results)

print("\n" + "=" * 100)
print("PRODUCTION INFERENCE SMOKE TEST SUMMARY")
print("=" * 100)

display(inference_report_df)

print("Models tested:", len(inference_report_df))
print(
    "Passed:",
    (inference_report_df["status"] == "PASS").sum()
)
print(
    "Warnings:",
    (inference_report_df["status"] == "WARNING").sum()
)
print(
    "Failed:",
    (inference_report_df["status"] == "FAIL").sum()
)

print("\nCELL 22 COMPLETED — READ ONLY")


DATASETS LOADED
batting: (18083, 56)
bowling: (14123, 43)

TARGET: target_next_match_runs_scored
SELECTED MODEL: ML
Raw features: 43
Transformed features: 74
Prediction count: 10
Finite predictions: True
Non-negative predictions: True
Prediction range: 7.985569144138036 to 14.15448473509751
Status: PASS

TARGET: target_next_match_balls_faced
SELECTED MODEL: DL
Raw features: 42
Transformed features: 59
Prediction count: 10
Finite predictions: True
Non-negative predictions: True
Prediction range: 6.399984359741211 to 11.329374313354492
Status: PASS

TARGET: target_next_match_fours
SELECTED MODEL: DL
Raw features: 42
Transformed features: 59
Prediction count: 10
Finite predictions: True
Non-negative predictions: True
Prediction range: 0.67821204662323 to 1.2342395782470703
Status: PASS

TARGET: target_next_match_sixes
SELECTED MODEL: DL
Raw features: 42
Transformed features: 59
Prediction count: 10
Finite predictions: True
Non-negative predictions: True
Prediction range: 0.252925038337707

,target,model_type,sample_rows,raw_features,transformed_features,prediction_count,finite_predictions,nonnegative_predictions,prediction_min,prediction_max,prediction_mean,status
0,target_next_match_runs_scored,ML,10,43,74,10,True,True,7.985569,14.154485,10.884477,PASS
1,target_next_match_balls_faced,DL,10,42,59,10,True,True,6.399984,11.329374,8.550847,PASS
2,target_next_match_fours,DL,10,42,59,10,True,True,0.678212,1.234240,0.878420,PASS
3,target_next_match_sixes,DL,10,42,59,10,True,True,0.252925,0.546892,0.397079,PASS
4,target_next_match_strike_rate,DL,10,42,59,10,True,True,100.796806,116.028305,105.296677,PASS
5,target_next_match_dismissals,DL,10,42,59,10,True,True,0.628087,0.847530,0.766173,PASS
6,target_next_match_wickets,ML,10,35,49,10,True,True,0.844992,1.052713,0.910536,PASS
7,target_next_match_runs_conceded,DL,10,30,30,10,True,True,22.589649,28.880535,25.370483,PASS
8,target_next_match_legal_balls,DL,10,30,30,10,True,True,18.076576,22.684864,19.805801,PASS
9,target_next_match_economy_rate,DL,10,30,30,10,True,True,7.815112,8.764125,8.224109,PASS


Models tested: 10
Passed: 10
Warnings: 0
Failed: 0

CELL 22 COMPLETED — READ ONLY


## 23. Target-Specific Prediction Contract Validation

### Objective
Validate the predictions generated by the selected ML and DL models against
basic target-specific output contracts.

### Validation checks
- Prediction values must be finite.
- Predictions must be non-negative.
- Count-based targets must produce numeric regression outputs.
- Rate-based targets must produce numeric regression outputs.
- No arbitrary upper bounds, rounding, or clipping will be applied.

### Important
This validation does not establish predictive accuracy. It verifies
basic output integrity and compatibility with the expected target types.

All checks are read-only. No model training or artifact modification occurs.

In [27]:
# ============================================================
# CELL 23 — TARGET-SPECIFIC PREDICTION CONTRACT VALIDATION
# READ-ONLY | NO TRAINING | NO ARTIFACT MODIFICATION
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Validate input structure
# ------------------------------------------------------------

if not isinstance(inference_results, list):
    raise TypeError("inference_results must be a list.")

if len(inference_results) != 10:
    raise ValueError(
        f"Expected 10 inference records, found {len(inference_results)}."
    )

# ------------------------------------------------------------
# 2. Define target categories
# ------------------------------------------------------------

count_targets = {
    "target_next_match_runs_scored",
    "target_next_match_balls_faced",
    "target_next_match_fours",
    "target_next_match_sixes",
    "target_next_match_dismissals",
    "target_next_match_wickets",
    "target_next_match_runs_conceded",
    "target_next_match_legal_balls",
}

rate_targets = {
    "target_next_match_strike_rate",
    "target_next_match_economy_rate",
}

# ------------------------------------------------------------
# 3. Validate each inference record
# ------------------------------------------------------------

validation_records = []

for result in inference_results:

    if not isinstance(result, dict):
        raise TypeError("Each inference record must be a dictionary.")

    target = result["target"]

    # Target classification
    if target in count_targets:
        target_type = "Count-like regression"
        mapping_valid = True

    elif target in rate_targets:
        target_type = "Rate-like regression"
        mapping_valid = True

    else:
        target_type = "Unknown"
        mapping_valid = False

    # Prediction metadata
    prediction_count = result["prediction_count"]
    sample_rows = result["sample_rows"]

    prediction_min = float(result["prediction_min"])
    prediction_max = float(result["prediction_max"])
    prediction_mean = float(result["prediction_mean"])

    # Validation checks
    count_check = prediction_count == sample_rows

    finite_check = (
        result["finite_predictions"]
        and np.isfinite([
            prediction_min,
            prediction_max,
            prediction_mean
        ]).all()
    )

    nonnegative_check = (
        result["nonnegative_predictions"]
        and prediction_min >= 0
    )

    numeric_summary_check = (
        prediction_min <= prediction_mean <= prediction_max
    )

    passed = all([
        count_check,
        finite_check,
        nonnegative_check,
        numeric_summary_check,
        mapping_valid
    ])

    validation_records.append({
        "target": target,
        "model_type": result["model_type"],
        "target_type": target_type,
        "sample_rows": sample_rows,
        "prediction_count": prediction_count,
        "count_check": count_check,
        "finite_check": finite_check,
        "nonnegative_check": nonnegative_check,
        "numeric_summary_check": numeric_summary_check,
        "target_mapping_valid": mapping_valid,
        "prediction_min": prediction_min,
        "prediction_max": prediction_max,
        "prediction_mean": prediction_mean,
        "status": "PASS" if passed else "FAIL"
    })

# ------------------------------------------------------------
# 4. Create audit DataFrame
# ------------------------------------------------------------

contract_audit_df = pd.DataFrame(validation_records)

display(contract_audit_df)

# ------------------------------------------------------------
# 5. Final validation summary
# ------------------------------------------------------------

print("\nTARGET CONTRACT VALIDATION SUMMARY")
print("=" * 70)

print("Targets checked:", len(contract_audit_df))
print(
    "Passed:",
    (contract_audit_df["status"] == "PASS").sum()
)
print(
    "Failed:",
    (contract_audit_df["status"] == "FAIL").sum()
)

assert contract_audit_df["target"].nunique() == 10
assert len(contract_audit_df) == 10
assert (contract_audit_df["status"] == "PASS").all()

print("\nCELL 23 COMPLETED — READ ONLY")

,target,model_type,target_type,sample_rows,prediction_count,count_check,finite_check,nonnegative_check,numeric_summary_check,target_mapping_valid,prediction_min,prediction_max,prediction_mean,status
0,target_next_match_runs_scored,ML,Count-like regression,10,10,True,True,True,True,True,7.985569,14.154485,10.884477,PASS
1,target_next_match_balls_faced,DL,Count-like regression,10,10,True,True,True,True,True,6.399984,11.329374,8.550847,PASS
2,target_next_match_fours,DL,Count-like regression,10,10,True,True,True,True,True,0.678212,1.234240,0.878420,PASS
3,target_next_match_sixes,DL,Count-like regression,10,10,True,True,True,True,True,0.252925,0.546892,0.397079,PASS
4,target_next_match_strike_rate,DL,Rate-like regression,10,10,True,True,True,True,True,100.796806,116.028305,105.296677,PASS
5,target_next_match_dismissals,DL,Count-like regression,10,10,True,True,True,True,True,0.628087,0.847530,0.766173,PASS
6,target_next_match_wickets,ML,Count-like regression,10,10,True,True,True,True,True,0.844992,1.052713,0.910536,PASS
7,target_next_match_runs_conceded,DL,Count-like regression,10,10,True,True,True,True,True,22.589649,28.880535,25.370483,PASS
8,target_next_match_legal_balls,DL,Count-like regression,10,10,True,True,True,True,True,18.076576,22.684864,19.805801,PASS
9,target_next_match_economy_rate,DL,Rate-like regression,10,10,True,True,True,True,True,7.815112,8.764125,8.224109,PASS



TARGET CONTRACT VALIDATION SUMMARY
Targets checked: 10
Passed: 10
Failed: 0

CELL 23 COMPLETED — READ ONLY


## 24. Chronological Test Evaluation Artifact Audit

### Objective
Inspect the existing chronological test evaluation artifacts generated
during model evaluation.

### Validation scope
- Identify saved actual-versus-predicted outputs.
- Identify saved evaluation metrics.
- Verify target and model coverage.
- Verify chronological test-period information.
- Check artifact availability and schema.

### Important
This cell performs artifact discovery and inspection only.
It does not retrain models, generate new predictions, or modify artifacts.

In [28]:
# ============================================================
# CELL 24 — CHRONOLOGICAL TEST EVALUATION ARTIFACT AUDIT
# READ-ONLY | NO TRAINING | NO ARTIFACT MODIFICATION
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1. Define artifact locations
# ------------------------------------------------------------

artifact_roots = [
    PROJECT_ROOT / "artifacts",
    PROJECT_ROOT / "metrics",
    PROJECT_ROOT / "notebooks" / "artifacts",
    PROJECT_ROOT / "notebooks" / "metrics",
]

# ------------------------------------------------------------
# 2. Discover existing evaluation files
# ------------------------------------------------------------

evaluation_files = []

for root in artifact_roots:
    if root.exists():
        for file_path in root.rglob("*"):
            if file_path.is_file():
                name = file_path.name.lower()

                if any(keyword in name for keyword in [
                    "prediction",
                    "evaluation",
                    "metric",
                    "constraint",
                    "audit",
                    "comparison"
                ]):
                    evaluation_files.append(file_path)

evaluation_files = sorted(set(evaluation_files))

print("DISCOVERED EVALUATION ARTIFACTS")
print("=" * 80)

for file_path in evaluation_files:
    print(file_path.relative_to(PROJECT_ROOT))

print("\nTotal files discovered:", len(evaluation_files))

# ------------------------------------------------------------
# 3. Inspect CSV schemas and dimensions
# ------------------------------------------------------------

csv_records = []

for file_path in evaluation_files:

    if file_path.suffix.lower() != ".csv":
        continue

    try:
        df = pd.read_csv(file_path, nrows=5)

        csv_records.append({
            "file": str(file_path.relative_to(PROJECT_ROOT)),
            "columns": len(df.columns),
            "sample_rows": len(df),
            "column_names": list(df.columns),
            "status": "READABLE"
        })

    except Exception as exc:

        csv_records.append({
            "file": str(file_path.relative_to(PROJECT_ROOT)),
            "columns": None,
            "sample_rows": None,
            "column_names": None,
            "status": f"ERROR: {exc}"
        })

evaluation_artifact_audit_df = pd.DataFrame(csv_records)

if not evaluation_artifact_audit_df.empty:
    display(evaluation_artifact_audit_df)

print("\nCELL 24 COMPLETED — READ ONLY")

DISCOVERED EVALUATION ARTIFACTS
artifacts\dl_validation\mlp_generalization_audit.csv
artifacts\final_reports\DL_MODEL_EVALUATION_REPORT.md
artifacts\final_reports\REPRODUCIBILITY_AUDIT.json
artifacts\final_reports\selected_model_artifact_audit.csv
artifacts\model_comparison\constrained_test_predictions.csv
artifacts\model_comparison\corrected_comparison_alignment_audit.csv
artifacts\model_comparison\corrected_ml_dl_baseline_comparison.csv
artifacts\model_comparison\ml_dl_test_comparison_20261102_161403.csv
artifacts\model_comparison\ml_dl_test_comparison_20261102_161447.csv
artifacts\model_comparison\output_constraint_audit.csv
artifacts\model_comparison\output_constraint_policy.json
artifacts\model_comparison\realistic_inference_audit.csv
notebooks\artifacts\dl_validation\mlp_generalization_audit.csv

Total files discovered: 13


,file,columns,sample_rows,column_names,status
0,artifacts\dl_validation\mlp_generalization_aud...,27,5,"[Target, Validation_Samples, DL_MAE_Validation...",READABLE
1,artifacts\final_reports\selected_model_artifac...,5,5,"[target, selected_model, artifact, exists, non...",READABLE
2,artifacts\model_comparison\constrained_test_pr...,10,5,"[category, target, selected_model, source_inde...",READABLE
3,artifacts\model_comparison\corrected_compariso...,7,5,"[target, test_rows, unique_indices, target_val...",READABLE
4,artifacts\model_comparison\corrected_ml_dl_bas...,9,5,"[category, target, model, model_type, test_row...",READABLE
5,artifacts\model_comparison\ml_dl_test_comparis...,9,5,"[category, target, model, rows, MAE, RMSE, R2,...",READABLE
6,artifacts\model_comparison\ml_dl_test_comparis...,9,5,"[category, target, model, rows, MAE, RMSE, R2,...",READABLE
7,artifacts\model_comparison\output_constraint_a...,11,5,"[target, selected_model, output_type, precisio...",READABLE
8,artifacts\model_comparison\realistic_inference...,10,5,"[category, target, selected_model, input_rows,...",READABLE
9,notebooks\artifacts\dl_validation\mlp_generali...,27,5,"[Target, Validation_Samples, DL_MAE_Validation...",READABLE



CELL 24 COMPLETED — READ ONLY


## 24B. Detailed Evaluation Artifact Inspection

### Objective
Inspect the full schemas, dimensions, data types, and sample records
of the existing evaluation artifacts.

### Focus
- Actual and predicted values
- Target and model identifiers
- Match/player identifiers
- Test-period dates
- Evaluation metrics
- Constraint and alignment audit fields

### Scope
Read-only inspection. No predictions, training, or artifact modifications.

In [29]:
# ============================================================
# CELL 24B — DETAILED EVALUATION ARTIFACT INSPECTION
# READ-ONLY
# ============================================================

import pandas as pd

inspection_files = [
    PROJECT_ROOT / "artifacts/model_comparison/constrained_test_predictions.csv",
    PROJECT_ROOT / "artifacts/model_comparison/ml_dl_test_comparison_20261102_161447.csv",
    PROJECT_ROOT / "artifacts/model_comparison/corrected_comparison_alignment_audit.csv",
    PROJECT_ROOT / "artifacts/model_comparison/realistic_inference_audit.csv",
    PROJECT_ROOT / "artifacts/model_comparison/output_constraint_audit.csv",
]

evaluation_dataframes = {}

for file_path in inspection_files:

    print("\n" + "=" * 100)
    print("FILE:", file_path.relative_to(PROJECT_ROOT))
    print("=" * 100)

    if not file_path.exists():
        print("STATUS: FILE NOT FOUND")
        continue

    df = pd.read_csv(file_path)

    evaluation_dataframes[file_path.name] = df

    print("Shape:", df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nData types:")
    print(df.dtypes.to_string())

    print("\nSample records:")
    display(df.head(3))

    print("\nMissing values:")
    missing = df.isna().sum()
    print(missing[missing > 0].to_string())

print("\n" + "=" * 100)
print("DETAILED ARTIFACT INSPECTION COMPLETED")
print("Readable artifacts:", len(evaluation_dataframes))
print("CELL 24B COMPLETED — READ ONLY")


FILE: artifacts\model_comparison\constrained_test_predictions.csv
Shape: (21468, 10)

Columns:
['category', 'target', 'selected_model', 'source_index', 'match_id', 'player', 'date', 'actual', 'raw_prediction', 'constrained_prediction']

Data types:
category                      str
target                        str
selected_model                str
source_index                int64
match_id                    int64
player                        str
date                          str
actual                    float64
raw_prediction            float64
constrained_prediction    float64

Sample records:


,category,target,selected_model,source_index,match_id,player,date,actual,raw_prediction,constrained_prediction
0,batting,target_next_match_runs_scored,batting_original,53,1426292,A Badoni,2024-05-05,55.0,19.488948,19.0
1,batting,target_next_match_runs_scored,batting_original,54,1426295,A Badoni,2024-05-08,6.0,18.356989,18.0
2,batting,target_next_match_runs_scored,batting_original,55,1426302,A Badoni,2024-05-14,22.0,19.582317,20.0



Missing values:
Series([], )

FILE: artifacts\model_comparison\ml_dl_test_comparison_20261102_161447.csv
Shape: (11, 9)

Columns:
['category', 'target', 'model', 'rows', 'MAE', 'RMSE', 'R2', 'Bias', 'negative_predictions']

Data types:
category                    str
target                      str
model                       str
rows                      int64
MAE                     float64
RMSE                    float64
R2                      float64
Bias                    float64
negative_predictions      int64

Sample records:


,category,target,model,rows,MAE,RMSE,R2,Bias,negative_predictions
0,Batting,target_next_match_runs_scored,batting_sqrt,2375,16.543208,23.685276,0.012017,-7.613649,0
1,Batting,target_next_match_runs_scored,batting_original,2375,16.643802,22.459996,0.111593,-2.564665,0
2,Batting,target_next_match_runs_scored,DL_MLP,2375,16.763107,22.593155,0.101028,-3.007407,0



Missing values:
Series([], )

FILE: artifacts\model_comparison\corrected_comparison_alignment_audit.csv
Shape: (10, 7)

Columns:
['target', 'test_rows', 'unique_indices', 'target_values_match', 'DL_predictions_finite', 'test_start', 'test_end']

Data types:
target                     str
test_rows                int64
unique_indices            bool
target_values_match       bool
DL_predictions_finite     bool
test_start                 str
test_end                   str

Sample records:


,target,test_rows,unique_indices,target_values_match,DL_predictions_finite,test_start,test_end
0,target_next_match_runs_scored,2375,True,True,True,2024-05-02,2026-05-29
1,target_next_match_balls_faced,2375,True,True,True,2024-05-02,2026-05-29
2,target_next_match_fours,2375,True,True,True,2024-05-02,2026-05-29



Missing values:
Series([], )

FILE: artifacts\model_comparison\realistic_inference_audit.csv
Shape: (10, 10)

Columns:
['category', 'target', 'selected_model', 'input_rows', 'input_features', 'predictions_finite', 'schema_valid', 'MAE', 'RMSE', 'R2']

Data types:
category                  str
target                    str
selected_model            str
input_rows              int64
input_features          int64
predictions_finite       bool
schema_valid             bool
MAE                   float64
RMSE                  float64
R2                    float64

Sample records:


,category,target,selected_model,input_rows,input_features,predictions_finite,schema_valid,MAE,RMSE,R2
0,batting,target_next_match_runs_scored,batting_original,2375,43,True,True,16.643802,22.459996,0.111593
1,batting,target_next_match_balls_faced,DL_MLP,2375,42,True,True,9.364080,12.018598,0.127566
2,batting,target_next_match_fours,DL_MLP,2375,42,True,True,1.671235,2.295062,0.111262



Missing values:
Series([], )

FILE: artifacts\model_comparison\output_constraint_audit.csv
Shape: (10, 11)

Columns:
['target', 'selected_model', 'output_type', 'precision', 'test_rows', 'raw_negative_predictions', 'constrained_negative_predictions', 'raw_predictions_finite', 'constrained_predictions_finite', 'raw_MAE', 'constrained_MAE']

Data types:
target                                  str
selected_model                          str
output_type                             str
precision                               str
test_rows                             int64
raw_negative_predictions              int64
constrained_negative_predictions      int64
raw_predictions_finite                 bool
constrained_predictions_finite         bool
raw_MAE                             float64
constrained_MAE                     float64

Sample records:


,target,selected_model,output_type,precision,test_rows,raw_negative_predictions,constrained_negative_predictions,raw_predictions_finite,constrained_predictions_finite,raw_MAE,constrained_MAE
0,target_next_match_runs_scored,batting_original,non_negative_integer,integer,2375,0,0,True,True,16.643802,16.639579
1,target_next_match_balls_faced,DL_MLP,non_negative_integer,integer,2375,0,0,True,True,9.364080,9.358737
2,target_next_match_fours,DL_MLP,non_negative_integer,integer,2375,0,0,True,True,1.671235,1.666526



Missing values:
Series([], )

DETAILED ARTIFACT INSPECTION COMPLETED
Readable artifacts: 5
CELL 24B COMPLETED — READ ONLY


## 24C. Actual-versus-Predicted Metric Reconciliation

### Objective
Recalculate evaluation metrics from the saved chronological test predictions
and compare raw versus constrained predictions.

### Metrics
- MAE
- RMSE
- R²
- Bias (mean prediction error)

### Data integrity checks
- Target coverage
- Records per target
- Missing values
- Duplicate prediction identifiers
- Finite actual and predicted values

### Important
The existing saved predictions are reused. No new train/test split,
model training, or prediction generation is performed.

In [30]:
# ============================================================
# CELL 24C — ACTUAL VS PREDICTED METRIC RECONCILIATION
# READ-ONLY | EXISTING TEST PREDICTIONS ONLY
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ------------------------------------------------------------
# 1. Load existing prediction artifact
# ------------------------------------------------------------

prediction_path = (
    PROJECT_ROOT
    / "artifacts"
    / "model_comparison"
    / "constrained_test_predictions.csv"
)

predictions_df = pd.read_csv(prediction_path)

required_columns = {
    "target",
    "selected_model",
    "source_index",
    "match_id",
    "player",
    "date",
    "actual",
    "raw_prediction",
    "constrained_prediction"
}

missing_columns = required_columns - set(predictions_df.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# ------------------------------------------------------------
# 2. Dataset integrity audit
# ------------------------------------------------------------

print("PREDICTION DATASET INTEGRITY")
print("=" * 75)

print("Total records:", len(predictions_df))
print("Unique targets:", predictions_df["target"].nunique())
print("Unique selected models:", predictions_df["selected_model"].nunique())

print("\nRecords per target:")
display(
    predictions_df.groupby("target")
    .agg(
        records=("actual", "size"),
        unique_players=("player", "nunique"),
        unique_matches=("match_id", "nunique"),
        unique_source_indices=("source_index", "nunique")
    )
    .reset_index()
)

print("\nMissing values:")
display(
    predictions_df.isna().sum()
    .loc[lambda x: x > 0]
    .rename("missing_count")
    .to_frame()
)

# ------------------------------------------------------------
# 3. Duplicate identifier audit
# ------------------------------------------------------------

duplicate_keys = predictions_df.duplicated(
    subset=["target", "source_index"],
    keep=False
)

print("\nDuplicate target/source_index records:", duplicate_keys.sum())

# ------------------------------------------------------------
# 4. Date range audit
# ------------------------------------------------------------

predictions_df["date_parsed"] = pd.to_datetime(
    predictions_df["date"],
    errors="coerce"
)

print("\nTest date range:")
print("Start:", predictions_df["date_parsed"].min())
print("End:", predictions_df["date_parsed"].max())

# ------------------------------------------------------------
# 5. Recalculate metrics from saved predictions
# ------------------------------------------------------------

metric_records = []

for target, group in predictions_df.groupby("target"):

    actual = group["actual"].to_numpy(dtype=float)

    for prediction_column, prediction_label in [
        ("raw_prediction", "Raw"),
        ("constrained_prediction", "Constrained")
    ]:

        predicted = group[prediction_column].to_numpy(dtype=float)

        valid = np.isfinite(actual) & np.isfinite(predicted)

        y_true = actual[valid]
        y_pred = predicted[valid]

        if len(y_true) == 0:
            continue

        metric_records.append({
            "target": target,
            "prediction_type": prediction_label,
            "rows": len(y_true),
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "R2": r2_score(y_true, y_pred),
            "Bias": float(np.mean(y_pred - y_true))
        })

recalculated_metrics_df = pd.DataFrame(metric_records)

print("\nRECALCULATED TEST METRICS")
print("=" * 75)

display(
    recalculated_metrics_df.sort_values(
        ["target", "prediction_type"]
    ).reset_index(drop=True)
)

# ------------------------------------------------------------
# 6. Inspect saved model comparison records
# ------------------------------------------------------------

comparison_path = (
    PROJECT_ROOT
    / "artifacts"
    / "model_comparison"
    / "ml_dl_test_comparison_20261102_161447.csv"
)

saved_comparison_df = pd.read_csv(comparison_path)

print("\nEXISTING SAVED MODEL COMPARISON")
print("=" * 75)

display(saved_comparison_df)

print("\nSaved comparison rows:", len(saved_comparison_df))
print("Saved comparison targets:", saved_comparison_df["target"].nunique())
print("Models per target:")
display(
    saved_comparison_df.groupby("target")["model"]
    .apply(list)
    .to_frame()
)

print("\nCELL 24C COMPLETED — READ ONLY")

PREDICTION DATASET INTEGRITY
Total records: 21468
Unique targets: 10
Unique selected models: 3

Records per target:


,target,records,unique_players,unique_matches,unique_source_indices
0,target_next_match_balls_faced,2375,202,167,2375
1,target_next_match_dismissals,2375,202,167,2375
2,target_next_match_economy_rate,1805,156,167,1805
3,target_next_match_fours,2375,202,167,2375
4,target_next_match_legal_balls,1805,156,167,1805
5,target_next_match_runs_conceded,1805,156,167,1805
6,target_next_match_runs_scored,2375,202,167,2375
7,target_next_match_sixes,2375,202,167,2375
8,target_next_match_strike_rate,2373,202,167,2373
9,target_next_match_wickets,1805,156,167,1805



Missing values:


,missing_count



Duplicate target/source_index records: 0

Test date range:
Start: 2024-05-02 00:00:00
End: 2026-05-29 00:00:00

RECALCULATED TEST METRICS


,target,prediction_type,rows,MAE,RMSE,R2,Bias
0,target_next_match_balls_faced,Constrained,2375,9.358737,12.023153,0.126904,-0.352421
1,target_next_match_balls_faced,Raw,2375,9.364080,12.018598,0.127566,-0.350296
2,target_next_match_dismissals,Constrained,2375,0.216000,0.464758,-0.284615,0.180632
3,target_next_match_dismissals,Raw,2375,0.272703,0.401838,0.039667,0.062993
4,target_next_match_economy_rate,Constrained,1805,2.821272,4.058322,-0.117300,-1.334887
5,target_next_match_economy_rate,Raw,1805,2.821380,4.058312,-0.117294,-1.334910
6,target_next_match_fours,Constrained,2375,1.666526,2.314440,0.096191,-0.308211
7,target_next_match_fours,Raw,2375,1.671235,2.295062,0.111262,-0.309986
8,target_next_match_legal_balls,Constrained,1805,3.869252,5.379967,0.191017,1.340720
9,target_next_match_legal_balls,Raw,1805,3.872398,5.370624,0.193824,1.346289



EXISTING SAVED MODEL COMPARISON


,category,target,model,rows,MAE,RMSE,R2,Bias,negative_predictions
0,Batting,target_next_match_runs_scored,batting_sqrt,2375,16.543208,23.685276,0.012017,-7.613649,0
1,Batting,target_next_match_runs_scored,batting_original,2375,16.643802,22.459996,0.111593,-2.564665,0
2,Batting,target_next_match_runs_scored,DL_MLP,2375,16.763107,22.593155,0.101028,-3.007407,0
3,Bowling,target_next_match_economy_rate,DL_MLP,1805,2.821380,4.058312,-0.117294,-1.334910,0
4,Bowling,target_next_match_economy_rate,bowling_economy,1805,2.829871,4.068635,-0.122985,-1.346444,0
5,Bowling,target_next_match_legal_balls,DL_MLP,1805,3.872398,5.370624,0.193824,1.346289,0
6,Bowling,target_next_match_legal_balls,bowling_balls,1805,5.052258,5.982909,-0.000472,0.129998,0
7,Bowling,target_next_match_runs_conceded,DL_MLP,1805,9.227933,11.577274,0.103529,-1.037495,0
8,Bowling,target_next_match_runs_conceded,bowling_runs,1805,10.334704,12.830062,-0.100985,-3.885670,0
9,Bowling,target_next_match_wickets,bowling_wickets,1805,0.763657,0.993646,0.023595,-0.070748,0



Saved comparison rows: 11
Saved comparison targets: 5
Models per target:


,model
target,
target_next_match_economy_rate,"[DL_MLP, bowling_economy]"
target_next_match_legal_balls,"[DL_MLP, bowling_balls]"
target_next_match_runs_conceded,"[DL_MLP, bowling_runs]"
target_next_match_runs_scored,"[batting_sqrt, batting_original, DL_MLP]"
target_next_match_wickets,"[bowling_wickets, DL_MLP]"



CELL 24C COMPLETED — READ ONLY


## 24D. Saved-versus-Recalculated Metric Reconciliation

### Objective
Compare the existing model comparison metrics against metrics
recalculated from the saved test predictions.

### Checks
- Target and model matching
- Evaluation row-count matching
- MAE, RMSE and R² differences
- Identification of unmatched comparison records

### Scope
Read-only metric reconciliation. No training or prediction generation.

In [31]:
# ============================================================
# CELL 24D — SAVED VS RECALCULATED METRIC RECONCILIATION
# READ-ONLY
# ============================================================

# ------------------------------------------------------------
# 1. Load saved comparison
# ------------------------------------------------------------

comparison_df = pd.read_csv(
    PROJECT_ROOT
    / "artifacts"
    / "model_comparison"
    / "ml_dl_test_comparison_20261102_161447.csv"
)

# ------------------------------------------------------------
# 2. Map selected model names to prediction artifact names
# ------------------------------------------------------------

model_mapping = {
    "batting_original": "batting_original",
    "DL_MLP": "DL_MLP",
    "bowling_wickets": "bowling_wickets",
    "bowling_economy": "bowling_economy",
    "bowling_balls": "bowling_balls",
    "bowling_runs": "bowling_runs",
    "batting_sqrt": "batting_sqrt"
}

# ------------------------------------------------------------
# 3. Build reconciliation records
# ------------------------------------------------------------

reconciliation_records = []

for _, saved in comparison_df.iterrows():

    target = saved["target"]
    model = saved["model"]

    # Find matching selected-model prediction records
    prediction_group = predictions_df[
        (predictions_df["target"] == target) &
        (predictions_df["selected_model"] == model)
    ]

    if prediction_group.empty:
        reconciliation_records.append({
            "target": target,
            "model": model,
            "saved_rows": saved["rows"],
            "prediction_rows": 0,
            "status": "NO MATCHING PREDICTIONS"
        })
        continue

    actual = prediction_group["actual"].to_numpy(dtype=float)

    raw = prediction_group["raw_prediction"].to_numpy(dtype=float)

    valid = np.isfinite(actual) & np.isfinite(raw)

    y_true = actual[valid]
    y_pred = raw[valid]

    recalculated_mae = mean_absolute_error(y_true, y_pred)
    recalculated_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    recalculated_r2 = r2_score(y_true, y_pred)

    rows_match = len(y_true) == int(saved["rows"])

    mae_difference = abs(recalculated_mae - saved["MAE"])
    rmse_difference = abs(recalculated_rmse - saved["RMSE"])
    r2_difference = abs(recalculated_r2 - saved["R2"])

    metrics_match = np.allclose(
        [recalculated_mae, recalculated_rmse, recalculated_r2],
        [saved["MAE"], saved["RMSE"], saved["R2"]],
        rtol=1e-7,
        atol=1e-9
    )

    reconciliation_records.append({
        "target": target,
        "model": model,
        "saved_rows": int(saved["rows"]),
        "prediction_rows": len(y_true),
        "rows_match": rows_match,
        "saved_MAE": saved["MAE"],
        "recalculated_MAE": recalculated_mae,
        "MAE_difference": mae_difference,
        "saved_RMSE": saved["RMSE"],
        "recalculated_RMSE": recalculated_rmse,
        "RMSE_difference": rmse_difference,
        "saved_R2": saved["R2"],
        "recalculated_R2": recalculated_r2,
        "R2_difference": r2_difference,
        "metrics_match": metrics_match,
        "status": "PASS" if rows_match and metrics_match else "REVIEW"
    })

metric_reconciliation_df = pd.DataFrame(reconciliation_records)

display(metric_reconciliation_df)

# ------------------------------------------------------------
# 4. Summary
# ------------------------------------------------------------

print("\nMETRIC RECONCILIATION SUMMARY")
print("=" * 75)

print("Comparison records:", len(metric_reconciliation_df))
print(
    "Passed:",
    (metric_reconciliation_df["status"] == "PASS").sum()
)
print(
    "Needs review:",
    (metric_reconciliation_df["status"] != "PASS").sum()
)

print("\nCELL 24D COMPLETED — READ ONLY")

,target,model,saved_rows,prediction_rows,status,rows_match,saved_MAE,recalculated_MAE,MAE_difference,saved_RMSE,recalculated_RMSE,RMSE_difference,saved_R2,recalculated_R2,R2_difference,metrics_match
0,target_next_match_runs_scored,batting_sqrt,2375,0,NO MATCHING PREDICTIONS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,target_next_match_runs_scored,batting_original,2375,2375,PASS,True,16.643802,16.643802,0.0,22.459996,22.459996,0.0,0.111593,0.111593,0.0,True
2,target_next_match_runs_scored,DL_MLP,2375,0,NO MATCHING PREDICTIONS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,target_next_match_economy_rate,DL_MLP,1805,1805,PASS,True,2.821380,2.821380,0.0,4.058312,4.058312,0.0,-0.117294,-0.117294,0.0,True
4,target_next_match_economy_rate,bowling_economy,1805,0,NO MATCHING PREDICTIONS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,target_next_match_legal_balls,DL_MLP,1805,1805,PASS,True,3.872398,3.872398,0.0,5.370624,5.370624,0.0,0.193824,0.193824,0.0,True
6,target_next_match_legal_balls,bowling_balls,1805,0,NO MATCHING PREDICTIONS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,target_next_match_runs_conceded,DL_MLP,1805,1805,PASS,True,9.227933,9.227933,0.0,11.577274,11.577274,0.0,0.103529,0.103529,0.0,True
8,target_next_match_runs_conceded,bowling_runs,1805,0,NO MATCHING PREDICTIONS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,target_next_match_wickets,bowling_wickets,1805,1805,PASS,True,0.763657,0.763657,0.0,0.993646,0.993646,0.0,0.023595,0.023595,0.0,True



METRIC RECONCILIATION SUMMARY
Comparison records: 11
Passed: 5
Needs review: 6

CELL 24D COMPLETED — READ ONLY


## 24E. Selected Production Model Reconciliation

### Objective
Verify that every selected production model has corresponding
saved test predictions and that its evaluation metrics reconcile.

### Validation
- Confirm all 10 targets are represented.
- Confirm prediction model selection matches the production selection.
- Confirm saved and recalculated metrics agree where comparison records exist.
- Classify alternative candidate models separately.

### Important
Alternative candidate models are not expected to appear in the
selected production prediction artifact.
No model training or prediction generation is performed.

In [32]:
# ============================================================
# CELL 24E — SELECTED PRODUCTION MODEL RECONCILIATION
# READ-ONLY
# ============================================================

# 1. Verify all ten production targets exist
production_targets = set(predictions_df["target"].unique())

expected_targets = set(contract_audit_df["target"].unique())

assert production_targets == expected_targets, (
    "Production target coverage does not match the contract audit."
)

# 2. Verify the selected model recorded for each target
selected_model_records = (
    predictions_df.groupby("target")["selected_model"]
    .agg(lambda values: values.unique().tolist())
    .reset_index(name="prediction_models")
)

assert selected_model_records["prediction_models"].apply(
    len
).eq(1).all(), "Multiple selected models found for a target."

# 3. Compare the five existing saved comparison records
selected_comparison = metric_reconciliation_df[
    metric_reconciliation_df["status"] == "PASS"
].copy()

# 4. Identify alternative candidate records
candidate_comparison = metric_reconciliation_df[
    metric_reconciliation_df["status"] == "NO MATCHING PREDICTIONS"
].copy()

print("SELECTED PRODUCTION MODEL AUDIT")
print("=" * 75)

display(selected_model_records)

print("\nSelected-model metric records reconciled:",
      len(selected_comparison))

print("Alternative candidate records not present in selected artifact:",
      len(candidate_comparison))

print("\nSelected model metric reconciliation:")
display(selected_comparison[[
    "target",
    "model",
    "saved_rows",
    "prediction_rows",
    "metrics_match",
    "status"
]])

print("\nAlternative candidate models:")
display(candidate_comparison[[
    "target",
    "model",
    "saved_rows",
    "status"
]])

assert len(selected_model_records) == 10
assert len(selected_comparison) == 5
assert len(candidate_comparison) == 6

print("\nCELL 24E COMPLETED — READ ONLY")

SELECTED PRODUCTION MODEL AUDIT


,target,prediction_models
0,target_next_match_balls_faced,[DL_MLP]
1,target_next_match_dismissals,[DL_MLP]
2,target_next_match_economy_rate,[DL_MLP]
3,target_next_match_fours,[DL_MLP]
4,target_next_match_legal_balls,[DL_MLP]
5,target_next_match_runs_conceded,[DL_MLP]
6,target_next_match_runs_scored,[batting_original]
7,target_next_match_sixes,[DL_MLP]
8,target_next_match_strike_rate,[DL_MLP]
9,target_next_match_wickets,[bowling_wickets]



Selected-model metric records reconciled: 5
Alternative candidate records not present in selected artifact: 6

Selected model metric reconciliation:


,target,model,saved_rows,prediction_rows,metrics_match,status
1,target_next_match_runs_scored,batting_original,2375,2375,True,PASS
3,target_next_match_economy_rate,DL_MLP,1805,1805,True,PASS
5,target_next_match_legal_balls,DL_MLP,1805,1805,True,PASS
7,target_next_match_runs_conceded,DL_MLP,1805,1805,True,PASS
9,target_next_match_wickets,bowling_wickets,1805,1805,True,PASS



Alternative candidate models:


,target,model,saved_rows,status
0,target_next_match_runs_scored,batting_sqrt,2375,NO MATCHING PREDICTIONS
2,target_next_match_runs_scored,DL_MLP,2375,NO MATCHING PREDICTIONS
4,target_next_match_economy_rate,bowling_economy,1805,NO MATCHING PREDICTIONS
6,target_next_match_legal_balls,bowling_balls,1805,NO MATCHING PREDICTIONS
8,target_next_match_runs_conceded,bowling_runs,1805,NO MATCHING PREDICTIONS
10,target_next_match_wickets,DL_MLP,1805,NO MATCHING PREDICTIONS



CELL 24E COMPLETED — READ ONLY


## 25. Consolidated Production Inference Audit

### Objective
Consolidate the results of production inference, output contract
validation, evaluation artifact inspection, and selected-model
metric reconciliation.

### Report sections
1. Selected production model mapping
2. Inference smoke-test results
3. Target contract validation
4. Chronological test prediction coverage
5. Reconciled evaluation metrics
6. Outstanding evaluation limitations

### Scope
Read-only consolidation of existing audit results.
No model training or artifact modification.

In [33]:
# ============================================================
# CELL 25 — CONSOLIDATED PRODUCTION INFERENCE AUDIT
# READ-ONLY | NO TRAINING | NO ARTIFACT MODIFICATION
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Production inference summary
# ------------------------------------------------------------

inference_summary_df = pd.DataFrame(inference_results)

# ------------------------------------------------------------
# 2. Target contract summary
# ------------------------------------------------------------

contract_summary_df = contract_audit_df[[
    "target",
    "target_type",
    "status"
]].copy()

# ------------------------------------------------------------
# 3. Chronological test coverage
# ------------------------------------------------------------

test_coverage_df = (
    predictions_df.groupby("target")
    .agg(
        test_rows=("actual", "size"),
        unique_players=("player", "nunique"),
        unique_matches=("match_id", "nunique"),
        test_start=("date_parsed", "min"),
        test_end=("date_parsed", "max")
    )
    .reset_index()
)

# ------------------------------------------------------------
# 4. Selected-model metric reconciliation
# ------------------------------------------------------------

selected_metric_df = selected_comparison[[
    "target",
    "model",
    "saved_rows",
    "prediction_rows",
    "saved_MAE",
    "recalculated_MAE",
    "saved_RMSE",
    "recalculated_RMSE",
    "saved_R2",
    "recalculated_R2",
    "metrics_match",
    "status"
]].copy()

# ------------------------------------------------------------
# 5. Consolidated target-level report
# ------------------------------------------------------------

production_audit_df = (
    inference_summary_df[
        ["target", "model_type", "status"]
    ]
    .merge(
        contract_summary_df,
        on="target",
        suffixes=("_inference", "_contract")
    )
    .merge(
        test_coverage_df,
        on="target",
        how="left"
    )
)

print("CONSOLIDATED PRODUCTION INFERENCE AUDIT")
print("=" * 85)

display(production_audit_df)

print("\nSELECTED MODEL METRIC RECONCILIATION")
print("=" * 85)

display(selected_metric_df)

# ------------------------------------------------------------
# 6. Overall audit summary
# ------------------------------------------------------------

print("\nFINAL AUDIT SUMMARY")
print("=" * 85)

print("Production targets:", len(production_audit_df))

print(
    "Smoke-test passed:",
    (production_audit_df["status_inference"] == "PASS").sum()
)

print(
    "Contract validation passed:",
    (production_audit_df["status_contract"] == "PASS").sum()
)

print(
    "Targets with test predictions:",
    production_audit_df["test_rows"].notna().sum()
)

print(
    "Selected-model metric reconciliations:",
    selected_metric_df["metrics_match"].sum()
)

print(
    "Prediction records:",
    len(predictions_df)
)

print(
    "Chronological test period:",
    predictions_df["date_parsed"].min(),
    "to",
    predictions_df["date_parsed"].max()
)

print("\nKNOWN LIMITATIONS")
print("- Six alternative candidate models are absent from the selected prediction artifact.")
print("- Test sample counts differ across targets.")
print("- Five selected-model comparison records were available for metric reconciliation.")
print("- Smoke-test inference uses 10 sample rows per model; it is not an accuracy evaluation.")

print("\nCELL 25 COMPLETED — READ ONLY")

CONSOLIDATED PRODUCTION INFERENCE AUDIT


,target,model_type,status_inference,target_type,status_contract,test_rows,unique_players,unique_matches,test_start,test_end
0,target_next_match_runs_scored,ML,PASS,Count-like regression,PASS,2375,202,167,2024-05-02,2026-05-29
1,target_next_match_balls_faced,DL,PASS,Count-like regression,PASS,2375,202,167,2024-05-02,2026-05-29
2,target_next_match_fours,DL,PASS,Count-like regression,PASS,2375,202,167,2024-05-02,2026-05-29
3,target_next_match_sixes,DL,PASS,Count-like regression,PASS,2375,202,167,2024-05-02,2026-05-29
4,target_next_match_strike_rate,DL,PASS,Rate-like regression,PASS,2373,202,167,2024-05-02,2026-05-29
5,target_next_match_dismissals,DL,PASS,Count-like regression,PASS,2375,202,167,2024-05-02,2026-05-29
6,target_next_match_wickets,ML,PASS,Count-like regression,PASS,1805,156,167,2024-05-02,2026-05-29
7,target_next_match_runs_conceded,DL,PASS,Count-like regression,PASS,1805,156,167,2024-05-02,2026-05-29
8,target_next_match_legal_balls,DL,PASS,Count-like regression,PASS,1805,156,167,2024-05-02,2026-05-29
9,target_next_match_economy_rate,DL,PASS,Rate-like regression,PASS,1805,156,167,2024-05-02,2026-05-29



SELECTED MODEL METRIC RECONCILIATION


,target,model,saved_rows,prediction_rows,saved_MAE,recalculated_MAE,saved_RMSE,recalculated_RMSE,saved_R2,recalculated_R2,metrics_match,status
1,target_next_match_runs_scored,batting_original,2375,2375,16.643802,16.643802,22.459996,22.459996,0.111593,0.111593,True,PASS
3,target_next_match_economy_rate,DL_MLP,1805,1805,2.821380,2.821380,4.058312,4.058312,-0.117294,-0.117294,True,PASS
5,target_next_match_legal_balls,DL_MLP,1805,1805,3.872398,3.872398,5.370624,5.370624,0.193824,0.193824,True,PASS
7,target_next_match_runs_conceded,DL_MLP,1805,1805,9.227933,9.227933,11.577274,11.577274,0.103529,0.103529,True,PASS
9,target_next_match_wickets,bowling_wickets,1805,1805,0.763657,0.763657,0.993646,0.993646,0.023595,0.023595,True,PASS



FINAL AUDIT SUMMARY
Production targets: 10
Smoke-test passed: 10
Contract validation passed: 10
Targets with test predictions: 10
Selected-model metric reconciliations: 5
Prediction records: 21468
Chronological test period: 2024-05-02 00:00:00 to 2026-05-29 00:00:00

KNOWN LIMITATIONS
- Six alternative candidate models are absent from the selected prediction artifact.
- Test sample counts differ across targets.
- Five selected-model comparison records were available for metric reconciliation.
- Smoke-test inference uses 10 sample rows per model; it is not an accuracy evaluation.

CELL 25 COMPLETED — READ ONLY


## 26. Clean-Kernel Reproducibility Audit

### Objective
Verify that the production inference and model contract validation
workflow can execute from a clean Jupyter kernel.

### Validation scope
- Project paths and environment
- Dataset availability
- Selected model artifact availability
- Inference smoke-test completion
- Target contract validation
- Evaluation prediction artifact integrity
- Selected-model metric reconciliation
- Consolidated audit completion

### Important
This audit must be executed after restarting the kernel and running
all preceding notebook cells sequentially.

No model retraining or artifact modification is performed.

In [35]:
# ============================================================
# CELL 26 — CLEAN-KERNEL REPRODUCIBILITY AUDIT
# READ-ONLY | NO TRAINING | NO ARTIFACT MODIFICATION
# ============================================================

from pathlib import Path
import sys
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Environment validation
# ------------------------------------------------------------

print("ENVIRONMENT VALIDATION")
print("=" * 75)

print("Python:", sys.version)
print("Project root:", PROJECT_ROOT)
print("Project root exists:", PROJECT_ROOT.exists())

assert PROJECT_ROOT.exists()

# ------------------------------------------------------------
# 2. Dataset availability
# ------------------------------------------------------------

dataset_paths = [
    PROJECT_ROOT / "data/processed/features/batting_model_dataset.csv",
    PROJECT_ROOT / "data/processed/features/bowling_model_dataset.csv"
]

for path in dataset_paths:
    assert path.exists(), f"Missing dataset: {path}"

print("\nDataset files available:", len(dataset_paths))

# ------------------------------------------------------------
# 3. Production artifact availability
# ------------------------------------------------------------

required_artifacts = [
    PROJECT_ROOT / "models/batting/original_gb_model.joblib",
    PROJECT_ROOT / "models/bowling/wickets_model.joblib",
]

for path in required_artifacts:
    assert path.exists(), f"Missing ML artifact: {path}"

# Validate selected DL artifacts
for result in inference_results:

    if result["model_type"] != "DL":
        continue

    target = result["target"]

    category = (
        "batting"
        if target in {
            "target_next_match_balls_faced",
            "target_next_match_fours",
            "target_next_match_sixes",
            "target_next_match_strike_rate",
            "target_next_match_dismissals"
        }
        else "bowling"
    )

    target_name = target

    artifact_dir = (
        PROJECT_ROOT
        / "artifacts"
        / "dl_models"
        / category
        / target_name
    )

    for filename in [
        "model.keras",
        "preprocessor.joblib",
        "target_scaler.joblib"
    ]:
        artifact_path = artifact_dir / filename
        assert artifact_path.exists(), (
            f"Missing artifact: {artifact_path}"
        )

print("Production model artifacts available: PASS")

# ------------------------------------------------------------
# 4. Inference smoke-test validation
# ------------------------------------------------------------

assert len(inference_results) == 10
assert all(
    record["status"] == "PASS"
    for record in inference_results
)

print("Inference smoke test: PASS")

# ------------------------------------------------------------
# 5. Contract validation
# ------------------------------------------------------------

assert len(contract_audit_df) == 10
assert (contract_audit_df["status"] == "PASS").all()

print("Target contract validation: PASS")

# ------------------------------------------------------------
# 6. Evaluation artifact validation
# ------------------------------------------------------------

assert len(predictions_df) == 21468
assert predictions_df["target"].nunique() == 10

assert not predictions_df.isna().any().any()

assert not predictions_df.duplicated(
    subset=["target", "source_index"]
).any()

print("Chronological prediction artifact: PASS")

# ------------------------------------------------------------
# 7. Metric reconciliation
# ------------------------------------------------------------

assert len(selected_comparison) == 5
assert selected_comparison["metrics_match"].all()

print("Selected-model metric reconciliation: PASS")

# ------------------------------------------------------------
# 8. Consolidated audit
# ------------------------------------------------------------

assert len(production_audit_df) == 10

print("Consolidated production audit: PASS")

# ------------------------------------------------------------
# 9. Final summary
# ------------------------------------------------------------

print("\nCLEAN-KERNEL REPRODUCIBILITY SUMMARY")
print("=" * 75)

checks = {
    "Project environment": True,
    "Datasets": True,
    "Production artifacts": True,
    "Inference smoke test": True,
    "Target contracts": True,
    "Evaluation predictions": True,
    "Metric reconciliation": True,
    "Consolidated audit": True
}

for check, status in checks.items():
    print(f"{check}: {'PASS' if status else 'FAIL'}")

print("\nTotal checks:", len(checks))
print("Passed:", sum(checks.values()))
print("Failed:", len(checks) - sum(checks.values()))

print("\nCELL 26 COMPLETED — READ ONLY")

ENVIRONMENT VALIDATION
Python: 3.12.16 | packaged by Anaconda, Inc. | (main, Oct 26 2026, 20:13:05) [MSC v.1942 64 bit (AMD64)]
Project root: D:\Git\Personal\ipl-player-performance-v2
Project root exists: True

Dataset files available: 2
Production model artifacts available: PASS
Inference smoke test: PASS
Target contract validation: PASS
Chronological prediction artifact: PASS
Selected-model metric reconciliation: PASS
Consolidated production audit: PASS

CLEAN-KERNEL REPRODUCIBILITY SUMMARY
Project environment: PASS
Datasets: PASS
Production artifacts: PASS
Inference smoke test: PASS
Target contracts: PASS
Evaluation predictions: PASS
Metric reconciliation: PASS
Consolidated audit: PASS

Total checks: 8
Passed: 8
Failed: 0

CELL 26 COMPLETED — READ ONLY


## 27. Final Production Inference and Model Contract Validation — Completion

### Objective
Validate the selected IPL player-performance models for production inference readiness, input-schema compatibility, output contracts, artifact availability, and reproducibility.

### Selected Production Models

| Prediction target | Selected model |
|---|---|
| Next-match runs scored | Batting Original ML |
| Next-match balls faced | DL MLP |
| Next-match fours | DL MLP |
| Next-match sixes | DL MLP |
| Next-match strike rate | DL MLP |
| Next-match dismissals | DL MLP |
| Next-match wickets | Bowling Wickets ML |
| Next-match runs conceded | DL MLP |
| Next-match legal balls | DL MLP |
| Next-match economy rate | DL MLP |

### Validation Results

- All 10 selected models passed the inference smoke test.
- All 10 target contracts passed validation.
- Required production model artifacts were available.
- Chronological test prediction artifacts were verified.
- Five selected-model metric records were successfully reconciled.
- The consolidated production inference audit passed.
- Clean-kernel reproducibility validation passed all 8 checks.

### Evaluation Period
- Start: 2024-05-02
- End: 2026-05-29

### Known Limitations

1. Six alternative candidate-model records were absent from the selected prediction artifact.
2. Test sample counts vary across prediction targets.
3. Only five selected-model metric records were available for metric reconciliation.
4. The inference smoke test used 10 sample rows per model; it is not an accuracy evaluation.
5. Some target-level evaluation metrics indicate limited predictive performance. Model quality must be communicated using the actual evaluation metrics rather than inference validation results.

### Final Status

**Notebook 12: COMPLETED — VALIDATION PASSED**

The notebook validates the existing selected-model artifacts and their inference contracts. No model retraining or production artifact modification was performed during this validation process.

In [6]:

from pathlib import Path

def find_project_root(start_path=None):
    current = Path(start_path or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        if (
            (candidate / "notebooks").is_dir()
            and (candidate / "src").is_dir()
            and (candidate / "data").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Project root not found. Check your notebook location."
    )

ROOT = find_project_root()

print("Project root:", ROOT)
print(
    "Batting dataset exists:",
    (ROOT / "data/processed/features/batting_model_dataset.csv").is_file()
)
print(
    "Bowling dataset exists:",
    (ROOT / "data/processed/features/bowling_model_dataset.csv").is_file()
)


Project root: D:\Git\Personal\ipl-player-performance-v2
Batting dataset exists: True
Bowling dataset exists: True


In [9]:

# PHASE 3 — STEP 4
# Corrected target-specific eligible sample audit
# READ-ONLY: No source files or artifacts are modified.

from pathlib import Path
import pandas as pd

def find_project_root(start_path=None):
    current = Path(start_path or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        if (
            (candidate / "notebooks").is_dir()
            and (candidate / "src").is_dir()
            and (candidate / "data").is_dir()
        ):
            return candidate

    raise FileNotFoundError("Project root not found.")

ROOT = find_project_root()

TRAIN_END = pd.Timestamp("2022-04-11")
VALIDATION_END = pd.Timestamp("2024-05-01")

configs = {
    "BATTING": {
        "model": "data/processed/features/batting_model_dataset.csv",
        "history": "data/processed/player_match/batting_player_match_regular.csv",
        "targets": [
            "target_next_match_runs_scored",
            "target_next_match_balls_faced",
            "target_next_match_fours",
            "target_next_match_sixes",
            "target_next_match_strike_rate",
            "target_next_match_dismissals",
        ],
    },
    "BOWLING": {
        "model": "data/processed/features/bowling_model_dataset.csv",
        "history": "data/processed/player_match/bowling_player_match_regular.csv",
        "targets": [
            "target_next_match_wickets",
            "target_next_match_runs_conceded",
            "target_next_match_legal_balls",
            "target_next_match_economy_rate",
        ],
    },
}

audit_records = []
partition_records = []

for category, config in configs.items():

    model_df = pd.read_csv(ROOT / config["model"])
    history_df = pd.read_csv(ROOT / config["history"])

    model_df["date"] = pd.to_datetime(model_df["date"])
    history_df["date"] = pd.to_datetime(history_df["date"])

    history_df = history_df.sort_values(
        ["player", "date", "match_id"]
    ).copy()

    history_df["next_appearance_date"] = (
        history_df.groupby("player")["date"].shift(-1)
    )

    model_df = model_df.merge(
        history_df[
            ["match_id", "player", "date", "next_appearance_date"]
        ],
        on=["match_id", "player", "date"],
        how="left",
        validate="one_to_one",
    )

    # A missing next appearance date indicates a mapping problem
    # or an unexpected final-appearance record.
    unmatched = model_df["next_appearance_date"].isna().sum()

    assert unmatched == 0, (
        f"{category}: {unmatched} records have no matched next appearance date."
    )

    train_mask = (
        (model_df["date"] <= TRAIN_END)
        & (model_df["next_appearance_date"] <= TRAIN_END)
    )

    validation_mask = (
        (model_df["date"] > TRAIN_END)
        & (model_df["date"] <= VALIDATION_END)
        & (model_df["next_appearance_date"] <= VALIDATION_END)
    )

    test_mask = model_df["date"] > VALIDATION_END

    # Rows excluded by the boundary-purge policy.
    purged_mask = ~(train_mask | validation_mask | test_mask)

    # Verify masks are mutually exclusive.
    membership_count = (
        train_mask.astype(int)
        + validation_mask.astype(int)
        + test_mask.astype(int)
        + purged_mask.astype(int)
    )

    assert membership_count.eq(1).all(), (
        f"{category}: Partition overlap or unclassified rows detected."
    )

    partitions = {
        "TRAIN": model_df.loc[train_mask].copy(),
        "VALIDATION": model_df.loc[validation_mask].copy(),
        "TEST": model_df.loc[test_mask].copy(),
    }

    purged_df = model_df.loc[purged_mask].copy()

    # Correct coverage assertion: retained + purged = original.
    retained_count = sum(len(df) for df in partitions.values())

    assert retained_count + len(purged_df) == len(model_df), (
        f"{category}: Partition accounting mismatch."
    )

    partition_records.append({
        "category": category,
        "original_rows": len(model_df),
        "train_rows": len(partitions["TRAIN"]),
        "validation_rows": len(partitions["VALIDATION"]),
        "test_rows": len(partitions["TEST"]),
        "purged_rows": len(purged_df),
        "retained_rows": retained_count,
        "accounting_status": "PASS",
    })

    # Target-specific eligibility.
    for target in config["targets"]:

        for split_name, split_df in partitions.items():

            total = len(split_df)
            missing = int(split_df[target].isna().sum())
            eligible = total - missing

            audit_records.append({
                "category": category,
                "target": target,
                "split": split_name,
                "total_rows": total,
                "missing_targets": missing,
                "eligible_rows": eligible,
                "eligibility_pct": round(
                    eligible / total * 100, 2
                ) if total else 0,
            })

# --------------------------------------------------
# Display partition accounting
# --------------------------------------------------

partition_audit_df = pd.DataFrame(partition_records)

print("CHRONOLOGICAL PARTITION ACCOUNTING")
display(partition_audit_df)

assert (
    partition_audit_df["accounting_status"] == "PASS"
).all()

# --------------------------------------------------
# Display target eligibility
# --------------------------------------------------

eligible_audit_df = pd.DataFrame(audit_records)

print("\nTARGET-SPECIFIC ELIGIBLE SAMPLE COUNTS")
display(
    eligible_audit_df.sort_values(
        ["category", "target", "split"]
    ).reset_index(drop=True)
)

summary_df = eligible_audit_df.pivot(
    index=["category", "target"],
    columns="split",
    values="eligible_rows",
).reset_index()

summary_df.columns.name = None

print("\nFINAL TRAIN / VALIDATION / TEST SUMMARY")
display(summary_df)

# --------------------------------------------------
# Final integrity checks
# --------------------------------------------------

assert len(eligible_audit_df) == 30
assert (eligible_audit_df["eligible_rows"] >= 0).all()
assert (
    eligible_audit_df["eligible_rows"]
    <= eligible_audit_df["total_rows"]
).all()

print("\nPASS — Corrected target eligibility audit completed.")
print("Targets audited:", eligible_audit_df["target"].nunique())
print("Partition records:", len(eligible_audit_df))
print("Original files and model artifacts unchanged.")


CHRONOLOGICAL PARTITION ACCOUNTING


,category,original_rows,train_rows,validation_rows,test_rows,purged_rows,retained_rows,accounting_status
0,BATTING,18083,12861,2496,2375,351,17732,PASS
1,BOWLING,14123,10159,1902,1805,257,13866,PASS



TARGET-SPECIFIC ELIGIBLE SAMPLE COUNTS


,category,target,split,total_rows,missing_targets,eligible_rows,eligibility_pct
0,BATTING,target_next_match_balls_faced,TEST,2375,0,2375,100.00
1,BATTING,target_next_match_balls_faced,TRAIN,12861,0,12861,100.00
2,BATTING,target_next_match_balls_faced,VALIDATION,2496,0,2496,100.00
3,BATTING,target_next_match_dismissals,TEST,2375,0,2375,100.00
4,BATTING,target_next_match_dismissals,TRAIN,12861,0,12861,100.00
5,BATTING,target_next_match_dismissals,VALIDATION,2496,0,2496,100.00
6,BATTING,target_next_match_fours,TEST,2375,0,2375,100.00
7,BATTING,target_next_match_fours,TRAIN,12861,0,12861,100.00
8,BATTING,target_next_match_fours,VALIDATION,2496,0,2496,100.00
9,BATTING,target_next_match_runs_scored,TEST,2375,0,2375,100.00



FINAL TRAIN / VALIDATION / TEST SUMMARY


,category,target,TEST,TRAIN,VALIDATION
0,BATTING,target_next_match_balls_faced,2375,12861,2496
1,BATTING,target_next_match_dismissals,2375,12861,2496
2,BATTING,target_next_match_fours,2375,12861,2496
3,BATTING,target_next_match_runs_scored,2375,12861,2496
4,BATTING,target_next_match_sixes,2375,12861,2496
5,BATTING,target_next_match_strike_rate,2373,12817,2488
6,BOWLING,target_next_match_economy_rate,1805,10158,1902
7,BOWLING,target_next_match_legal_balls,1805,10159,1902
8,BOWLING,target_next_match_runs_conceded,1805,10159,1902
9,BOWLING,target_next_match_wickets,1805,10159,1902



PASS — Corrected target eligibility audit completed.
Targets audited: 10
Partition records: 30
Original files and model artifacts unchanged.


In [10]:

# PHASE 3 — STEP 5
# Prediction-time feature availability audit
# CELL 1: Inspect production selection manifest and dataset schemas
# READ-ONLY

from pathlib import Path
import json
import pandas as pd

ROOT = find_project_root()

selection_path = (
    ROOT
    / "artifacts/model_comparison/final_target_model_selection.json"
)

assert selection_path.is_file(), (
    f"Production selection manifest not found: {selection_path}"
)

with open(selection_path, "r", encoding="utf-8") as file:
    selection_manifest = json.load(file)

print("PRODUCTION SELECTION MANIFEST")
print("Path:", selection_path)
print("Top-level structure:", type(selection_manifest).__name__)

if isinstance(selection_manifest, dict):
    print("Top-level keys:", list(selection_manifest.keys()))

print("\nMANIFEST CONTENT")
print(json.dumps(selection_manifest, indent=2, default=str))

# Inspect feature dataset schemas.
dataset_paths = {
    "BATTING": ROOT / "data/processed/features/batting_model_dataset.csv",
    "BOWLING": ROOT / "data/processed/features/bowling_model_dataset.csv",
}

for category, path in dataset_paths.items():

    assert path.is_file(), f"Missing dataset: {path}"

    df = pd.read_csv(path, nrows=5)

    print(f"\n{category} DATASET")
    print("Path:", path)
    print("Columns:", len(df.columns))
    print("Column names:")
    print(df.columns.tolist())

print("\nPASS — Selection manifest and dataset schemas inspected.")
print("No files or artifacts modified.")


PRODUCTION SELECTION MANIFEST
Path: D:\Git\Personal\ipl-player-performance-v2\artifacts\model_comparison\final_target_model_selection.json
Top-level structure: dict
Top-level keys: ['project', 'selection_version', 'created_at_utc', 'selection_policy', 'models']

MANIFEST CONTENT
{
  "project": "IPL Player Performance Analytics V2",
  "selection_version": "1.0",
  "created_at_utc": "2026-11-02T16:39:09.294479+00:00",
  "selection_policy": {
    "primary_metric": "MAE",
    "supporting_metrics": [
      "RMSE",
      "R2",
      "Bias"
    ],
    "test_period": "2024-05-02 to 2026-05-29",
    "baseline": "Training_Mean_Baseline"
  },
  "models": {
    "target_next_match_runs_scored": {
      "model": "batting_original",
      "reason": "Best balanced RMSE and R2; batting_sqrt has marginally lower MAE."
    },
    "target_next_match_balls_faced": {
      "model": "DL_MLP",
      "reason": "Lower MAE and RMSE than training-mean baseline."
    },
    "target_next_match_fours": {
      "mode

In [11]:

# PHASE 3 — STEP 5
# CELL 1: Discover production model contracts
# READ-ONLY: No model or dataset modifications.

from pathlib import Path
import json
import joblib
import pandas as pd

ROOT = find_project_root()

manifest_path = (
    ROOT / "artifacts/model_comparison/final_target_model_selection.json"
)

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

selected_models = manifest["models"]

# Known traditional ML artifact locations.
ml_artifacts = {
    "batting_original": (
        ROOT / "models/batting/original_gb_model.joblib"
    ),
    "bowling_wickets": (
        ROOT / "models/bowling/wickets_model.joblib"
    ),
}

# Discover DL metadata without assuming directory names.
dl_metadata_files = list(
    (ROOT / "artifacts/dl_models").rglob("metadata.json")
)

dl_metadata = {}

for metadata_path in dl_metadata_files:
    try:
        with open(metadata_path, "r", encoding="utf-8") as f:
            metadata = json.load(f)

        target = metadata.get("target")

        if target:
            dl_metadata[target] = {
                "path": metadata_path,
                "metadata": metadata,
            }

    except Exception as exc:
        print("Metadata read warning:", metadata_path, exc)

contract_records = []

for target, selection in selected_models.items():

    selected_model = selection["model"]

    print(f"\n{'=' * 70}")
    print("TARGET:", target)
    print("SELECTED MODEL:", selected_model)

    if selected_model in ml_artifacts:

        artifact_path = ml_artifacts[selected_model]

        assert artifact_path.is_file(), (
            f"Missing ML artifact: {artifact_path}"
        )

        model = joblib.load(artifact_path)

        if not hasattr(model, "feature_names_in_"):
            raise ValueError(
                f"{selected_model}: feature_names_in_ unavailable."
            )

        raw_features = list(model.feature_names_in_)

        model_type = "TRADITIONAL_ML"

    elif selected_model == "DL_MLP":

        if target not in dl_metadata:
            print("Matching DL metadata not found.")
            print("Available metadata targets:", list(dl_metadata))
            raise KeyError(f"Missing DL metadata for {target}")

        metadata_info = dl_metadata[target]
        metadata_path = metadata_info["path"]

        preprocessor_path = metadata_path.parent / "preprocessor.joblib"

        assert preprocessor_path.is_file(), (
            f"Missing preprocessor: {preprocessor_path}"
        )

        preprocessor = joblib.load(preprocessor_path)

        if not hasattr(preprocessor, "feature_names_in_"):
            raise ValueError(
                f"{target}: preprocessor raw input contract unavailable."
            )

        raw_features = list(preprocessor.feature_names_in_)

        model_type = "DEEP_LEARNING"

    else:
        raise ValueError(
            f"Unexpected selected model: {selected_model}"
        )

    category = (
        "BATTING"
        if "runs_scored" in target
        or any(
            x in target
            for x in [
                "balls_faced", "fours", "sixes",
                "strike_rate", "dismissals"
            ]
        )
        else "BOWLING"
    )

    dataset_path = (
        ROOT / "data/processed/features/"
        / (
            "batting_model_dataset.csv"
            if category == "BATTING"
            else "bowling_model_dataset.csv"
        )
    )

    dataset_columns = set(
        pd.read_csv(dataset_path, nrows=0).columns
    )

    missing_features = [
        feature
        for feature in raw_features
        if feature not in dataset_columns
    ]

    print("Model type:", model_type)
    print("Raw feature count:", len(raw_features))
    print("Missing dataset features:", missing_features)
    print("Raw features:")
    print(raw_features)

    contract_records.append({
        "target": target,
        "selected_model": selected_model,
        "model_type": model_type,
        "raw_feature_count": len(raw_features),
        "missing_dataset_features": missing_features,
        "feature_schema_status": (
            "PASS" if not missing_features else "FAIL"
        ),
    })

contract_audit_df = pd.DataFrame(contract_records)

print("\n" + "=" * 70)
print("SELECTED MODEL RAW FEATURE CONTRACT AUDIT")
display(contract_audit_df)

assert len(contract_audit_df) == 10
assert (
    contract_audit_df["feature_schema_status"] == "PASS"
).all(), "One or more selected models have missing raw input features."

print("\nPASS — All 10 selected model raw feature contracts found in datasets.")
print("No model artifacts or source datasets modified.")



TARGET: target_next_match_runs_scored
SELECTED MODEL: batting_original
Model type: TRADITIONAL_ML
Raw feature count: 43
Missing dataset features: []
Raw features:
['batting_team', 'runs_scored', 'balls_faced', 'fours', 'sixes', 'dismissals', 'retired_hurt_count', 'strike_rate', 'batting_average', 'not_out_innings', 'season', 'days_since_previous_match', 'previous_runs_scored', 'runs_scored_rolling_mean_3', 'runs_scored_rolling_mean_5', 'runs_scored_rolling_mean_10', 'runs_scored_expanding_mean', 'previous_balls_faced', 'balls_faced_rolling_mean_3', 'balls_faced_rolling_mean_5', 'balls_faced_rolling_mean_10', 'balls_faced_expanding_mean', 'previous_fours', 'fours_rolling_mean_3', 'fours_rolling_mean_5', 'fours_rolling_mean_10', 'fours_expanding_mean', 'previous_sixes', 'sixes_rolling_mean_3', 'sixes_rolling_mean_5', 'sixes_rolling_mean_10', 'sixes_expanding_mean', 'previous_strike_rate', 'strike_rate_rolling_mean_3', 'strike_rate_rolling_mean_5', 'strike_rate_rolling_mean_10', 'strike_

,target,selected_model,model_type,raw_feature_count,missing_dataset_features,feature_schema_status
0,target_next_match_runs_scored,batting_original,TRADITIONAL_ML,43,[],PASS
1,target_next_match_balls_faced,DL_MLP,DEEP_LEARNING,42,[],PASS
2,target_next_match_fours,DL_MLP,DEEP_LEARNING,42,[],PASS
3,target_next_match_sixes,DL_MLP,DEEP_LEARNING,42,[],PASS
4,target_next_match_strike_rate,DL_MLP,DEEP_LEARNING,42,[],PASS
5,target_next_match_dismissals,DL_MLP,DEEP_LEARNING,42,[],PASS
6,target_next_match_wickets,bowling_wickets,TRADITIONAL_ML,35,[],PASS
7,target_next_match_runs_conceded,DL_MLP,DEEP_LEARNING,30,[],PASS
8,target_next_match_legal_balls,DL_MLP,DEEP_LEARNING,30,[],PASS
9,target_next_match_economy_rate,DL_MLP,DEEP_LEARNING,30,[],PASS



PASS — All 10 selected model raw feature contracts found in datasets.
No model artifacts or source datasets modified.


In [12]:

# PHASE 3 — STEP 5
# CELL 2: Feature origin and prediction-time availability
# READ-ONLY: No datasets or model artifacts modified.

from pathlib import Path
import numpy as np
import pandas as pd
import json

ROOT = find_project_root()

with open(
    ROOT / "artifacts/model_comparison/final_target_model_selection.json",
    "r",
    encoding="utf-8"
) as f:
    manifest = json.load(f)

# --------------------------------------------------
# 1. Define feature categories
# --------------------------------------------------

current_match_features = {
    "runs_scored", "balls_faced", "fours", "sixes",
    "dismissals", "retired_hurt_count",
    "runs_conceded", "legal_balls",
    "wides_conceded", "noballs_conceded",
    "innings_bowled", "wickets",
    "not_out_innings", "innings_batted",
    "is_super_over",
}

metadata_features = {
    "batting_team", "season", "match_type",
}

derived_current_features = {
    "strike_rate", "batting_average",
    "overs_bowled", "bowling_average",
    "bowling_strike_rate", "economy_rate",
}

def classify_feature(feature):

    if (
        feature.startswith("previous_")
        or "_rolling_mean_" in feature
        or feature.endswith("_expanding_mean")
        or feature in {
            "days_since_previous_match",
            "matches_played_before",
        }
    ):
        return "HISTORICAL"

    if feature in current_match_features:
        return "CURRENT_MATCH"

    if feature in metadata_features:
        return "MATCH_METADATA"

    if feature in derived_current_features:
        return "DERIVED_CURRENT_MATCH"

    return "UNCLASSIFIED"


# --------------------------------------------------
# 2. Build feature-level audit
# --------------------------------------------------

feature_records = []

for target, selection in manifest["models"].items():

    selected_model = selection["model"]

    if selected_model == "batting_original":
        model = __import__("joblib").load(
            ROOT / "models/batting/original_gb_model.joblib"
        )
        features = list(model.feature_names_in_)
        category = "BATTING"

    elif selected_model == "bowling_wickets":
        model = __import__("joblib").load(
            ROOT / "models/bowling/wickets_model.joblib"
        )
        features = list(model.feature_names_in_)
        category = "BOWLING"

    else:
        metadata_files = list(
            (ROOT / "artifacts/dl_models").rglob("metadata.json")
        )

        matching_metadata = None

        for path in metadata_files:
            with open(path, "r", encoding="utf-8") as f:
                metadata = json.load(f)

            if metadata.get("target") == target:
                matching_metadata = path
                break

        assert matching_metadata is not None, (
            f"Missing DL metadata for {target}"
        )

        import joblib

        preprocessor = joblib.load(
            matching_metadata.parent / "preprocessor.joblib"
        )

        features = list(preprocessor.feature_names_in_)
        category = (
            "BATTING"
            if target in {
                "target_next_match_runs_scored",
                "target_next_match_balls_faced",
                "target_next_match_fours",
                "target_next_match_sixes",
                "target_next_match_strike_rate",
                "target_next_match_dismissals",
            }
            else "BOWLING"
        )

    for feature in features:
        feature_records.append({
            "target": target,
            "selected_model": selected_model,
            "category": category,
            "feature": feature,
            "feature_origin": classify_feature(feature),
        })

feature_origin_df = pd.DataFrame(feature_records)

print("FEATURE ORIGIN SUMMARY")
display(
    feature_origin_df.groupby(
        ["category", "feature_origin"]
    ).size().unstack(fill_value=0)
)

print("\nUNCLASSIFIED FEATURES")
display(
    feature_origin_df[
        feature_origin_df["feature_origin"] == "UNCLASSIFIED"
    ]
)

# --------------------------------------------------
# 3. Check target/future columns in model contracts
# --------------------------------------------------

forbidden_features = [
    record["feature"]
    for record in feature_records
    if (
        record["feature"].startswith("target_next_match_")
        or record["feature"] in {
            "next_appearance_date",
            "next_match_date",
        }
    )
]

print("\nTARGET / FUTURE FEATURE CHECK")
print("Forbidden features found:", forbidden_features)

assert not forbidden_features, (
    "Target or future-date columns found in model feature contracts."
)

# --------------------------------------------------
# 4. Inspect derived current-match metrics
# --------------------------------------------------

batting = pd.read_csv(
    ROOT / "data/processed/features/batting_model_dataset.csv"
)

bowling = pd.read_csv(
    ROOT / "data/processed/features/bowling_model_dataset.csv"
)

def ratio_check(df, numerator, denominator, actual, label):

    valid = (
        df[numerator].notna()
        & df[denominator].notna()
        & df[actual].notna()
        & df[denominator].ne(0)
    )

    expected = (
        df.loc[valid, numerator]
        / df.loc[valid, denominator]
    )

    observed = df.loc[valid, actual]

    exact_match = np.isclose(
        expected,
        observed,
        rtol=1e-4,
        atol=1e-3,
        equal_nan=False,
    )

    return {
        "metric": label,
        "valid_rows": int(valid.sum()),
        "matching_rows": int(exact_match.sum()),
        "non_matching_rows": int((~exact_match).sum()),
        "match_pct": round(exact_match.mean() * 100, 2)
        if len(exact_match) else np.nan,
    }

derived_checks = [
    ratio_check(
        batting, "runs_scored", "balls_faced",
        "strike_rate", "Batting strike rate"
    ),
    ratio_check(
        batting, "runs_scored", "dismissals",
        "batting_average", "Batting average"
    ),
    ratio_check(
        bowling, "runs_conceded", "wickets",
        "bowling_average", "Bowling average"
    ),
    ratio_check(
        bowling, "legal_balls", "wickets",
        "bowling_strike_rate", "Bowling strike rate"
    ),
    ratio_check(
        bowling, "runs_conceded", "overs_bowled",
        "economy_rate", "Bowling economy rate"
    ),
]

derived_check_df = pd.DataFrame(derived_checks)

print("\nDERIVED METRIC FORMULA AUDIT")
display(derived_check_df)

print("\nOVERS CONVERSION AUDIT")

valid_overs = (
    bowling["legal_balls"].notna()
    & bowling["overs_bowled"].notna()
)

overs_expected = bowling.loc[valid_overs, "legal_balls"] / 6

overs_match = np.isclose(
    overs_expected,
    bowling.loc[valid_overs, "overs_bowled"],
    rtol=1e-4,
    atol=1e-3,
)

print("Valid rows:", int(valid_overs.sum()))
print("Matching rows:", int(overs_match.sum()))
print("Non-matching rows:", int((~overs_match).sum()))

# --------------------------------------------------
# 5. Final report
# --------------------------------------------------

print("\nFEATURE AVAILABILITY AUDIT — PRELIMINARY")

print(
    "Total feature records:",
    len(feature_origin_df)
)

print(
    "Unclassified feature records:",
    int(
        (feature_origin_df["feature_origin"] == "UNCLASSIFIED").sum()
    )
)

print(
    "Target/future features:",
    len(forbidden_features)
)

print("\nNo source files or model artifacts modified.")


FEATURE ORIGIN SUMMARY


feature_origin,CURRENT_MATCH,DERIVED_CURRENT_MATCH,HISTORICAL,MATCH_METADATA
category,,,,
BATTING,42,7,192,12
BOWLING,22,10,88,5



UNCLASSIFIED FEATURES


,target,selected_model,category,feature,feature_origin



TARGET / FUTURE FEATURE CHECK
Forbidden features found: []

DERIVED METRIC FORMULA AUDIT


,metric,valid_rows,matching_rows,non_matching_rows,match_pct
0,Batting strike rate,18026,1509,16517,8.37
1,Batting average,14121,14121,0,100.00
2,Bowling average,8239,8239,0,100.00
3,Bowling strike rate,8239,8239,0,100.00
4,Bowling economy rate,14122,14122,0,100.00



OVERS CONVERSION AUDIT
Valid rows: 14123
Matching rows: 14123
Non-matching rows: 0

FEATURE AVAILABILITY AUDIT — PRELIMINARY
Total feature records: 378
Unclassified feature records: 0
Target/future features: 0

No source files or model artifacts modified.


In [13]:

# PHASE 3 — STEP 5
# CELL 3: Correct batting strike-rate formula validation
# READ-ONLY

import numpy as np
import pandas as pd

ROOT = find_project_root()

batting = pd.read_csv(
    ROOT / "data/processed/features/batting_model_dataset.csv"
)

valid = (
    batting["runs_scored"].notna()
    & batting["balls_faced"].notna()
    & batting["strike_rate"].notna()
    & batting["balls_faced"].gt(0)
)

expected_strike_rate = (
    batting.loc[valid, "runs_scored"]
    / batting.loc[valid, "balls_faced"]
) * 100

actual_strike_rate = batting.loc[valid, "strike_rate"]

matches = np.isclose(
    expected_strike_rate,
    actual_strike_rate,
    rtol=1e-4,
    atol=1e-3
)

audit = pd.DataFrame([{
    "metric": "Batting strike rate",
    "valid_rows": int(valid.sum()),
    "matching_rows": int(matches.sum()),
    "non_matching_rows": int((~matches).sum()),
    "match_pct": round(matches.mean() * 100, 4),
}])

print("CORRECTED STRIKE RATE FORMULA AUDIT")
display(audit)

if (~matches).any():
    mismatch_positions = np.flatnonzero(~matches)
    mismatch_indices = actual_strike_rate.index[mismatch_positions]

    print("\nFIRST 20 MISMATCHED RECORDS")

    display(
        batting.loc[
            mismatch_indices,
            [
                "match_id",
                "player",
                "runs_scored",
                "balls_faced",
                "strike_rate",
            ]
        ].head(20).assign(
            expected_strike_rate=expected_strike_rate.loc[
                mismatch_indices
            ].head(20).values
        )
    )

assert matches.all(), (
    "Unexpected batting strike-rate formula mismatches found."
)

print("\nPASS — Batting strike rate matches the corrected formula.")
print("No files or artifacts modified.")


CORRECTED STRIKE RATE FORMULA AUDIT


,metric,valid_rows,matching_rows,non_matching_rows,match_pct
0,Batting strike rate,18026,18026,0,100.0



PASS — Batting strike rate matches the corrected formula.
No files or artifacts modified.


In [14]:

# PHASE 3 — STEP 6
# Cross-target cricket-domain consistency audit
# READ-ONLY

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = find_project_root()

prediction_path = (
    ROOT / "artifacts/model_comparison/constrained_test_predictions.csv"
)

assert prediction_path.is_file(), (
    f"Prediction file not found: {prediction_path}"
)

predictions = pd.read_csv(prediction_path)

required_columns = {
    "category",
    "target",
    "match_id",
    "player",
    "date",
    "constrained_prediction",
}

missing_columns = required_columns - set(predictions.columns)

assert not missing_columns, (
    f"Missing prediction columns: {missing_columns}"
)

# --------------------------------------------------
# 1. Pivot target predictions into player-match rows
# --------------------------------------------------

index_columns = ["category", "match_id", "player", "date"]

wide = predictions.pivot(
    index=index_columns,
    columns="target",
    values="constrained_prediction",
).reset_index()

wide.columns.name = None

print("Combined player-match prediction rows:", len(wide))
print("Available prediction targets:", len(wide.columns) - len(index_columns))

# --------------------------------------------------
# 2. Batting consistency checks
# --------------------------------------------------

batting = wide[wide["category"] == "batting"].copy()

def available(df, columns):
    return all(column in df.columns for column in columns)

batting_checks = []

batting_targets = [
    "target_next_match_runs_scored",
    "target_next_match_balls_faced",
    "target_next_match_fours",
    "target_next_match_sixes",
    "target_next_match_strike_rate",
    "target_next_match_dismissals",
]

if available(batting, batting_targets):

    runs = batting["target_next_match_runs_scored"]
    balls = batting["target_next_match_balls_faced"]
    fours = batting["target_next_match_fours"]
    sixes = batting["target_next_match_sixes"]
    strike_rate = batting["target_next_match_strike_rate"]
    dismissals = batting["target_next_match_dismissals"]

    checks = {
        "Runs >= boundary runs": runs >= (4 * fours + 6 * sixes),
        "Boundary count <= balls faced": (fours + sixes) <= balls,
        "Dismissals nonnegative": dismissals >= 0,
        "Runs nonnegative": runs >= 0,
        "Balls faced nonnegative": balls >= 0,
    }

    valid_sr = balls > 0

    expected_sr = np.where(
        valid_sr,
        runs / balls * 100,
        np.nan,
    )

    sr_match = np.isclose(
        expected_sr,
        strike_rate,
        atol=0.02,
        rtol=0,
        equal_nan=False,
    )

    checks["Strike rate matches predicted components"] = (
        pd.Series(sr_match, index=batting.index).where(valid_sr, True)
    )

    for name, result in checks.items():
        batting_checks.append({
            "category": "BATTING",
            "consistency_rule": name,
            "checked_rows": len(result),
            "violations": int((~result).sum()),
            "status": "PASS" if result.all() else "REVIEW",
        })

else:
    print("Batting targets incomplete; batting checks skipped.")

# --------------------------------------------------
# 3. Bowling consistency checks
# --------------------------------------------------

bowling = wide[wide["category"] == "bowling"].copy()

bowling_targets = [
    "target_next_match_wickets",
    "target_next_match_runs_conceded",
    "target_next_match_legal_balls",
    "target_next_match_economy_rate",
]

bowling_checks = []

if available(bowling, bowling_targets):

    wickets = bowling["target_next_match_wickets"]
    runs_conceded = bowling["target_next_match_runs_conceded"]
    legal_balls = bowling["target_next_match_legal_balls"]
    economy = bowling["target_next_match_economy_rate"]

    checks = {
        "Wickets <= legal balls": wickets <= legal_balls,
        "Wickets nonnegative": wickets >= 0,
        "Runs conceded nonnegative": runs_conceded >= 0,
        "Legal balls nonnegative": legal_balls >= 0,
    }

    valid_economy = legal_balls > 0

    expected_economy = np.where(
        valid_economy,
        runs_conceded * 6 / legal_balls,
        np.nan,
    )

    economy_match = np.isclose(
        expected_economy,
        economy,
        atol=0.02,
        rtol=0,
        equal_nan=False,
    )

    checks["Economy matches predicted components"] = (
        pd.Series(economy_match, index=bowling.index)
        .where(valid_economy, True)
    )

    for name, result in checks.items():
        bowling_checks.append({
            "category": "BOWLING",
            "consistency_rule": name,
            "checked_rows": len(result),
            "violations": int((~result).sum()),
            "status": "PASS" if result.all() else "REVIEW",
        })

else:
    print("Bowling targets incomplete; bowling checks skipped.")

# --------------------------------------------------
# 4. Final audit report
# --------------------------------------------------

consistency_audit_df = pd.DataFrame(
    batting_checks + bowling_checks
)

print("\nCROSS-TARGET CONSISTENCY RESULTS")
display(consistency_audit_df)

# Show offending records for investigation.
failed_rules = consistency_audit_df[
    consistency_audit_df["violations"] > 0
]

if not failed_rules.empty:

    print("\nRULES REQUIRING REVIEW:")
    display(failed_rules)

    print("\nCombined prediction sample:")
    display(wide.head(10))

else:
    print("\nPASS — All tested cross-target consistency rules satisfied.")

print("\nNo predictions, models, or source files modified.")


Combined player-match prediction rows: 4180
Available prediction targets: 10

CROSS-TARGET CONSISTENCY RESULTS


,category,consistency_rule,checked_rows,violations,status
0,BATTING,Runs >= boundary runs,2375,10,REVIEW
1,BATTING,Boundary count <= balls faced,2375,0,PASS
2,BATTING,Dismissals nonnegative,2375,0,PASS
3,BATTING,Runs nonnegative,2375,0,PASS
4,BATTING,Balls faced nonnegative,2375,0,PASS
5,BATTING,Strike rate matches predicted components,2375,2374,REVIEW
6,BOWLING,Wickets <= legal balls,1805,0,PASS
7,BOWLING,Wickets nonnegative,1805,0,PASS
8,BOWLING,Runs conceded nonnegative,1805,0,PASS
9,BOWLING,Legal balls nonnegative,1805,0,PASS



RULES REQUIRING REVIEW:


,category,consistency_rule,checked_rows,violations,status
0,BATTING,Runs >= boundary runs,2375,10,REVIEW
5,BATTING,Strike rate matches predicted components,2375,2374,REVIEW
10,BOWLING,Economy matches predicted components,1805,1746,REVIEW



Combined prediction sample:


,category,match_id,player,date,target_next_match_balls_faced,target_next_match_dismissals,target_next_match_economy_rate,target_next_match_fours,target_next_match_legal_balls,target_next_match_runs_conceded,target_next_match_runs_scored,target_next_match_sixes,target_next_match_strike_rate,target_next_match_wickets
0,batting,1426288,Abhishek Sharma,2024-05-02,17.0,1.0,NaN,2.0,NaN,NaN,20.0,1.0,113.570000,NaN
1,batting,1426288,Dhruv Jurel,2024-05-02,11.0,1.0,NaN,1.0,NaN,NaN,14.0,1.0,116.599998,NaN
2,batting,1426288,H Klaasen,2024-05-02,16.0,1.0,NaN,2.0,NaN,NaN,20.0,1.0,126.080002,NaN
3,batting,1426288,JC Buttler,2024-05-02,24.0,1.0,NaN,3.0,NaN,NaN,31.0,1.0,107.989998,NaN
4,batting,1426288,Nithish Kumar Reddy,2024-05-02,14.0,1.0,NaN,2.0,NaN,NaN,22.0,1.0,124.239998,NaN
5,batting,1426288,R Ashwin,2024-05-02,9.0,1.0,NaN,1.0,NaN,NaN,12.0,0.0,115.459999,NaN
6,batting,1426288,R Parag,2024-05-02,17.0,1.0,NaN,2.0,NaN,NaN,22.0,1.0,125.669998,NaN
7,batting,1426288,R Powell,2024-05-02,13.0,1.0,NaN,1.0,NaN,NaN,20.0,1.0,129.809998,NaN
8,batting,1426288,SO Hetmyer,2024-05-02,9.0,1.0,NaN,1.0,NaN,NaN,15.0,1.0,121.930000,NaN
9,batting,1426288,SV Samson,2024-05-02,19.0,1.0,NaN,2.0,NaN,NaN,30.0,1.0,115.730003,NaN



No predictions, models, or source files modified.


In [15]:

# PHASE 3 — STEP 6
# CELL 2: Detailed cross-target inconsistency diagnosis
# READ-ONLY

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = find_project_root()

prediction_path = (
    ROOT / "artifacts/model_comparison/constrained_test_predictions.csv"
)

pred = pd.read_csv(prediction_path)

wide = pred.pivot(
    index=["category", "match_id", "player", "date"],
    columns="target",
    values="constrained_prediction",
).reset_index()

wide.columns.name = None

# --------------------------------------------------
# 1. Batting boundary consistency
# --------------------------------------------------

bat = wide[wide["category"] == "batting"].copy()

bat["minimum_boundary_runs"] = (
    4 * bat["target_next_match_fours"]
    + 6 * bat["target_next_match_sixes"]
)

bat["boundary_run_deficit"] = (
    bat["minimum_boundary_runs"]
    - bat["target_next_match_runs_scored"]
)

bat_violations = bat[
    bat["boundary_run_deficit"] > 0
].copy()

print("BATTING BOUNDARY VIOLATIONS:", len(bat_violations))

display(
    bat_violations[
        [
            "match_id", "player",
            "target_next_match_runs_scored",
            "target_next_match_fours",
            "target_next_match_sixes",
            "minimum_boundary_runs",
            "boundary_run_deficit",
        ]
    ].sort_values(
        "boundary_run_deficit",
        ascending=False
    )
)

# --------------------------------------------------
# 2. Strike-rate consistency
# --------------------------------------------------

bat["derived_strike_rate"] = np.where(
    bat["target_next_match_balls_faced"] > 0,
    (
        bat["target_next_match_runs_scored"]
        / bat["target_next_match_balls_faced"]
    ) * 100,
    np.nan,
)

bat["strike_rate_abs_difference"] = (
    bat["target_next_match_strike_rate"]
    - bat["derived_strike_rate"]
).abs()

sr_diagnostics = bat[
    bat["strike_rate_abs_difference"].notna()
].copy()

print("\nSTRIKE-RATE DISCREPANCY SUMMARY")

display(
    sr_diagnostics["strike_rate_abs_difference"].describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    ).to_frame("absolute_difference")
)

print("\nLARGEST STRIKE-RATE DISCREPANCIES")

display(
    sr_diagnostics.nlargest(
        15, "strike_rate_abs_difference"
    )[
        [
            "match_id", "player",
            "target_next_match_runs_scored",
            "target_next_match_balls_faced",
            "derived_strike_rate",
            "target_next_match_strike_rate",
            "strike_rate_abs_difference",
        ]
    ]
)

# --------------------------------------------------
# 3. Economy-rate consistency
# --------------------------------------------------

bowl = wide[wide["category"] == "bowling"].copy()

bowl["derived_economy_rate"] = np.where(
    bowl["target_next_match_legal_balls"] > 0,
    (
        bowl["target_next_match_runs_conceded"] * 6
        / bowl["target_next_match_legal_balls"]
    ),
    np.nan,
)

bowl["economy_abs_difference"] = (
    bowl["target_next_match_economy_rate"]
    - bowl["derived_economy_rate"]
).abs()

economy_diagnostics = bowl[
    bowl["economy_abs_difference"].notna()
].copy()

print("\nECONOMY-RATE DISCREPANCY SUMMARY")

display(
    economy_diagnostics["economy_abs_difference"].describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    ).to_frame("absolute_difference")
)

print("\nLARGEST ECONOMY-RATE DISCREPANCIES")

display(
    economy_diagnostics.nlargest(
        15, "economy_abs_difference"
    )[
        [
            "match_id", "player",
            "target_next_match_runs_conceded",
            "target_next_match_legal_balls",
            "derived_economy_rate",
            "target_next_match_economy_rate",
            "economy_abs_difference",
        ]
    ]
)

# --------------------------------------------------
# 4. Summary
# --------------------------------------------------

diagnostic_summary = pd.DataFrame([
    {
        "issue": "Runs below boundary runs",
        "records": len(bat_violations),
    },
    {
        "issue": "Strike-rate discrepancies",
        "records": int(
            (sr_diagnostics["strike_rate_abs_difference"] > 0.02).sum()
        ),
    },
    {
        "issue": "Economy-rate discrepancies",
        "records": int(
            (economy_diagnostics["economy_abs_difference"] > 0.02).sum()
        ),
    },
])

print("\nDIAGNOSTIC SUMMARY")
display(diagnostic_summary)

print("\nPASS — Diagnostic extraction completed.")
print("No prediction files, models, or artifacts modified.")


BATTING BOUNDARY VIOLATIONS: 10


,match_id,player,target_next_match_runs_scored,target_next_match_fours,target_next_match_sixes,minimum_boundary_runs,boundary_run_deficit
1197,1473494,Ramandeep Singh,7.0,1.0,1.0,10.0,3.0
171,1426298,Ramandeep Singh,8.0,1.0,1.0,10.0,2.0
1445,1527675,Ramandeep Singh,8.0,1.0,1.0,10.0,2.0
1321,1473505,Ramandeep Singh,8.0,1.0,1.0,10.0,2.0
254,1426305,R Shepherd,9.0,1.0,1.0,10.0,1.0
354,1473438,Ramandeep Singh,9.0,1.0,1.0,10.0,1.0
1510,1527679,Ramandeep Singh,9.0,1.0,1.0,10.0,1.0
1576,1527683,Shivang Kumar,9.0,1.0,1.0,10.0,1.0
1705,1527693,R Shepherd,9.0,1.0,1.0,10.0,1.0
2023,1529285,JO Holder,9.0,1.0,1.0,10.0,1.0



STRIKE-RATE DISCREPANCY SUMMARY


,absolute_difference
count,2373.000000
mean,27.341240
std,21.864088
min,0.018830
50%,23.232726
90%,53.663713
95%,66.472244
99%,95.754001
max,253.800003



LARGEST STRIKE-RATE DISCREPANCIES


,match_id,player,target_next_match_runs_scored,target_next_match_balls_faced,derived_strike_rate,target_next_match_strike_rate,strike_rate_abs_difference
641,1473456,Abdul Samad,14.0,4.0,350.000000,96.199997,253.800003
1111,1473488,Rashid Khan,11.0,4.0,275.000000,79.360001,195.639999
1607,1527686,SN Thakur,8.0,3.0,266.666667,83.570000,183.096667
1006,1473482,Abdul Samad,14.0,5.0,280.000000,114.459999,165.540001
1282,1473503,B Kumar,7.0,3.0,233.333333,81.529999,151.803335
808,1473467,SN Thakur,7.0,3.0,233.333333,87.949997,145.383336
857,1473471,B Kumar,9.0,4.0,225.000000,81.160004,143.839996
980,1473480,A Kamboj,6.0,3.0,200.000000,70.339996,129.660004
1071,1473486,A Kamboj,6.0,3.0,200.000000,71.570000,128.430000
799,1473467,Abdul Samad,14.0,6.0,233.333333,106.930000,126.403333


... (3 outputs left out)

,match_id,player,target_next_match_runs_conceded,target_next_match_legal_balls,derived_economy_rate,target_next_match_economy_rate,economy_abs_difference
2674,1473441,T Stubbs,10.0,5.0,12.000000,6.87,5.130000
2456,1426295,A Badoni,12.0,6.0,12.000000,6.88,5.120000
3392,1473507,A Badoni,10.0,5.0,12.000000,7.15,4.850000
2854,1473457,PHKD Mendis,12.0,7.0,10.285714,7.07,3.215714
3368,1473504,S Dube,11.0,6.0,11.000000,7.92,3.080000
3757,1529272,AK Markram,12.0,6.0,12.000000,9.00,3.000000
2914,1473462,RA Jadeja,20.0,11.0,10.909091,8.49,2.419091
3742,1529270,PP Hinge,36.0,24.0,9.000000,6.70,2.300000
2592,1426309,TM Head,21.0,12.0,10.500000,8.28,2.220000
2938,1473464,Shashank Singh,11.0,7.0,9.428571,7.31,2.118571



DIAGNOSTIC SUMMARY


,issue,records
0,Runs below boundary runs,10
1,Strike-rate discrepancies,2372
2,Economy-rate discrepancies,1746



PASS — Diagnostic extraction completed.
No prediction files, models, or artifacts modified.


In [16]:

# PHASE 4 — STEP 1
# CELL 1: Live MySQL schema inspection
# READ-ONLY — No database modifications.

from pathlib import Path
import os
import pandas as pd
import mysql.connector
from dotenv import load_dotenv

ROOT = find_project_root()

# Load credentials without displaying them.
load_dotenv(ROOT / ".env", override=False)

DB_CONFIG = {
    "host": os.getenv("DB_HOST", "127.0.0.1"),
    "port": int(os.getenv("DB_PORT", "3306")),
    "user": os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD"),
    "database": os.getenv("DB_NAME", "ipl_analytics"),
    "connection_timeout": 10,
}

if not DB_CONFIG["user"] or not DB_CONFIG["password"]:
    raise ValueError(
        "DB_USER or DB_PASSWORD missing. "
        "Check your local .env file."
    )

connection = None

try:
    connection = mysql.connector.connect(**DB_CONFIG)

    cursor = connection.cursor(dictionary=True)

    print("MYSQL CONNECTION")
    print("Status: CONNECTED")
    print("Database:", DB_CONFIG["database"])
    print("Host:", DB_CONFIG["host"])
    print("Port:", DB_CONFIG["port"])

    # --------------------------------------------------
    # 1. Database and table inventory
    # --------------------------------------------------

    cursor.execute("""
        SELECT
            TABLE_NAME,
            ENGINE,
            TABLE_COLLATION
        FROM information_schema.TABLES
        WHERE TABLE_SCHEMA = %s
          AND TABLE_TYPE = 'BASE TABLE'
        ORDER BY TABLE_NAME
    """, (DB_CONFIG["database"],))

    table_records = cursor.fetchall()
    tables_df = pd.DataFrame(table_records)

    print("\nDATABASE TABLE INVENTORY")
    display(tables_df)

    # --------------------------------------------------
    # 2. Exact row counts
    # --------------------------------------------------

    row_count_records = []

    for record in table_records:
        table_name = record["TABLE_NAME"]

        # Table name originates from information_schema.
        cursor.execute(
            f"SELECT COUNT(*) AS row_count FROM `{table_name}`"
        )

        count = cursor.fetchone()["row_count"]

        row_count_records.append({
            "table_name": table_name,
            "exact_row_count": count,
        })

    row_counts_df = pd.DataFrame(row_count_records)

    print("\nEXACT TABLE ROW COUNTS")
    display(row_counts_df)

    # --------------------------------------------------
    # 3. Column definitions
    # --------------------------------------------------

    cursor.execute("""
        SELECT
            TABLE_NAME,
            COLUMN_NAME,
            COLUMN_TYPE,
            IS_NULLABLE,
            COLUMN_DEFAULT,
            COLUMN_KEY,
            EXTRA
        FROM information_schema.COLUMNS
        WHERE TABLE_SCHEMA = %s
        ORDER BY TABLE_NAME, ORDINAL_POSITION
    """, (DB_CONFIG["database"],))

    columns_df = pd.DataFrame(cursor.fetchall())

    print("\nCOLUMN DEFINITIONS")
    display(columns_df)

    # --------------------------------------------------
    # 4. Primary and foreign keys
    # --------------------------------------------------

    cursor.execute("""
        SELECT
            TABLE_NAME,
            COLUMN_NAME,
            CONSTRAINT_NAME,
            REFERENCED_TABLE_NAME,
            REFERENCED_COLUMN_NAME
        FROM information_schema.KEY_COLUMN_USAGE
        WHERE TABLE_SCHEMA = %s
        ORDER BY TABLE_NAME, CONSTRAINT_NAME, ORDINAL_POSITION
    """, (DB_CONFIG["database"],))

    constraints_df = pd.DataFrame(cursor.fetchall())

    print("\nPRIMARY AND FOREIGN KEY CONSTRAINTS")
    display(constraints_df)

    # --------------------------------------------------
    # 5. Index definitions
    # --------------------------------------------------

    cursor.execute("""
        SELECT
            TABLE_NAME,
            INDEX_NAME,
            COLUMN_NAME,
            NON_UNIQUE,
            SEQ_IN_INDEX,
            INDEX_TYPE
        FROM information_schema.STATISTICS
        WHERE TABLE_SCHEMA = %s
        ORDER BY TABLE_NAME, INDEX_NAME, SEQ_IN_INDEX
    """, (DB_CONFIG["database"],))

    indexes_df = pd.DataFrame(cursor.fetchall())

    print("\nINDEX DEFINITIONS")
    display(indexes_df)

    # --------------------------------------------------
    # 6. Final inspection summary
    # --------------------------------------------------

    print("\nDATABASE INSPECTION SUMMARY")
    print("Tables:", len(tables_df))
    print("Columns:", len(columns_df))
    print("Constraints:", len(constraints_df))
    print("Index entries:", len(indexes_df))
    print("Total records:", row_counts_df["exact_row_count"].sum())

    print("\nPASS — Live MySQL schema inspection completed.")
    print("No database objects or records modified.")

finally:
    if connection is not None and connection.is_connected():
        cursor.close()
        connection.close()
        print("\nMySQL connection closed.")


MYSQL CONNECTION
Status: CONNECTED
Database: ipl_analytics
Host: 127.0.0.1
Port: 3306

DATABASE TABLE INVENTORY


,TABLE_NAME,ENGINE,TABLE_COLLATION
0,batting_next_match_features,InnoDB,utf8mb4_unicode_ci
1,batting_player_match_regular,InnoDB,utf8mb4_unicode_ci
2,bowling_next_match_features,InnoDB,utf8mb4_unicode_ci
3,bowling_player_match_regular,InnoDB,utf8mb4_unicode_ci
4,deliveries,InnoDB,utf8mb4_unicode_ci
5,innings,InnoDB,utf8mb4_unicode_ci
6,matches,InnoDB,utf8mb4_unicode_ci



EXACT TABLE ROW COUNTS


,table_name,exact_row_count
0,batting_next_match_features,18822
1,batting_player_match_regular,18822
2,bowling_next_match_features,14700
3,bowling_player_match_regular,14700
4,deliveries,295732
5,innings,2514
6,matches,1243



COLUMN DEFINITIONS


,TABLE_NAME,COLUMN_NAME,COLUMN_TYPE,IS_NULLABLE,COLUMN_DEFAULT,COLUMN_KEY,EXTRA
0,batting_next_match_features,match_id,varchar(20),NO,None,PRI,
1,batting_next_match_features,player,varchar(100),NO,None,PRI,
2,batting_next_match_features,is_super_over,tinyint(1),NO,None,,
3,batting_next_match_features,batting_team,varchar(100),NO,None,,
4,batting_next_match_features,runs_scored,int,NO,None,,
...,...,...,...,...,...,...,...
188,matches,bowl_out,int,YES,None,,
189,matches,player_of_match,varchar(100),NO,None,,
190,matches,event_name,varchar(150),NO,None,,
191,matches,event_match_number,int,YES,None,,



PRIMARY AND FOREIGN KEY CONSTRAINTS


,TABLE_NAME,COLUMN_NAME,CONSTRAINT_NAME,REFERENCED_TABLE_NAME,REFERENCED_COLUMN_NAME
0,batting_next_match_features,match_id,fk_batting_next_match_features_matches,matches,match_id
1,batting_next_match_features,match_id,PRIMARY,NaN,NaN
2,batting_next_match_features,player,PRIMARY,NaN,NaN
3,batting_player_match_regular,match_id,fk_batting_player_match_regular_matches,matches,match_id
4,batting_player_match_regular,match_id,PRIMARY,NaN,NaN
5,batting_player_match_regular,player,PRIMARY,NaN,NaN
6,bowling_next_match_features,match_id,fk_bowling_next_match_features_matches,matches,match_id
7,bowling_next_match_features,match_id,PRIMARY,NaN,NaN
8,bowling_next_match_features,player,PRIMARY,NaN,NaN
9,bowling_player_match_regular,match_id,fk_bowling_player_match_regular_matches,matches,match_id



INDEX DEFINITIONS


,TABLE_NAME,INDEX_NAME,COLUMN_NAME,NON_UNIQUE,SEQ_IN_INDEX,INDEX_TYPE
0,batting_next_match_features,PRIMARY,match_id,0,1,BTREE
1,batting_next_match_features,PRIMARY,player,0,2,BTREE
2,batting_player_match_regular,idx_batting_player_date,player,1,1,BTREE
3,batting_player_match_regular,idx_batting_player_date,date,1,2,BTREE
4,batting_player_match_regular,PRIMARY,match_id,0,1,BTREE
5,batting_player_match_regular,PRIMARY,player,0,2,BTREE
6,bowling_next_match_features,PRIMARY,match_id,0,1,BTREE
7,bowling_next_match_features,PRIMARY,player,0,2,BTREE
8,bowling_player_match_regular,idx_bowling_player_date,player,1,1,BTREE
9,bowling_player_match_regular,idx_bowling_player_date,date,1,2,BTREE



DATABASE INSPECTION SUMMARY
Tables: 7
Columns: 193
Constraints: 22
Index entries: 19
Total records: 366533

PASS — Live MySQL schema inspection completed.
No database objects or records modified.

MySQL connection closed.


In [17]:

from pathlib import Path
import yaml
import importlib.util

ROOT = find_project_root()

print("PROJECT ROOT:", ROOT)

print("\nEXISTING APPLICATION DIRECTORIES")
for folder in [
    "config",
    "src/ipl_analytics",
    "scripts",
    "tests",
    "deployment",
    "docs",
    "models",
    "artifacts",
]:
    path = ROOT / folder
    print(f"{folder:25} Exists={path.exists()}")
    if path.exists():
        print("   ", [p.name for p in path.iterdir()][:20])

print("\nDJANGO PACKAGE AVAILABILITY")
for package in ["django", "rest_framework", "mysql.connector", "dotenv"]:
    try:
        print(package, "AVAILABLE" if importlib.util.find_spec(package) else "NOT INSTALLED")
    except ModuleNotFoundError:
        print(package, "NOT INSTALLED")

print("\nENVIRONMENT FILE")
environment_path = ROOT / "environment.yml"

if environment_path.exists():
    with open(environment_path, encoding="utf-8") as f:
        environment = yaml.safe_load(f)

    print("Environment:", environment.get("name"))
    print("Python:", [
        dep for dep in environment.get("dependencies", [])
        if isinstance(dep, str) and dep.startswith("python=")
    ])

print("\nPASS — Existing Django architecture prerequisites inspected.")
print("No files modified.")


PROJECT ROOT: D:\Git\Personal\ipl-player-performance-v2

EXISTING APPLICATION DIRECTORIES
config                    Exists=True
    []
src/ipl_analytics         Exists=True
    ['data', 'database', 'features', 'models', '__init__.py']
scripts                   Exists=True
    ['import_csv_to_mysql.py']
tests                     Exists=True
    []
deployment                Exists=True
    []
docs                      Exists=True
    ['data_dictionary.csv', 'schema.sql']
models                    Exists=True
    ['batting', 'bowling', 'bowling_economy_rate', 'bowling_legal_balls', 'bowling_runs_conceded', 'bowling_wickets', 'manifests']
artifacts                 Exists=True
    ['dl_models', 'dl_validation', 'figures', 'final_reports', 'metrics', 'model_comparison', 'reports']

DJANGO PACKAGE AVAILABILITY
django NOT INSTALLED
rest_framework NOT INSTALLED
mysql.connector AVAILABLE
dotenv AVAILABLE

ENVIRONMENT FILE
Environment: ipl-analytics-v2
Python: ['python=3.12']

PASS — Existing Dja

In [18]:

# PHASE 4 — STEP 2
# CELL 1: Django dependency compatibility inspection
# READ-ONLY — No package installation or file modifications.

from pathlib import Path
import importlib.metadata as metadata
import yaml

ROOT = find_project_root()

packages = [
    "Django",
    "djangorestframework",
    "mysqlclient",
    "PyMySQL",
    "mysql-connector-python",
    "python-dotenv",
    "Pillow",
]

print("INSTALLED PACKAGE VERSIONS")

for package in packages:
    try:
        version = metadata.version(package)
        print(f"{package:28} {version}")
    except metadata.PackageNotFoundError:
        print(f"{package:28} NOT INSTALLED")

print("\nENVIRONMENT.YML CONTENT")

environment_file = ROOT / "environment.yml"

if environment_file.exists():
    with open(environment_file, encoding="utf-8") as file:
        env_data = yaml.safe_load(file)

    print("Environment:", env_data.get("name"))

    print("\nConda dependencies:")
    for dependency in env_data.get("dependencies", []):
        if isinstance(dependency, str):
            print(" ", dependency)
        elif isinstance(dependency, dict):
            for manager, values in dependency.items():
                print(f"\n{manager}:")
                for value in values:
                    print(" ", value)

print("\nCONFIG DIRECTORY")

config_dir = ROOT / "config"

if config_dir.exists():
    for path in config_dir.rglob("*"):
        if path.is_file():
            print(path.relative_to(ROOT))

print("\nGITIGNORE STATUS")
gitignore = ROOT / ".gitignore"
print("Exists:", gitignore.exists())

print("\nPASS — Dependency inspection completed.")
print("No packages or files modified.")


INSTALLED PACKAGE VERSIONS
Django                       NOT INSTALLED
djangorestframework          NOT INSTALLED
mysqlclient                  NOT INSTALLED
PyMySQL                      1.2.3
mysql-connector-python       26.10.0
python-dotenv                1.2.4
Pillow                       13.0.0

ENVIRONMENT.YML CONTENT
Environment: ipl-analytics-v2

Conda dependencies:
  python=3.12
  pip
  numpy
  pandas
  matplotlib
  seaborn
  scikit-learn
  scipy
  pyyaml
  python-dotenv
  joblib
  pytest
  jupyter
  ipykernel
  sqlalchemy
  xgboost
  catboost
  tensorflow

CONFIG DIRECTORY

GITIGNORE STATUS
Exists: True

PASS — Dependency inspection completed.
No packages or files modified.


In [19]:

import django
import pymysql
import sys

print("PYTHON:", sys.version)
print("DJANGO:", django.get_version())
print("PYMYSQL:", pymysql.__version__)

assert django.VERSION[:2] == (5, 2), (
    f"Expected Django 5.2, found {django.get_version()}"
)

assert pymysql.__version__, "PyMySQL version unavailable"

print("\nPASS — Django and PyMySQL are available.")


PYTHON: 3.12.16 | packaged by Anaconda, Inc. | (main, Oct 26 2026, 20:13:05) [MSC v.1942 64 bit (AMD64)]
DJANGO: 5.2.19
PYMYSQL: 2.2.8

PASS — Django and PyMySQL are available.


In [20]:

# PHASE 4 — STEP 2.3
# Django + PyMySQL + MySQL integration test
# READ-ONLY — No files or database objects modified.

import subprocess
import sys

ROOT = find_project_root()

test_script = r'''
import os
import django
import pymysql

from pathlib import Path
from dotenv import load_dotenv

# Resolve project root independently in the child process.
root = Path.cwd()
load_dotenv(root / ".env", override=False)

# Register PyMySQL as the MySQLdb-compatible driver.
pymysql.install_as_MySQLdb()

from django.conf import settings

settings.configure(
    SECRET_KEY="temporary-read-only-connection-test",
    DEBUG=False,
    INSTALLED_APPS=[],
    DATABASES={
        "default": {
            "ENGINE": "django.db.backends.mysql",
            "NAME": os.getenv("DB_NAME", "ipl_analytics"),
            "USER": os.getenv("DB_USER"),
            "PASSWORD": os.getenv("DB_PASSWORD"),
            "HOST": os.getenv("DB_HOST", "127.0.0.1"),
            "PORT": os.getenv("DB_PORT", "3306"),
            "OPTIONS": {
                "charset": "utf8mb4",
                "connect_timeout": 10,
            },
        }
    },
)

if not settings.DATABASES["default"]["USER"]:
    raise RuntimeError("DB_USER is missing from .env")

if not settings.DATABASES["default"]["PASSWORD"]:
    raise RuntimeError("DB_PASSWORD is missing from .env")

django.setup()

from django.db import connection

with connection.cursor() as cursor:
    cursor.execute("SELECT VERSION(), DATABASE()")
    mysql_version, database_name = cursor.fetchone()

tables = connection.introspection.table_names()

print("DJANGO VERSION:", django.get_version())
print("PYMYSQL VERSION:", pymysql.__version__)
print("DATABASE BACKEND:", connection.settings_dict["ENGINE"])
print("MYSQL VERSION:", mysql_version)
print("DATABASE:", database_name)
print("TABLE COUNT:", len(tables))
print("TABLES:", tables)

assert database_name == "ipl_analytics"
assert len(tables) == 7
assert mysql_version.startswith("8.0")

print("\\nPASS — Django MySQL backend connection validated.")
'''

result = subprocess.run(
    [sys.executable, "-c", test_script],
    cwd=ROOT,
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.returncode != 0:
    print("ERROR OUTPUT:")
    print(result.stderr)
    raise RuntimeError("Django MySQL backend validation failed.")

print("Child process exit code:", result.returncode)


DJANGO VERSION: 5.2.19
PYMYSQL VERSION: 2.2.8
DATABASE BACKEND: django.db.backends.mysql
MYSQL VERSION: 8.0.46
DATABASE: ipl_analytics
TABLE COUNT: 7
TABLES: ['batting_next_match_features', 'batting_player_match_regular', 'bowling_next_match_features', 'bowling_player_match_regular', 'deliveries', 'innings', 'matches']
\nPASS — Django MySQL backend connection validated.

Child process exit code: 0


In [22]:

from pathlib import Path

def find_project_root(start_path=None):
    current = Path(start_path or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        if (
            (candidate / "manage.py").is_file()
            and (candidate / "config").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate

    raise FileNotFoundError("Django project root not found.")

ROOT = find_project_root()

print("PROJECT ROOT:", ROOT)

files_to_inspect = [
    ROOT / "manage.py",
    ROOT / "config" / "settings.py",
    ROOT / "config" / "urls.py",
    ROOT / ".env",
    ROOT / ".env.example",
    ROOT / ".gitignore",
]

for file_path in files_to_inspect:
    print(f"\n{'=' * 60}")
    print("FILE:", file_path.relative_to(ROOT))
    print("EXISTS:", file_path.exists())

    if not file_path.exists():
        continue

    if file_path.name == ".env":
        print("[CONTENTS NOT DISPLAYED — credentials protected]")

    elif file_path.name == "settings.py":
        for line in file_path.read_text(encoding="utf-8").splitlines():
            if any(key in line.upper() for key in [
                "SECRET_KEY", "PASSWORD", "TOKEN"
            ]):
                print("[REDACTED — sensitive setting]")
            else:
                print(line)

    else:
        print(file_path.read_text(encoding="utf-8"))


PROJECT ROOT: D:\Git\Personal\ipl-player-performance-v2

FILE: manage.py
EXISTS: True
#!/usr/bin/env python
"""Django's command-line utility for administrative tasks."""
import os
import sys


def main():
    """Run administrative tasks."""
    os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
    try:
        from django.core.management import execute_from_command_line
    except ImportError as exc:
        raise ImportError(
            "Couldn't import Django. Are you sure it's installed and "
            "available on your PYTHONPATH environment variable? Did you "
            "forget to activate a virtual environment?"
        ) from exc
    execute_from_command_line(sys.argv)


if __name__ == "__main__":
    main()


FILE: config\settings.py
EXISTS: True
"""
Django settings for config project.

Generated by 'django-admin startproject' using Django 5.2.19.

For more information on this file, see
https://docs.djangoproject.com/en/5.2/topics/settings/

For the full 